In [1]:
# ============================================================
# INTERNATIONAL HETEROGENEITY:
# COMPOSITIONAL PCA / AITCHISON-SPACE GRADIENTS
# ============================================================
#
# PURPOSE
# -------
# We already established a conserved international backbone.
#
# Now ask:
#
#   1. Along what continuous dimensions do education systems
#      differ from one another?
#
#   2. Which predictors define those dimensions?
#
#   3. Which systems are most distinctive from the
#      international compositional center?
#
#   4. Are the same systems distinctive across different
#      bullying outcomes?
#
#   5. Do systems occupy similar relative configurations
#      across bullying outcomes?
#
#
# ANALYSIS
# --------
# - CLR transformation
# - PCA in Aitchison geometry
# - compositional center
# - system-to-center Aitchison distance
# - system-specific deviation drivers
# - cross-outcome distance-structure agreement
#
#
# PRIMARY:
#   Full 14-predictor architecture
#
# SENSITIVITY:
#   BELONG removed -> 13 predictors
#
#
# IMPORTANT
# ---------
# NO MODEL FITTING.
# NO STANDARDIZATION AFTER CLR.
# NO CLUSTERING YET.
#
# ============================================================


from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

from sklearn.decomposition import PCA

from scipy.spatial.distance import (
    pdist,
    squareform,
)

from scipy.stats import (
    spearmanr,
)


# ============================================================
# 1. Paths
# ============================================================

RANKING_PATH = Path(
    "../results/primary_attribution/"
    "oof_shap_rankings.csv"
)


OUT_DIR = Path(
    "../results/international_heterogeneity"
)


OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. Frozen predictor sets
# ============================================================

FEATURES_14 = [
    "ADAPTIVITY",
    "AGE",
    "ATTLNACT",
    "BELONG",
    "DISCLIMA",
    "EDUSHORT",
    "ESCS",
    "JOYREAD",
    "MASTGOAL",
    "METASPAM",
    "METASUM",
    "ST004D01T",
    "STIMREAD",
    "UNDREM",
]


FEATURES_13 = [
    feature
    for feature in FEATURES_14
    if feature != "BELONG"
]


OUTCOMES = [
    "BEINGBULLIED",
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


PROFILE_SPECS = {
    "Full14": FEATURES_14,
    "NoBELONG13": FEATURES_13,
}


N_PCS_PRINT = 3

N_AXIS_FEATURES_PRINT = 4

N_DISTINCTIVE_PRINT = 10

N_DEVIATION_DRIVERS = 3


# ============================================================
# 3. Load frozen attribution results
# ============================================================

rankings = pd.read_csv(
    RANKING_PATH
)


required_columns = {
    "Country",
    "Outcome",
    "Feature",
    "ImportanceShare",
}


missing_columns = (
    required_columns
    -
    set(
        rankings.columns
    )
)


if missing_columns:

    raise RuntimeError(
        "Missing required columns: "
        f"{sorted(missing_columns)}"
    )


rankings[
    "ImportanceShare"
] = pd.to_numeric(
    rankings[
        "ImportanceShare"
    ],
    errors="coerce",
)


# ============================================================
# 4. Basic QA
# ============================================================

if rankings[
    "ImportanceShare"
].isna().any():

    raise RuntimeError(
        "Missing ImportanceShare values."
    )


if (
    rankings[
        "ImportanceShare"
    ]
    <=
    0
).any():

    raise RuntimeError(
        "Zero/negative shares found. "
        "CLR requires strictly positive compositions."
    )


duplicate_rows = rankings.duplicated(
    subset=[
        "Country",
        "Outcome",
        "Feature",
    ]
)


if duplicate_rows.any():

    raise RuntimeError(
        "Duplicate Country/Outcome/Feature rows found."
    )


# ============================================================
# 5. Compositional helpers
# ============================================================

def close_composition(
    matrix,
):

    matrix = np.asarray(
        matrix,
        dtype=float,
    )


    row_sums = matrix.sum(
        axis=1,
        keepdims=True,
    )


    if (
        row_sums <= 0
    ).any():

        raise RuntimeError(
            "Non-positive composition total."
        )


    return (
        matrix
        /
        row_sums
    )


def clr_transform(
    matrix,
):

    matrix = close_composition(
        matrix
    )


    if (
        matrix <= 0
    ).any():

        raise RuntimeError(
            "CLR requires positive compositions."
        )


    log_matrix = np.log(
        matrix
    )


    return (
        log_matrix
        -
        log_matrix.mean(
            axis=1,
            keepdims=True,
        )
    )


def inverse_clr_center(
    center_clr,
):

    values = np.exp(
        center_clr
    )


    return (
        values
        /
        values.sum()
    )


# ============================================================
# 6. Build outcome/profile matrix
# ============================================================

def build_matrix(
    outcome,
    features,
):

    temp = (
        rankings[
            rankings[
                "Outcome"
            ]
            ==
            outcome
        ]
        .copy()
    )


    systems = sorted(
        temp[
            "Country"
        ]
        .unique()
    )


    rows = []


    for country in systems:

        sub = (
            temp[
                temp[
                    "Country"
                ]
                ==
                country
            ]
            .set_index(
                "Feature"
            )
        )


        missing = (
            set(
                features
            )
            -
            set(
                sub.index
            )
        )


        if missing:

            raise RuntimeError(
                f"{country} — {outcome}: "
                f"missing {sorted(missing)}"
            )


        vector = (
            sub.loc[
                features,
                "ImportanceShare",
            ]
            .to_numpy(
                dtype=float
            )
        )


        vector = (
            vector
            /
            vector.sum()
        )


        rows.append(
            vector
        )


    return (
        systems,
        np.vstack(
            rows
        ),
    )


# ============================================================
# 7. Deterministic PCA sign orientation
#
# PCA signs are mathematically arbitrary.
#
# For reproducibility:
#   find the coefficient with largest absolute magnitude;
#   orient each PC so that this coefficient is positive.
#
# This does NOT change:
#   - variance explained
#   - system distances
#   - substantive PCA geometry
# ============================================================

def orient_pca_signs(
    scores,
    components,
):

    scores = (
        scores.copy()
    )


    components = (
        components.copy()
    )


    for component_index in range(
        components.shape[
            0
        ]
    ):

        row = components[
            component_index
        ]


        strongest_index = np.argmax(
            np.abs(
                row
            )
        )


        if (
            row[
                strongest_index
            ]
            <
            0
        ):

            components[
                component_index
            ] *= -1


            scores[
                :,
                component_index
            ] *= -1


    return (
        scores,
        components,
    )


# ============================================================
# 8. Storage
# ============================================================

pca_variance_rows = []

pca_score_rows = []

pca_loading_rows = []

center_rows = []

distinctiveness_rows = []

deviation_driver_rows = []

axis_summary_rows = []


analysis_store = {}


# ============================================================
# 9. Run compositional PCA
# ============================================================

for profile_name, features in (
    PROFILE_SPECS.items()
):

    print("\n")
    print("#" * 120)

    print(
        f"PROFILE: {profile_name}"
    )

    print("#" * 120)


    for outcome in OUTCOMES:

        print("\n")
        print("=" * 105)

        print(
            outcome
        )

        print("=" * 105)


        (
            systems,
            composition
        ) = build_matrix(
            outcome=outcome,
            features=features,
        )


        # ----------------------------------------------------
        # CLR representation
        # ----------------------------------------------------

        clr = clr_transform(
            composition
        )


        # ----------------------------------------------------
        # Compositional center
        #
        # Mean CLR coordinate -> inverse CLR.
        # This is the Aitchison/compositional center.
        # ----------------------------------------------------

        center_clr = (
            clr.mean(
                axis=0
            )
        )


        center_composition = (
            inverse_clr_center(
                center_clr
            )
        )


        # ----------------------------------------------------
        # PCA
        #
        # D-part compositions have D-1 degrees of freedom.
        #
        # sklearn centers the CLR matrix internally.
        # DO NOT standardize the individual CLR columns.
        # ----------------------------------------------------

        n_components = (
            len(
                features
            )
            -
            1
        )


        pca = PCA(
            n_components=n_components
        )


        scores = pca.fit_transform(
            clr
        )


        components = (
            pca.components_
            .copy()
        )


        (
            scores,
            components,
        ) = orient_pca_signs(
            scores,
            components,
        )


        explained = (
            pca.explained_variance_ratio_
        )


        eigenvalues = (
            pca.explained_variance_
        )


        cumulative = np.cumsum(
            explained
        )


        # ----------------------------------------------------
        # Aitchison pairwise distances
        # ----------------------------------------------------

        distance_matrix = squareform(
            pdist(
                clr,
                metric="euclidean",
            )
        )


        # ----------------------------------------------------
        # Distance from international compositional center
        # ----------------------------------------------------

        distance_to_center = np.linalg.norm(
            clr
            -
            center_clr,
            axis=1,
        )


        # ----------------------------------------------------
        # Store for cross-outcome comparisons
        # ----------------------------------------------------

        analysis_store[
            (
                profile_name,
                outcome,
            )
        ] = {

            "systems":
                systems,

            "features":
                features,

            "composition":
                composition,

            "clr":
                clr,

            "center_clr":
                center_clr,

            "center_composition":
                center_composition,

            "distance_matrix":
                distance_matrix,

            "distance_to_center":
                distance_to_center,

            "scores":
                scores,

            "components":
                components,

            "explained":
                explained,
        }


        # ----------------------------------------------------
        # Variance summary
        # ----------------------------------------------------

        n80 = int(
            np.argmax(
                cumulative
                >=
                0.80
            )
            +
            1
        )


        n90 = int(
            np.argmax(
                cumulative
                >=
                0.90
            )
            +
            1
        )


        for component_index in range(
            n_components
        ):

            pca_variance_rows.append(
                {
                    "Profile":
                        profile_name,

                    "Outcome":
                        outcome,

                    "PC":
                        component_index
                        +
                        1,

                    "ExplainedVariance":
                        explained[
                            component_index
                        ],

                    "CumulativeVariance":
                        cumulative[
                            component_index
                        ],

                    "Eigenvalue":
                        eigenvalues[
                            component_index
                        ],

                    "ComponentsFor80Pct":
                        n80,

                    "ComponentsFor90Pct":
                        n90,
                }
            )


        # ----------------------------------------------------
        # System PC scores
        # ----------------------------------------------------

        for system_index, country in enumerate(
            systems
        ):

            row = {

                "Profile":
                    profile_name,

                "Outcome":
                    outcome,

                "Country":
                    country,
            }


            for component_index in range(
                n_components
            ):

                row[
                    f"PC{component_index + 1}"
                ] = (
                    scores[
                        system_index,
                        component_index
                    ]
                )


            pca_score_rows.append(
                row
            )


        # ----------------------------------------------------
        # PCA coefficients / loadings
        # ----------------------------------------------------

        for component_index in range(
            n_components
        ):

            # PCA component coefficients
            coefficients = (
                components[
                    component_index
                ]
            )


            # Classical loading:
            # eigenvector * sqrt(eigenvalue)
            loadings = (
                coefficients
                *
                np.sqrt(
                    eigenvalues[
                        component_index
                    ]
                )
            )


            for feature_index, feature in enumerate(
                features
            ):

                pca_loading_rows.append(
                    {
                        "Profile":
                            profile_name,

                        "Outcome":
                            outcome,

                        "PC":
                            component_index
                            +
                            1,

                        "Feature":
                            feature,

                        "ComponentCoefficient":
                            coefficients[
                                feature_index
                            ],

                        "Loading":
                            loadings[
                                feature_index
                            ],
                    }
                )


        # ----------------------------------------------------
        # Axis interpretation
        # ----------------------------------------------------

        for component_index in range(
            min(
                N_PCS_PRINT,
                n_components,
            )
        ):

            coefficients = (
                components[
                    component_index
                ]
            )


            order_positive = np.argsort(
                coefficients
            )[
                ::-1
            ]


            order_negative = np.argsort(
                coefficients
            )


            top_positive = [
                features[
                    index
                ]
                for index
                in order_positive[
                    :
                    N_AXIS_FEATURES_PRINT
                ]
            ]


            top_negative = [
                features[
                    index
                ]
                for index
                in order_negative[
                    :
                    N_AXIS_FEATURES_PRINT
                ]
            ]


            axis_summary_rows.append(
                {
                    "Profile":
                        profile_name,

                    "Outcome":
                        outcome,

                    "PC":
                        component_index
                        +
                        1,

                    "ExplainedVariance":
                        explained[
                            component_index
                        ],

                    "PositiveSide":
                        "; ".join(
                            top_positive
                        ),

                    "NegativeSide":
                        "; ".join(
                            top_negative
                        ),
                }
            )


        # ----------------------------------------------------
        # International compositional center
        # ----------------------------------------------------

        for feature_index, feature in enumerate(
            features
        ):

            center_rows.append(
                {
                    "Profile":
                        profile_name,

                    "Outcome":
                        outcome,

                    "Feature":
                        feature,

                    "CenterShare":
                        center_composition[
                            feature_index
                        ],

                    "CenterCLR":
                        center_clr[
                            feature_index
                        ],
                }
            )


        # ----------------------------------------------------
        # System distinctiveness
        # ----------------------------------------------------

        for system_index, country in enumerate(
            systems
        ):

            other_mask = (
                np.arange(
                    len(
                        systems
                    )
                )
                !=
                system_index
            )


            distinctiveness_rows.append(
                {
                    "Profile":
                        profile_name,

                    "Outcome":
                        outcome,

                    "Country":
                        country,

                    "AitchisonDistanceToCenter":
                        distance_to_center[
                            system_index
                        ],

                    "MeanPairwiseAitchisonDistance":
                        distance_matrix[
                            system_index,
                            other_mask
                        ]
                        .mean(),
                }
            )


        # ----------------------------------------------------
        # Deviation drivers
        #
        # Positive CLR difference:
        #   relatively more attribution than international
        #   compositional center.
        #
        # Negative:
        #   relatively less attribution.
        # ----------------------------------------------------

        clr_deviation = (
            clr
            -
            center_clr
        )


        for system_index, country in enumerate(
            systems
        ):

            deviation = clr_deviation[
                system_index
            ]


            positive_order = np.argsort(
                deviation
            )[
                ::-1
            ]


            negative_order = np.argsort(
                deviation
            )


            for rank_index in range(
                N_DEVIATION_DRIVERS
            ):

                pos_idx = (
                    positive_order[
                        rank_index
                    ]
                )


                deviation_driver_rows.append(
                    {
                        "Profile":
                            profile_name,

                        "Outcome":
                            outcome,

                        "Country":
                            country,

                        "Direction":
                            "RelativeOveremphasis",

                        "DriverRank":
                            rank_index
                            +
                            1,

                        "Feature":
                            features[
                                pos_idx
                            ],

                        "CLRDeviation":
                            deviation[
                                pos_idx
                            ],
                    }
                )


                neg_idx = (
                    negative_order[
                        rank_index
                    ]
                )


                deviation_driver_rows.append(
                    {
                        "Profile":
                            profile_name,

                        "Outcome":
                            outcome,

                        "Country":
                            country,

                        "Direction":
                            "RelativeUnderemphasis",

                        "DriverRank":
                            rank_index
                            +
                            1,

                        "Feature":
                            features[
                                neg_idx
                            ],

                        "CLRDeviation":
                            deviation[
                                neg_idx
                            ],
                    }
                )


        # ----------------------------------------------------
        # Console summary
        # ----------------------------------------------------

        print(
            "Systems:",
            len(
                systems
            )
        )


        print(
            "Features:",
            len(
                features
            )
        )


        print(
            f"PC1 variance: "
            f"{explained[0]:.4f}"
        )


        print(
            f"PC2 variance: "
            f"{explained[1]:.4f}"
        )


        print(
            f"PC3 variance: "
            f"{explained[2]:.4f}"
        )


        print(
            f"PC1+PC2: "
            f"{explained[:2].sum():.4f}"
        )


        print(
            f"PC1+PC2+PC3: "
            f"{explained[:3].sum():.4f}"
        )


        print(
            "Components for >=80%:",
            n80
        )


        print(
            "Components for >=90%:",
            n90
        )


# ============================================================
# 10. Convert outputs to DataFrames
# ============================================================

pca_variance = pd.DataFrame(
    pca_variance_rows
)


pca_scores = pd.DataFrame(
    pca_score_rows
)


pca_loadings = pd.DataFrame(
    pca_loading_rows
)


compositional_centers = pd.DataFrame(
    center_rows
)


system_distinctiveness = pd.DataFrame(
    distinctiveness_rows
)


system_deviation_drivers = pd.DataFrame(
    deviation_driver_rows
)


axis_summary = pd.DataFrame(
    axis_summary_rows
)


# ============================================================
# 11. Cross-outcome configuration agreement
#
# IMPORTANT:
#
# We do NOT directly correlate PC1 with PC1 across outcomes.
#
# Independently fitted PCA axes can:
#   - flip signs
#   - swap order
#   - rotate
#
# Instead, compare the complete pairwise Aitchison-distance
# structure, which is invariant to PCA orientation.
# ============================================================

cross_outcome_distance_rows = []


for profile_name in PROFILE_SPECS:

    for outcome_a, outcome_b in combinations(
        OUTCOMES,
        2,
    ):

        result_a = analysis_store[
            (
                profile_name,
                outcome_a,
            )
        ]


        result_b = analysis_store[
            (
                profile_name,
                outcome_b,
            )
        ]


        common_systems = sorted(
            set(
                result_a[
                    "systems"
                ]
            )
            &
            set(
                result_b[
                    "systems"
                ]
            )
        )


        index_a = {
            country: i
            for i, country
            in enumerate(
                result_a[
                    "systems"
                ]
            )
        }


        index_b = {
            country: i
            for i, country
            in enumerate(
                result_b[
                    "systems"
                ]
            )
        }


        distances_a = []

        distances_b = []


        for country_1, country_2 in combinations(
            common_systems,
            2,
        ):

            distances_a.append(
                result_a[
                    "distance_matrix"
                ][
                    index_a[
                        country_1
                    ],
                    index_a[
                        country_2
                    ],
                ]
            )


            distances_b.append(
                result_b[
                    "distance_matrix"
                ][
                    index_b[
                        country_1
                    ],
                    index_b[
                        country_2
                    ],
                ]
            )


        rho, _ = spearmanr(
            distances_a,
            distances_b,
        )


        cross_outcome_distance_rows.append(
            {
                "Profile":
                    profile_name,

                "Outcome1":
                    outcome_a,

                "Outcome2":
                    outcome_b,

                "N_common_systems":
                    len(
                        common_systems
                    ),

                "N_system_pairs":
                    len(
                        distances_a
                    ),

                "PairwiseAitchisonDistanceSpearman":
                    rho,
            }
        )


cross_outcome_distance_agreement = pd.DataFrame(
    cross_outcome_distance_rows
)


# ============================================================
# 12. Cross-outcome DISTINCTIVENESS agreement
#
# Question:
# If a system is especially unusual for one bullying outcome,
# is it also unusual for another?
# ============================================================

cross_outcome_distinctiveness_rows = []


for profile_name in PROFILE_SPECS:

    profile_dist = (
        system_distinctiveness[
            system_distinctiveness[
                "Profile"
            ]
            ==
            profile_name
        ]
    )


    for outcome_a, outcome_b in combinations(
        OUTCOMES,
        2,
    ):

        a = (
            profile_dist[
                profile_dist[
                    "Outcome"
                ]
                ==
                outcome_a
            ][
                [
                    "Country",
                    "AitchisonDistanceToCenter",
                ]
            ]
            .rename(
                columns={
                    "AitchisonDistanceToCenter":
                        "DistanceA"
                }
            )
        )


        b = (
            profile_dist[
                profile_dist[
                    "Outcome"
                ]
                ==
                outcome_b
            ][
                [
                    "Country",
                    "AitchisonDistanceToCenter",
                ]
            ]
            .rename(
                columns={
                    "AitchisonDistanceToCenter":
                        "DistanceB"
                }
            )
        )


        merged = a.merge(
            b,
            on="Country",
            how="inner",
            validate="one_to_one",
        )


        rho, _ = spearmanr(
            merged[
                "DistanceA"
            ],
            merged[
                "DistanceB"
            ],
        )


        cross_outcome_distinctiveness_rows.append(
            {
                "Profile":
                    profile_name,

                "Outcome1":
                    outcome_a,

                "Outcome2":
                    outcome_b,

                "N_common_systems":
                    len(
                        merged
                    ),

                "DistinctivenessSpearman":
                    rho,
            }
        )


cross_outcome_distinctiveness = pd.DataFrame(
    cross_outcome_distinctiveness_rows
)


# ============================================================
# 13. Between-outcome compositional-center distances
#
# This describes how much the INTERNATIONAL MEAN architecture
# changes between bullying types.
# ============================================================

center_distance_rows = []


for profile_name in PROFILE_SPECS:

    for outcome_a, outcome_b in combinations(
        OUTCOMES,
        2,
    ):

        center_a = analysis_store[
            (
                profile_name,
                outcome_a,
            )
        ][
            "center_clr"
        ]


        center_b = analysis_store[
            (
                profile_name,
                outcome_b,
            )
        ][
            "center_clr"
        ]


        center_distance_rows.append(
            {
                "Profile":
                    profile_name,

                "Outcome1":
                    outcome_a,

                "Outcome2":
                    outcome_b,

                "CenterAitchisonDistance":
                    float(
                        np.linalg.norm(
                            center_a
                            -
                            center_b
                        )
                    ),
            }
        )


center_distances = pd.DataFrame(
    center_distance_rows
)


# ============================================================
# 14. Save everything
# ============================================================

pca_variance.to_csv(
    OUT_DIR
    / "pca_explained_variance.csv",
    index=False,
)


pca_scores.to_csv(
    OUT_DIR
    / "system_pca_scores.csv",
    index=False,
)


pca_loadings.to_csv(
    OUT_DIR
    / "pca_loadings.csv",
    index=False,
)


axis_summary.to_csv(
    OUT_DIR
    / "pca_axis_summary.csv",
    index=False,
)


compositional_centers.to_csv(
    OUT_DIR
    / "international_compositional_centers.csv",
    index=False,
)


system_distinctiveness.to_csv(
    OUT_DIR
    / "system_aitchison_distinctiveness.csv",
    index=False,
)


system_deviation_drivers.to_csv(
    OUT_DIR
    / "system_deviation_drivers.csv",
    index=False,
)


cross_outcome_distance_agreement.to_csv(
    OUT_DIR
    / "cross_outcome_distance_agreement.csv",
    index=False,
)


cross_outcome_distinctiveness.to_csv(
    OUT_DIR
    / "cross_outcome_distinctiveness_agreement.csv",
    index=False,
)


center_distances.to_csv(
    OUT_DIR
    / "between_outcome_center_distances.csv",
    index=False,
)


# ============================================================
# 15. Print PCA variance
# ============================================================

print("\n")
print("=" * 120)
print("COMPOSITIONAL PCA — VARIANCE SUMMARY")
print("=" * 120)


variance_print = (
    pca_variance[
        pca_variance[
            "PC"
        ]
        <=
        3
    ]
    .pivot_table(
        index=[
            "Profile",
            "Outcome",
        ],
        columns="PC",
        values="ExplainedVariance",
    )
    .reset_index()
)


variance_print.columns = [
    (
        f"PC{column}"
        if isinstance(
            column,
            int
        )
        else column
    )
    for column
    in variance_print.columns
]


variance_print[
    "PC1to3"
] = (
    variance_print[
        [
            "PC1",
            "PC2",
            "PC3",
        ]
    ]
    .sum(
        axis=1
    )
)


print(
    variance_print
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 16. Print axis interpretation
# ============================================================

print("\n")
print("=" * 120)
print("PCA AXIS INTERPRETATION")
print("=" * 120)


print(
    axis_summary
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 17. Print most distinctive systems
# ============================================================

print("\n")
print("=" * 120)
print("MOST DISTINCTIVE SYSTEMS — FULL 14")
print("=" * 120)


for outcome in OUTCOMES:

    temp = (
        system_distinctiveness[
            (
                system_distinctiveness[
                    "Profile"
                ]
                ==
                "Full14"
            )
            &
            (
                system_distinctiveness[
                    "Outcome"
                ]
                ==
                outcome
            )
        ]
        .sort_values(
            "AitchisonDistanceToCenter",
            ascending=False,
        )
        .head(
            N_DISTINCTIVE_PRINT
        )
    )


    print("\n")
    print(
        outcome
    )

    print("-" * 80)


    print(
        temp[
            [
                "Country",
                "AitchisonDistanceToCenter",
                "MeanPairwiseAitchisonDistance",
            ]
        ]
        .round(
            4
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 18. Print deviation drivers for top 5 distinctive systems
# ============================================================

print("\n")
print("=" * 120)
print("DEVIATION DRIVERS — TOP 5 DISTINCTIVE SYSTEMS")
print("=" * 120)


for outcome in OUTCOMES:

    top_systems = (
        system_distinctiveness[
            (
                system_distinctiveness[
                    "Profile"
                ]
                ==
                "Full14"
            )
            &
            (
                system_distinctiveness[
                    "Outcome"
                ]
                ==
                outcome
            )
        ]
        .sort_values(
            "AitchisonDistanceToCenter",
            ascending=False,
        )
        .head(
            5
        )[
            "Country"
        ]
        .tolist()
    )


    print("\n")
    print(
        outcome
    )

    print("-" * 100)


    temp = (
        system_deviation_drivers[
            (
                system_deviation_drivers[
                    "Profile"
                ]
                ==
                "Full14"
            )
            &
            (
                system_deviation_drivers[
                    "Outcome"
                ]
                ==
                outcome
            )
            &
            (
                system_deviation_drivers[
                    "Country"
                ]
                .isin(
                    top_systems
                )
            )
        ]
    )


    print(
        temp[
            [
                "Country",
                "Direction",
                "DriverRank",
                "Feature",
                "CLRDeviation",
            ]
        ]
        .sort_values(
            [
                "Country",
                "Direction",
                "DriverRank",
            ]
        )
        .round(
            4
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 19. Print cross-outcome configuration agreement
# ============================================================

print("\n")
print("=" * 120)
print("CROSS-OUTCOME AITCHISON CONFIGURATION AGREEMENT")
print("=" * 120)


print(
    cross_outcome_distance_agreement
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 20. Print distinctiveness consistency
# ============================================================

print("\n")
print("=" * 120)
print("CROSS-OUTCOME DISTINCTIVENESS AGREEMENT")
print("=" * 120)


print(
    cross_outcome_distinctiveness
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 21. Print international-center differences
# ============================================================

print("\n")
print("=" * 120)
print("BETWEEN-OUTCOME INTERNATIONAL CENTER DISTANCES")
print("=" * 120)


print(
    center_distances
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 22. Files
# ============================================================

print("\n")
print("=" * 120)
print("FILES CREATED")
print("=" * 120)


for path in sorted(
    OUT_DIR.glob(
        "*.csv"
    )
):

    print(
        path
    )


print("\n")
print("=" * 120)
print("STAGE COMPLETE — DO NOT CLUSTER YET")
print("=" * 120)



########################################################################################################################
PROFILE: Full14
########################################################################################################################


BEINGBULLIED
Systems: 74
Features: 14
PC1 variance: 0.2403
PC2 variance: 0.1385
PC3 variance: 0.1291
PC1+PC2: 0.3788
PC1+PC2+PC3: 0.5079
Components for >=80%: 7
Components for >=90%: 10


BULLY_VERBAL
Systems: 75
Features: 14
PC1 variance: 0.3628
PC2 variance: 0.1296
PC3 variance: 0.1000
PC1+PC2: 0.4924
PC1+PC2+PC3: 0.5923
Components for >=80%: 7
Components for >=90%: 9


BULLY_THREAT
Systems: 75
Features: 14
PC1 variance: 0.2210
PC2 variance: 0.1578
PC3 variance: 0.1139
PC1+PC2: 0.3788
PC1+PC2+PC3: 0.4927
Components for >=80%: 7
Components for >=90%: 10


BULLY_PHYSICAL
Systems: 75
Features: 14
PC1 variance: 0.2097
PC2 variance: 0.1610
PC3 variance: 0.1143
PC1+PC2: 0.3707
PC1+PC2+PC3: 0.4850
Components for >=80%: 8
Components f

In [ ]:
# ============================================================
# MACRO-LEVEL CONTEXT OF INTERNATIONAL HETEROGENEITY
# ============================================================
#
# PURPOSE
# -------
# Test whether variation around the shared international
# bullying-predictor backbone is systematically associated
# with macro-level characteristics of education systems.
#
#
# MACRO VARIABLES
# ---------------
# 1. 2018 GDP per capita, PPP
# 2. World Bank FY2019 income group
#    (classification in force from July 1, 2018)
# 3. World Bank geographic region
# 4. Gini index nearest to 2018 within +/- 3 years
#
#
# PRIMARY RESPONSE
# ----------------
# Complete CLR-transformed SHAP composition.
#
# Therefore the main tests examine the ENTIRE predictor
# architecture, not one feature at a time.
#
#
# STATISTICS
# ----------
# Continuous macro variables:
#     multivariate CLR regression + permutation test
#
# Categorical macro variables:
#     permutation MANOVA / PERMANOVA-equivalent test in
#     Euclidean CLR space
#
# Categorical tests additionally use PERMDISP-style
# permutation diagnostics to determine whether apparent
# differences might reflect unequal within-group dispersion.
#
#
# PRIMARY:
#     Full14
#
# SENSITIVITY:
#     NoBELONG13
#
#
# IMPORTANT
# ---------
# NO XGBOOST FITTING.
# NO CLUSTERING YET.
# ============================================================


from pathlib import Path
import time

import numpy as np
import pandas as pd
import requests

from scipy.stats import (
    spearmanr,
)


# ============================================================
# 1. Configuration
# ============================================================

RANKING_PATH = Path(
    "../results/primary_attribution/"
    "oof_shap_rankings.csv"
)


PCA_SCORE_PATH = Path(
    "../results/international_heterogeneity/"
    "system_pca_scores.csv"
)


DISTINCTIVENESS_PATH = Path(
    "../results/international_heterogeneity/"
    "system_aitchison_distinctiveness.csv"
)


OUT_DIR = Path(
    "../results/macro_context"
)


OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


FEATURES_14 = [
    "ADAPTIVITY",
    "AGE",
    "ATTLNACT",
    "BELONG",
    "DISCLIMA",
    "EDUSHORT",
    "ESCS",
    "JOYREAD",
    "MASTGOAL",
    "METASPAM",
    "METASUM",
    "ST004D01T",
    "STIMREAD",
    "UNDREM",
]


FEATURES_13 = [
    feature
    for feature in FEATURES_14
    if feature != "BELONG"
]


PROFILE_SPECS = {
    "Full14": FEATURES_14,
    "NoBELONG13": FEATURES_13,
}


OUTCOMES = [
    "BEINGBULLIED",
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


N_PERMUTATIONS = 10_000

RANDOM_SEED = 42


# ============================================================
# 2. World Bank indicators
# ============================================================

INDICATORS = {

    # GDP per capita, purchasing-power adjusted
    "GDPpcPPP":
        "NY.GDP.PCAP.PP.KD",

    # GNI per capita, Atlas method
    # Used to reconstruct FY2019 income category.
    "GNIpcAtlas":
        "NY.GNP.PCAP.CD",

    # Gini index
    "Gini":
        "SI.POV.GINI",
}


# ============================================================
# 3. PISA -> World Bank code mapping
# ============================================================
#
# Most PISA three-letter codes already match World Bank codes.
#
# Kosovo:
#     PISA = KSV
#     World Bank = XKX
#
# QAZ:
#     Baku/Azerbaijan PISA entity
#     not identical to national Azerbaijan
#
# QCI:
#     B-S-J-Z (China)
#     not identical to national China
#
# TAP:
#     Chinese Taipei
#     not available as a directly corresponding World Bank
#     economy through the same WDI framework.
#
# Those three are therefore NOT assigned national proxies in
# the primary macro-context analysis.
#
# ============================================================

SPECIAL_WB_MAPPING = {
    "KSV": "XK",
    "QAZ": None,
    "QCI": None,
    "TAP": None,
}


SPECIAL_REASON = {

    "QAZ":
        "PISA subnational Baku/Azerbaijan entity; "
        "national Azerbaijan macro data not used as primary proxy.",

    "QCI":
        "PISA B-S-J-Z (China) entity; "
        "national China macro data not used as primary proxy.",

    "TAP":
        "Chinese Taipei lacks directly corresponding "
        "World Bank WDI economy data in this framework.",
}


# ============================================================
# 4. Load analytical outputs
# ============================================================

rankings = pd.read_csv(
    RANKING_PATH
)


pca_scores = pd.read_csv(
    PCA_SCORE_PATH
)


distinctiveness = pd.read_csv(
    DISTINCTIVENESS_PATH
)


all_systems = sorted(
    rankings[
        "Country"
    ]
    .unique()
)


# ============================================================
# 5. World Bank API helpers
# ============================================================

SESSION = requests.Session()


SESSION.headers.update(
    {
        "User-Agent":
            "PISA-2018-Bullying-Research/1.0"
    }
)


def wb_get_json(
    url,
    params=None,
    attempts=4,
):

    last_error = None


    for attempt in range(
        attempts
    ):

        try:

            response = SESSION.get(
                url,
                params=params,
                timeout=30,
            )


            response.raise_for_status()


            payload = response.json()


            return payload


        except Exception as e:

            last_error = e


            time.sleep(
                1.5
                *
                (
                    attempt
                    +
                    1
                )
            )


    raise RuntimeError(
        f"World Bank request failed: "
        f"{last_error}"
    )


def fetch_indicator(
    wb_code,
    indicator,
    start_year,
    end_year,
):

    url = (
        "https://api.worldbank.org/v2/"
        f"country/{wb_code}/"
        f"indicator/{indicator}"
    )


    payload = wb_get_json(
        url,
        params={
            "format": "json",
            "date":
                f"{start_year}:{end_year}",
            "per_page": 100,
        },
    )


    if (
        not isinstance(
            payload,
            list,
        )
        or
        len(
            payload
        )
        < 2
        or
        payload[
            1
        ]
        is None
    ):

        return pd.DataFrame(
            columns=[
                "Year",
                "Value",
            ]
        )


    rows = []


    for item in payload[
        1
    ]:

        value = item.get(
            "value"
        )


        year = item.get(
            "date"
        )


        if (
            value is None
            or
            year is None
        ):

            continue


        rows.append(
            {
                "Year":
                    int(
                        year
                    ),

                "Value":
                    float(
                        value
                    ),
            }
        )


    return pd.DataFrame(
        rows
    )


def fetch_country_metadata(
    wb_code,
):

    url = (
        "https://api.worldbank.org/v2/"
        f"country/{wb_code}"
    )


    payload = wb_get_json(
        url,
        params={
            "format": "json",
            "per_page": 10,
        },
    )


    if (
        not isinstance(
            payload,
            list,
        )
        or
        len(
            payload
        )
        < 2
        or
        not payload[
            1
        ]
    ):

        return {
            "WB_Name": None,
            "WB_Region": None,
        }


    record = payload[
        1
    ][
        0
    ]


    return {

        "WB_Name":
            record.get(
                "name"
            ),

        "WB_Region":
            (
                record.get(
                    "region",
                    {}
                )
                .get(
                    "value"
                )
            ),
    }


# ============================================================
# 6. Reconstruct FY2019 World Bank income category
#
# Classification released July 1, 2018.
#
# Based on 2017 Atlas-method GNI per capita.
#
# Thresholds:
#
# Low                 <= 995
# Lower-middle         996–3,895
# Upper-middle       3,896–12,055
# High               > 12,055
#
# NOTE:
# Current WDI historical GNI values can be revised.
# We label this explicitly as a reconstructed category.
#
# If income-group findings become manuscript-critical,
# we will verify against the World Bank historical
# classification workbook before publication.
# ============================================================

def reconstruct_income_group_fy2019(
    gni,
):

    if (
        pd.isna(
            gni
        )
        or
        not np.isfinite(
            gni
        )
    ):

        return np.nan


    if gni <= 995:

        return "Low"


    if gni <= 3895:

        return "Lower-middle"


    if gni <= 12055:

        return "Upper-middle"


    return "High"


# ============================================================
# 7. Retrieve system-level macro data
# ============================================================

metadata_rows = []

fetch_error_rows = []


print("\n")
print("=" * 110)
print("FETCHING WORLD BANK MACRO DATA")
print("=" * 110)


for number, country in enumerate(
    all_systems,
    start=1,
):

    wb_code = (
        SPECIAL_WB_MAPPING[
            country
        ]
        if country in SPECIAL_WB_MAPPING
        else country
    )


    print(
        f"[{number}/{len(all_systems)}] "
        f"{country}",
        end=" ... ",
    )


    if wb_code is None:

        metadata_rows.append(
            {
                "Country":
                    country,

                "WB_Code":
                    np.nan,

                "WB_Name":
                    np.nan,

                "WB_Region":
                    np.nan,

                "GDPpcPPP_2018":
                    np.nan,

                "LogGDPpcPPP_2018":
                    np.nan,

                "GNIpcAtlas_2017":
                    np.nan,

                "IncomeGroup_FY2019":
                    np.nan,

                "Gini_Nearest2018":
                    np.nan,

                "Gini_Year":
                    np.nan,

                "PrimaryMacroEligible":
                    False,

                "ExclusionReason":
                    SPECIAL_REASON.get(
                        country
                    ),
            }
        )


        print(
            "SPECIAL ENTITY — excluded from primary macro mapping"
        )


        continue


    try:

        country_meta = (
            fetch_country_metadata(
                wb_code
            )
        )


        # ----------------------------------------------------
        # 2018 GDP per capita PPP
        # ----------------------------------------------------

        gdp_data = fetch_indicator(
            wb_code,
            INDICATORS[
                "GDPpcPPP"
            ],
            2018,
            2018,
        )


        if len(
            gdp_data
        ) > 0:

            gdp_2018 = (
                gdp_data[
                    "Value"
                ]
                .iloc[
                    0
                ]
            )


        else:

            gdp_2018 = np.nan


        # ----------------------------------------------------
        # 2017 GNI per capita Atlas
        # ----------------------------------------------------

        gni_data = fetch_indicator(
            wb_code,
            INDICATORS[
                "GNIpcAtlas"
            ],
            2017,
            2017,
        )


        if len(
            gni_data
        ) > 0:

            gni_2017 = (
                gni_data[
                    "Value"
                ]
                .iloc[
                    0
                ]
            )


        else:

            gni_2017 = np.nan


        # ----------------------------------------------------
        # Gini nearest to 2018 within +/- 3 years
        # ----------------------------------------------------

        gini_data = fetch_indicator(
            wb_code,
            INDICATORS[
                "Gini"
            ],
            2015,
            2021,
        )


        if len(
            gini_data
        ) > 0:

            gini_data[
                "DistanceFrom2018"
            ] = np.abs(
                gini_data[
                    "Year"
                ]
                -
                2018
            )


            # Tie -> earlier year preferred.
            gini_data = (
                gini_data
                .sort_values(
                    [
                        "DistanceFrom2018",
                        "Year",
                    ],
                    ascending=[
                        True,
                        True,
                    ],
                )
            )


            gini_value = (
                gini_data[
                    "Value"
                ]
                .iloc[
                    0
                ]
            )


            gini_year = int(
                gini_data[
                    "Year"
                ]
                .iloc[
                    0
                ]
            )


        else:

            gini_value = np.nan

            gini_year = np.nan


        income_group = (
            reconstruct_income_group_fy2019(
                gni_2017
            )
        )


        eligible = bool(
            pd.notna(
                country_meta[
                    "WB_Region"
                ]
            )
        )


        metadata_rows.append(
            {
                "Country":
                    country,

                "WB_Code":
                    wb_code,

                "WB_Name":
                    country_meta[
                        "WB_Name"
                    ],

                "WB_Region":
                    country_meta[
                        "WB_Region"
                    ],

                "GDPpcPPP_2018":
                    gdp_2018,

                "LogGDPpcPPP_2018":
                    (
                        np.log(
                            gdp_2018
                        )
                        if (
                            pd.notna(
                                gdp_2018
                            )
                            and
                            gdp_2018 > 0
                        )
                        else np.nan
                    ),

                "GNIpcAtlas_2017":
                    gni_2017,

                "IncomeGroup_FY2019":
                    income_group,

                "Gini_Nearest2018":
                    gini_value,

                "Gini_Year":
                    gini_year,

                "PrimaryMacroEligible":
                    eligible,

                "ExclusionReason":
                    np.nan,
            }
        )


        print(
            "OK"
        )


    except Exception as e:

        fetch_error_rows.append(
            {
                "Country":
                    country,

                "WB_Code":
                    wb_code,

                "ErrorType":
                    type(
                        e
                    ).__name__,

                "Error":
                    str(
                        e
                    ),
            }
        )


        metadata_rows.append(
            {
                "Country":
                    country,

                "WB_Code":
                    wb_code,

                "WB_Name":
                    np.nan,

                "WB_Region":
                    np.nan,

                "GDPpcPPP_2018":
                    np.nan,

                "LogGDPpcPPP_2018":
                    np.nan,

                "GNIpcAtlas_2017":
                    np.nan,

                "IncomeGroup_FY2019":
                    np.nan,

                "Gini_Nearest2018":
                    np.nan,

                "Gini_Year":
                    np.nan,

                "PrimaryMacroEligible":
                    False,

                "ExclusionReason":
                    (
                        f"API fetch error: {e}"
                    ),
            }
        )


        print(
            f"ERROR: {e}"
        )


    time.sleep(
        0.05
    )


macro_metadata = pd.DataFrame(
    metadata_rows
)


fetch_errors = pd.DataFrame(
    fetch_error_rows,
    columns=[
        "Country",
        "WB_Code",
        "ErrorType",
        "Error",
    ],
)


macro_metadata.to_csv(
    OUT_DIR
    / "system_macro_metadata.csv",
    index=False,
)


fetch_errors.to_csv(
    OUT_DIR
    / "macro_fetch_errors.csv",
    index=False,
)


# ============================================================
# 8. Build CLR attribution matrices
# ============================================================

def build_clr_matrix(
    outcome,
    features,
):

    temp = (
        rankings[
            rankings[
                "Outcome"
            ]
            ==
            outcome
        ]
    )


    systems = sorted(
        temp[
            "Country"
        ]
        .unique()
    )


    matrix = []


    for country in systems:

        vector = (
            temp[
                temp[
                    "Country"
                ]
                ==
                country
            ]
            .set_index(
                "Feature"
            )
            .loc[
                features,
                "ImportanceShare",
            ]
            .to_numpy(
                dtype=float
            )
        )


        vector = (
            vector
            /
            vector.sum()
        )


        matrix.append(
            vector
        )


    matrix = np.vstack(
        matrix
    )


    log_matrix = np.log(
        matrix
    )


    clr = (
        log_matrix
        -
        log_matrix.mean(
            axis=1,
            keepdims=True,
        )
    )


    return (
        systems,
        clr,
    )


# ============================================================
# 9. Multivariate statistics
# ============================================================

def total_ss(
    Y
):

    centered = (
        Y
        -
        Y.mean(
            axis=0
        )
    )


    return float(
        np.sum(
            centered ** 2
        )
    )


def categorical_multivariate_stat(
    Y,
    groups,
):

    Y = np.asarray(
        Y,
        dtype=float,
    )


    groups = np.asarray(
        groups
    )


    unique_groups = np.unique(
        groups
    )


    n = len(
        groups
    )


    g = len(
        unique_groups
    )


    grand = Y.mean(
        axis=0
    )


    ss_total = np.sum(
        (
            Y
            -
            grand
        ) ** 2
    )


    ss_within = 0.0


    for group in unique_groups:

        group_Y = Y[
            groups
            ==
            group
        ]


        centroid = group_Y.mean(
            axis=0
        )


        ss_within += np.sum(
            (
                group_Y
                -
                centroid
            ) ** 2
        )


    ss_between = (
        ss_total
        -
        ss_within
    )


    df_between = (
        g
        -
        1
    )


    df_within = (
        n
        -
        g
    )


    F = (
        (
            ss_between
            /
            df_between
        )
        /
        (
            ss_within
            /
            df_within
        )
    )


    R2 = (
        ss_between
        /
        ss_total
    )


    return (
        float(
            F
        ),
        float(
            R2
        ),
    )


def continuous_multivariate_stat(
    Y,
    x,
):

    Y = np.asarray(
        Y,
        dtype=float,
    )


    x = np.asarray(
        x,
        dtype=float,
    )


    x = (
        x
        -
        x.mean()
    ) / x.std(
        ddof=1
    )


    n = len(
        x
    )


    design = np.column_stack(
        [
            np.ones(
                n
            ),
            x,
        ]
    )


    beta = np.linalg.lstsq(
        design,
        Y,
        rcond=None,
    )[
        0
    ]


    fitted = (
        design
        @
        beta
    )


    residual = (
        Y
        -
        fitted
    )


    grand = Y.mean(
        axis=0
    )


    ss_total = np.sum(
        (
            Y
            -
            grand
        ) ** 2
    )


    ss_residual = np.sum(
        residual ** 2
    )


    ss_model = (
        ss_total
        -
        ss_residual
    )


    df_model = 1

    df_residual = (
        n
        -
        2
    )


    F = (
        (
            ss_model
            /
            df_model
        )
        /
        (
            ss_residual
            /
            df_residual
        )
    )


    R2 = (
        ss_model
        /
        ss_total
    )


    return (
        float(
            F
        ),
        float(
            R2
        ),
    )


# ============================================================
# 10. Permutation tests
# ============================================================

def permutation_test_categorical(
    Y,
    groups,
    seed,
):

    observed_F, observed_R2 = (
        categorical_multivariate_stat(
            Y,
            groups,
        )
    )


    rng = np.random.default_rng(
        seed
    )


    null_F = np.empty(
        N_PERMUTATIONS,
        dtype=float,
    )


    for b in range(
        N_PERMUTATIONS
    ):

        permuted = rng.permutation(
            groups
        )


        null_F[
            b
        ] = (
            categorical_multivariate_stat(
                Y,
                permuted,
            )[
                0
            ]
        )


    p_value = (
        1
        +
        np.sum(
            null_F
            >=
            observed_F
        )
    ) / (
        N_PERMUTATIONS
        +
        1
    )


    return {
        "F":
            observed_F,

        "R2":
            observed_R2,

        "PermutationP":
            float(
                p_value
            ),

        "NullFMean":
            float(
                null_F.mean()
            ),
    }


def permutation_test_continuous(
    Y,
    x,
    seed,
):

    observed_F, observed_R2 = (
        continuous_multivariate_stat(
            Y,
            x,
        )
    )


    rng = np.random.default_rng(
        seed
    )


    null_F = np.empty(
        N_PERMUTATIONS,
        dtype=float,
    )


    for b in range(
        N_PERMUTATIONS
    ):

        permuted = rng.permutation(
            x
        )


        null_F[
            b
        ] = (
            continuous_multivariate_stat(
                Y,
                permuted,
            )[
                0
            ]
        )


    p_value = (
        1
        +
        np.sum(
            null_F
            >=
            observed_F
        )
    ) / (
        N_PERMUTATIONS
        +
        1
    )


    return {
        "F":
            observed_F,

        "R2":
            observed_R2,

        "PermutationP":
            float(
                p_value
            ),

        "NullFMean":
            float(
                null_F.mean()
            ),
    }


# ============================================================
# 11. PERMDISP-style diagnostic
# ============================================================

def dispersion_stat(
    Y,
    groups,
):

    groups = np.asarray(
        groups
    )


    distances = np.empty(
        len(
            groups
        ),
        dtype=float,
    )


    unique_groups = np.unique(
        groups
    )


    for group in unique_groups:

        mask = (
            groups
            ==
            group
        )


        centroid = (
            Y[
                mask
            ]
            .mean(
                axis=0
            )
        )


        distances[
            mask
        ] = np.linalg.norm(
            Y[
                mask
            ]
            -
            centroid,
            axis=1,
        )


    grand = (
        distances.mean()
    )


    ss_between = 0.0

    ss_within = 0.0


    for group in unique_groups:

        values = distances[
            groups
            ==
            group
        ]


        mean_group = (
            values.mean()
        )


        ss_between += (
            len(
                values
            )
            *
            (
                mean_group
                -
                grand
            ) ** 2
        )


        ss_within += np.sum(
            (
                values
                -
                mean_group
            ) ** 2
        )


    df_between = (
        len(
            unique_groups
        )
        -
        1
    )


    df_within = (
        len(
            groups
        )
        -
        len(
            unique_groups
        )
    )


    F = (
        (
            ss_between
            /
            df_between
        )
        /
        (
            ss_within
            /
            df_within
        )
    )


    return float(
        F
    )


def permutation_dispersion_test(
    Y,
    groups,
    seed,
):

    observed = dispersion_stat(
        Y,
        groups,
    )


    rng = np.random.default_rng(
        seed
    )


    null = np.empty(
        N_PERMUTATIONS
    )


    for b in range(
        N_PERMUTATIONS
    ):

        permuted = rng.permutation(
            groups
        )


        null[
            b
        ] = dispersion_stat(
            Y,
            permuted,
        )


    p = (
        1
        +
        np.sum(
            null
            >=
            observed
        )
    ) / (
        N_PERMUTATIONS
        +
        1
    )


    return (
        observed,
        float(
            p
        ),
    )


# ============================================================
# 12. Benjamini-Hochberg FDR
# ============================================================

def bh_fdr(
    p_values,
):

    p = np.asarray(
        p_values,
        dtype=float,
    )


    result = np.full(
        len(
            p
        ),
        np.nan,
    )


    valid = np.isfinite(
        p
    )


    valid_p = p[
        valid
    ]


    if len(
        valid_p
    ) == 0:

        return result


    order = np.argsort(
        valid_p
    )


    ranked = valid_p[
        order
    ]


    m = len(
        ranked
    )


    adjusted = (
        ranked
        *
        m
        /
        np.arange(
            1,
            m + 1
        )
    )


    adjusted = np.minimum.accumulate(
        adjusted[
            ::-1
        ]
    )[
        ::-1
    ]


    adjusted = np.minimum(
        adjusted,
        1.0
    )


    unsorted = np.empty(
        m
    )


    unsorted[
        order
    ] = adjusted


    result[
        valid
    ] = unsorted


    return result


# ============================================================
# 13. Macro metadata coverage
# ============================================================

print("\n")
print("=" * 115)
print("MACRO METADATA COVERAGE")
print("=" * 115)


coverage_rows = []


for variable in [
    "WB_Region",
    "IncomeGroup_FY2019",
    "GDPpcPPP_2018",
    "LogGDPpcPPP_2018",
    "Gini_Nearest2018",
]:

    coverage_rows.append(
        {
            "Variable":
                variable,

            "Available":
                int(
                    macro_metadata[
                        variable
                    ]
                    .notna()
                    .sum()
                ),

            "Missing":
                int(
                    macro_metadata[
                        variable
                    ]
                    .isna()
                    .sum()
                ),

            "PctAvailable":
                100
                *
                macro_metadata[
                    variable
                ]
                .notna()
                .mean(),
        }
    )


coverage = pd.DataFrame(
    coverage_rows
)


print(
    coverage
    .round(
        2
    )
    .to_string(
        index=False
    )
)


coverage.to_csv(
    OUT_DIR
    / "macro_metadata_coverage.csv",
    index=False,
)


print("\nINCOME GROUP COUNTS")

print(
    macro_metadata[
        "IncomeGroup_FY2019"
    ]
    .value_counts(
        dropna=False
    )
    .to_string()
)


print("\nREGION COUNTS")

print(
    macro_metadata[
        "WB_Region"
    ]
    .value_counts(
        dropna=False
    )
    .to_string()
)


# ============================================================
# 14. Main full-composition tests
# ============================================================

global_test_rows = []

dispersion_rows = []


categorical_variables = [
    "IncomeGroup_FY2019",
    "WB_Region",
]


continuous_variables = [
    "LogGDPpcPPP_2018",
    "Gini_Nearest2018",
]


test_counter = 0


for profile_name, features in (
    PROFILE_SPECS.items()
):

    for outcome in OUTCOMES:

        systems, clr = (
            build_clr_matrix(
                outcome,
                features,
            )
        )


        clr_df = pd.DataFrame(
            clr,
            index=systems,
            columns=features,
        )


        # ----------------------------------------------------
        # Categorical context
        # ----------------------------------------------------

        for variable in categorical_variables:

            test_counter += 1


            metadata_temp = (
                macro_metadata[
                    [
                        "Country",
                        variable,
                    ]
                ]
                .dropna()
            )


            common = sorted(
                set(
                    clr_df.index
                )
                &
                set(
                    metadata_temp[
                        "Country"
                    ]
                )
            )


            metadata_temp = (
                metadata_temp
                .set_index(
                    "Country"
                )
                .loc[
                    common
                ]
            )


            Y = (
                clr_df
                .loc[
                    common
                ]
                .to_numpy()
            )


            groups = (
                metadata_temp[
                    variable
                ]
                .astype(
                    str
                )
                .to_numpy()
            )


            group_counts = (
                pd.Series(
                    groups
                )
                .value_counts()
            )


            if (
                len(
                    group_counts
                )
                <
                2
            ):

                continue


            result = (
                permutation_test_categorical(
                    Y,
                    groups,
                    seed=(
                        RANDOM_SEED
                        +
                        test_counter
                        *
                        100
                    ),
                )
            )


            global_test_rows.append(
                {
                    "Profile":
                        profile_name,

                    "Outcome":
                        outcome,

                    "MacroVariable":
                        variable,

                    "VariableType":
                        "Categorical",

                    "N_systems":
                        len(
                            common
                        ),

                    "N_groups":
                        len(
                            group_counts
                        ),

                    "MinimumGroupN":
                        int(
                            group_counts.min()
                        ),

                    **result,
                }
            )


            # -----------------------------------------------
            # Dispersion diagnostic
            # -----------------------------------------------

            if (
                group_counts.min()
                >=
                2
            ):

                disp_F, disp_p = (
                    permutation_dispersion_test(
                        Y,
                        groups,
                        seed=(
                            RANDOM_SEED
                            +
                            test_counter
                            *
                            100
                            +
                            1
                        ),
                    )
                )


            else:

                disp_F = np.nan

                disp_p = np.nan


            dispersion_rows.append(
                {
                    "Profile":
                        profile_name,

                    "Outcome":
                        outcome,

                    "MacroVariable":
                        variable,

                    "N_systems":
                        len(
                            common
                        ),

                    "DispersionF":
                        disp_F,

                    "DispersionPermutationP":
                        disp_p,
                }
            )


        # ----------------------------------------------------
        # Continuous context
        # ----------------------------------------------------

        for variable in continuous_variables:

            test_counter += 1


            metadata_temp = (
                macro_metadata[
                    [
                        "Country",
                        variable,
                    ]
                ]
                .dropna()
            )


            common = sorted(
                set(
                    clr_df.index
                )
                &
                set(
                    metadata_temp[
                        "Country"
                    ]
                )
            )


            metadata_temp = (
                metadata_temp
                .set_index(
                    "Country"
                )
                .loc[
                    common
                ]
            )


            Y = (
                clr_df
                .loc[
                    common
                ]
                .to_numpy()
            )


            x = (
                metadata_temp[
                    variable
                ]
                .to_numpy(
                    dtype=float
                )
            )


            if len(
                common
            ) < 10:

                continue


            result = (
                permutation_test_continuous(
                    Y,
                    x,
                    seed=(
                        RANDOM_SEED
                        +
                        test_counter
                        *
                        100
                    ),
                )
            )


            global_test_rows.append(
                {
                    "Profile":
                        profile_name,

                    "Outcome":
                        outcome,

                    "MacroVariable":
                        variable,

                    "VariableType":
                        "Continuous",

                    "N_systems":
                        len(
                            common
                        ),

                    "N_groups":
                        np.nan,

                    "MinimumGroupN":
                        np.nan,

                    **result,
                }
            )


global_tests = pd.DataFrame(
    global_test_rows
)


dispersion_tests = pd.DataFrame(
    dispersion_rows
)


# ============================================================
# 15. FDR within primary and BELONG-removed analysis families
# ============================================================

global_tests[
    "FDR_Q"
] = np.nan


for profile_name in global_tests[
    "Profile"
].unique():

    mask = (
        global_tests[
            "Profile"
        ]
        ==
        profile_name
    )


    global_tests.loc[
        mask,
        "FDR_Q"
    ] = bh_fdr(
        global_tests.loc[
            mask,
            "PermutationP"
        ]
    )


# ============================================================
# 16. Group compositional centers
# ============================================================

group_center_rows = []

group_center_distance_rows = []


for profile_name, features in (
    PROFILE_SPECS.items()
):

    for outcome in OUTCOMES:

        systems, clr = (
            build_clr_matrix(
                outcome,
                features,
            )
        )


        clr_df = pd.DataFrame(
            clr,
            index=systems,
            columns=features,
        )


        for variable in categorical_variables:

            metadata_temp = (
                macro_metadata[
                    [
                        "Country",
                        variable,
                    ]
                ]
                .dropna()
                .set_index(
                    "Country"
                )
            )


            common = sorted(
                set(
                    clr_df.index
                )
                &
                set(
                    metadata_temp.index
                )
            )


            if len(
                common
            ) == 0:

                continue


            group_labels = (
                metadata_temp
                .loc[
                    common,
                    variable
                ]
                .astype(
                    str
                )
            )


            centers = {}


            for group in sorted(
                group_labels.unique()
            ):

                group_systems = (
                    group_labels[
                        group_labels
                        ==
                        group
                    ]
                    .index
                    .tolist()
                )


                center_clr = (
                    clr_df
                    .loc[
                        group_systems
                    ]
                    .mean(
                        axis=0
                    )
                    .to_numpy()
                )


                centers[
                    group
                ] = center_clr


                center_comp = np.exp(
                    center_clr
                )


                center_comp = (
                    center_comp
                    /
                    center_comp.sum()
                )


                for feature_index, feature in enumerate(
                    features
                ):

                    group_center_rows.append(
                        {
                            "Profile":
                                profile_name,

                            "Outcome":
                                outcome,

                            "MacroVariable":
                                variable,

                            "Group":
                                group,

                            "N_systems":
                                len(
                                    group_systems
                                ),

                            "Feature":
                                feature,

                            "CenterShare":
                                center_comp[
                                    feature_index
                                ],
                        }
                    )


            groups = sorted(
                centers.keys()
            )


            for i in range(
                len(
                    groups
                )
            ):

                for j in range(
                    i + 1,
                    len(
                        groups
                    )
                ):

                    group_a = groups[
                        i
                    ]

                    group_b = groups[
                        j
                    ]


                    distance = np.linalg.norm(
                        centers[
                            group_a
                        ]
                        -
                        centers[
                            group_b
                        ]
                    )


                    group_center_distance_rows.append(
                        {
                            "Profile":
                                profile_name,

                            "Outcome":
                                outcome,

                            "MacroVariable":
                                variable,

                            "Group1":
                                group_a,

                            "Group2":
                                group_b,

                            "CenterAitchisonDistance":
                                float(
                                    distance
                                ),
                        }
                    )


group_centers = pd.DataFrame(
    group_center_rows
)


group_center_distances = pd.DataFrame(
    group_center_distance_rows
)


# ============================================================
# 17. PCA correlations with continuous macro variables
#
# SECONDARY / INTERPRETIVE.
#
# The formal test above uses the complete CLR composition.
# These correlations help explain which PCA gradients relate
# to development/inequality.
# ============================================================

pca_macro_rows = []


for profile_name in [
    "Full14",
    "NoBELONG13",
]:

    for outcome in OUTCOMES:

        temp_scores = (
            pca_scores[
                (
                    pca_scores[
                        "Profile"
                    ]
                    ==
                    profile_name
                )
                &
                (
                    pca_scores[
                        "Outcome"
                    ]
                    ==
                    outcome
                )
            ]
        )


        merged = (
            temp_scores
            .merge(
                macro_metadata,
                on="Country",
                how="left",
                validate="one_to_one",
            )
        )


        for variable in continuous_variables:

            for pc in [
                "PC1",
                "PC2",
                "PC3",
            ]:

                valid = (
                    merged[
                        variable
                    ]
                    .notna()
                    &
                    merged[
                        pc
                    ]
                    .notna()
                )


                if valid.sum() < 10:

                    continue


                rho, p = spearmanr(
                    merged.loc[
                        valid,
                        variable
                    ],
                    merged.loc[
                        valid,
                        pc
                    ],
                )


                pca_macro_rows.append(
                    {
                        "Profile":
                            profile_name,

                        "Outcome":
                            outcome,

                        "MacroVariable":
                            variable,

                        "PC":
                            pc,

                        "N_systems":
                            int(
                                valid.sum()
                            ),

                        "SpearmanRho":
                            rho,

                        "P":
                            p,
                    }
                )


pca_macro = pd.DataFrame(
    pca_macro_rows
)


pca_macro[
    "FDR_Q"
] = bh_fdr(
    pca_macro[
        "P"
    ]
)


# ============================================================
# 18. Distinctiveness vs macro context
# ============================================================

distinctiveness_macro_rows = []


merged_distinctiveness = (
    distinctiveness
    .merge(
        macro_metadata,
        on="Country",
        how="left",
        validate="many_to_one",
    )
)


for profile_name in [
    "Full14",
    "NoBELONG13",
]:

    for outcome in OUTCOMES:

        temp = (
            merged_distinctiveness[
                (
                    merged_distinctiveness[
                        "Profile"
                    ]
                    ==
                    profile_name
                )
                &
                (
                    merged_distinctiveness[
                        "Outcome"
                    ]
                    ==
                    outcome
                )
            ]
        )


        for variable in continuous_variables:

            valid = (
                temp[
                    variable
                ]
                .notna()
                &
                temp[
                    "AitchisonDistanceToCenter"
                ]
                .notna()
            )


            if valid.sum() < 10:

                continue


            rho, p = spearmanr(
                temp.loc[
                    valid,
                    variable
                ],
                temp.loc[
                    valid,
                    "AitchisonDistanceToCenter"
                ],
            )


            distinctiveness_macro_rows.append(
                {
                    "Profile":
                        profile_name,

                    "Outcome":
                        outcome,

                    "MacroVariable":
                        variable,

                    "N_systems":
                        int(
                            valid.sum()
                        ),

                    "SpearmanRho":
                        rho,

                    "P":
                        p,
                }
            )


distinctiveness_macro = pd.DataFrame(
    distinctiveness_macro_rows
)


distinctiveness_macro[
    "FDR_Q"
] = bh_fdr(
    distinctiveness_macro[
        "P"
    ]
)


# ============================================================
# 19. Save outputs
# ============================================================

global_tests.to_csv(
    OUT_DIR
    / "macro_multivariate_tests.csv",
    index=False,
)


dispersion_tests.to_csv(
    OUT_DIR
    / "macro_dispersion_tests.csv",
    index=False,
)


group_centers.to_csv(
    OUT_DIR
    / "macro_group_compositional_centers.csv",
    index=False,
)


group_center_distances.to_csv(
    OUT_DIR
    / "macro_group_center_distances.csv",
    index=False,
)


pca_macro.to_csv(
    OUT_DIR
    / "pca_macro_correlations.csv",
    index=False,
)


distinctiveness_macro.to_csv(
    OUT_DIR
    / "distinctiveness_macro_correlations.csv",
    index=False,
)


# ============================================================
# 20. Print primary results
# ============================================================

print("\n")
print("=" * 120)
print("PRIMARY FULL14 MULTIVARIATE MACRO TESTS")
print("=" * 120)


primary_tests = (
    global_tests[
        global_tests[
            "Profile"
        ]
        ==
        "Full14"
    ]
    .copy()
)


print(
    primary_tests[
        [
            "Outcome",
            "MacroVariable",
            "VariableType",
            "N_systems",
            "N_groups",
            "MinimumGroupN",
            "F",
            "R2",
            "PermutationP",
            "FDR_Q",
        ]
    ]
    .sort_values(
        [
            "MacroVariable",
            "Outcome",
        ]
    )
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 21. Print BELONG-removed sensitivity
# ============================================================

print("\n")
print("=" * 120)
print("NO-BELONG13 MULTIVARIATE MACRO TESTS")
print("=" * 120)


no_belong_tests = (
    global_tests[
        global_tests[
            "Profile"
        ]
        ==
        "NoBELONG13"
    ]
)


print(
    no_belong_tests[
        [
            "Outcome",
            "MacroVariable",
            "VariableType",
            "N_systems",
            "N_groups",
            "MinimumGroupN",
            "F",
            "R2",
            "PermutationP",
            "FDR_Q",
        ]
    ]
    .sort_values(
        [
            "MacroVariable",
            "Outcome",
        ]
    )
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 22. Dispersion diagnostics
# ============================================================

print("\n")
print("=" * 120)
print("CATEGORICAL DISPERSION DIAGNOSTICS")
print("=" * 120)


print(
    dispersion_tests
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 23. PCA macro relationships
# ============================================================

print("\n")
print("=" * 120)
print("STRONGEST PCA-MACRO CORRELATIONS")
print("=" * 120)


print(
    pca_macro
    .assign(
        AbsRho=lambda x:
            x[
                "SpearmanRho"
            ].abs()
    )
    .sort_values(
        "AbsRho",
        ascending=False,
    )
    .head(
        30
    )[
        [
            "Profile",
            "Outcome",
            "MacroVariable",
            "PC",
            "N_systems",
            "SpearmanRho",
            "P",
            "FDR_Q",
        ]
    ]
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 24. Group-center differences
# ============================================================

print("\n")
print("=" * 120)
print("INCOME-GROUP CENTER DISTANCES — FULL14")
print("=" * 120)


print(
    group_center_distances[
        (
            group_center_distances[
                "Profile"
            ]
            ==
            "Full14"
        )
        &
        (
            group_center_distances[
                "MacroVariable"
            ]
            ==
            "IncomeGroup_FY2019"
        )
    ]
    .sort_values(
        [
            "Outcome",
            "CenterAitchisonDistance",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 25. Final status
# ============================================================

print("\n")
print("=" * 120)
print("MACRO-CONTEXT STAGE COMPLETE")
print("=" * 120)


print(
    "Macro systems:",
    len(
        macro_metadata
    )
)


print(
    "API errors:",
    len(
        fetch_errors
    )
)


print(
    "Full-composition tests:",
    len(
        global_tests
    )
)


print(
    "IMPORTANT: do not interpret clusters yet."
)



FETCHING WORLD BANK MACRO DATA
[1/75] ALB ... OK
[2/75] ARE ... OK
[3/75] ARG ... OK
[4/75] AUS ... OK
[5/75] AUT ... OK
[6/75] BEL ... OK
[7/75] BGR ... OK
[8/75] BIH ... OK
[9/75] BLR ... OK
[10/75] BRA ... OK
[11/75] BRN ... OK
[12/75] CAN ... OK
[13/75] CHE ... OK
[14/75] CHL ... OK
[15/75] COL ... OK
[16/75] CRI ... OK
[17/75] CZE ... OK
[18/75] DEU ... OK
[19/75] DNK ... OK
[20/75] DOM ... OK
[21/75] ESP ... OK
[22/75] EST ... OK
[23/75] FIN ... OK
[24/75] FRA ... OK
[25/75] GBR ... OK
[26/75] GEO ... OK
[27/75] GRC ... OK
[28/75] HKG ... OK
[29/75] HRV ... OK
[30/75] HUN ... OK
[31/75] IDN ... OK
[32/75] IRL ... OK
[33/75] ISL ... OK
[34/75] ITA ... OK
[35/75] JOR ... OK
[36/75] JPN ... OK
[37/75] KAZ ... OK
[38/75] KOR ... OK
[39/75] KSV ... OK
[40/75] LTU ... OK
[41/75] LUX ... OK
[42/75] LVA ... OK
[43/75] MAC ... OK
[44/75] MAR ... OK
[45/75] MDA ... OK
[46/75] MEX ... OK
[47/75] MLT ... OK
[48/75] MNE ... OK
[49/75] MYS ... OK
[50/75] NLD ... OK
[51/75] NOR ... OK
[52/75]

: 

In [1]:
# ============================================================
# MACRO-LEVEL CONTEXT OF INTERNATIONAL HETEROGENEITY
# ============================================================
#
# PURPOSE
# -------
# Test whether variation around the shared international
# bullying-predictor backbone is systematically associated
# with macro-level characteristics of education systems.
#
#
# MACRO VARIABLES
# ---------------
# 1. 2018 GDP per capita, PPP
# 2. World Bank FY2019 income group
#    (classification in force from July 1, 2018)
# 3. World Bank geographic region
# 4. Gini index nearest to 2018 within +/- 3 years
#
#
# PRIMARY RESPONSE
# ----------------
# Complete CLR-transformed SHAP composition.
#
# Therefore the main tests examine the ENTIRE predictor
# architecture, not one feature at a time.
#
#
# STATISTICS
# ----------
# Continuous macro variables:
#     multivariate CLR regression + permutation test
#
# Categorical macro variables:
#     permutation MANOVA / PERMANOVA-equivalent test in
#     Euclidean CLR space
#
# Categorical tests additionally use PERMDISP-style
# permutation diagnostics to determine whether apparent
# differences might reflect unequal within-group dispersion.
#
#
# PRIMARY:
#     Full14
#
# SENSITIVITY:
#     NoBELONG13
#
#
# IMPORTANT
# ---------
# NO XGBOOST FITTING.
# NO CLUSTERING YET.
# ============================================================


from pathlib import Path
import time

import numpy as np
import pandas as pd
import requests

from scipy.stats import (
    spearmanr,
)


# ============================================================
# 1. Configuration
# ============================================================

RANKING_PATH = Path(
    "../results/primary_attribution/"
    "oof_shap_rankings.csv"
)


PCA_SCORE_PATH = Path(
    "../results/international_heterogeneity/"
    "system_pca_scores.csv"
)


DISTINCTIVENESS_PATH = Path(
    "../results/international_heterogeneity/"
    "system_aitchison_distinctiveness.csv"
)


OUT_DIR = Path(
    "../results/macro_context"
)


OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


FEATURES_14 = [
    "ADAPTIVITY",
    "AGE",
    "ATTLNACT",
    "BELONG",
    "DISCLIMA",
    "EDUSHORT",
    "ESCS",
    "JOYREAD",
    "MASTGOAL",
    "METASPAM",
    "METASUM",
    "ST004D01T",
    "STIMREAD",
    "UNDREM",
]


FEATURES_13 = [
    feature
    for feature in FEATURES_14
    if feature != "BELONG"
]


PROFILE_SPECS = {
    "Full14": FEATURES_14,
    "NoBELONG13": FEATURES_13,
}


OUTCOMES = [
    "BEINGBULLIED",
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


N_PERMUTATIONS = 10_000

RANDOM_SEED = 42


# ============================================================
# 2. World Bank indicators
# ============================================================

INDICATORS = {

    # GDP per capita, purchasing-power adjusted
    "GDPpcPPP":
        "NY.GDP.PCAP.PP.KD",

    # GNI per capita, Atlas method
    # Used to reconstruct FY2019 income category.
    "GNIpcAtlas":
        "NY.GNP.PCAP.CD",

    # Gini index
    "Gini":
        "SI.POV.GINI",
}


# ============================================================
# 3. PISA -> World Bank code mapping
# ============================================================
#
# Most PISA three-letter codes already match World Bank codes.
#
# Kosovo:
#     PISA = KSV
#     World Bank = XKX
#
# QAZ:
#     Baku/Azerbaijan PISA entity
#     not identical to national Azerbaijan
#
# QCI:
#     B-S-J-Z (China)
#     not identical to national China
#
# TAP:
#     Chinese Taipei
#     not available as a directly corresponding World Bank
#     economy through the same WDI framework.
#
# Those three are therefore NOT assigned national proxies in
# the primary macro-context analysis.
#
# ============================================================

SPECIAL_WB_MAPPING = {
    "KSV": "XK",
    "QAZ": None,
    "QCI": None,
    "TAP": None,
}


SPECIAL_REASON = {

    "QAZ":
        "PISA subnational Baku/Azerbaijan entity; "
        "national Azerbaijan macro data not used as primary proxy.",

    "QCI":
        "PISA B-S-J-Z (China) entity; "
        "national China macro data not used as primary proxy.",

    "TAP":
        "Chinese Taipei lacks directly corresponding "
        "World Bank WDI economy data in this framework.",
}


# ============================================================
# 4. Load analytical outputs
# ============================================================

rankings = pd.read_csv(
    RANKING_PATH
)


pca_scores = pd.read_csv(
    PCA_SCORE_PATH
)


distinctiveness = pd.read_csv(
    DISTINCTIVENESS_PATH
)


all_systems = sorted(
    rankings[
        "Country"
    ]
    .unique()
)


# ============================================================
# 5. World Bank API helpers
# ============================================================

SESSION = requests.Session()


SESSION.headers.update(
    {
        "User-Agent":
            "PISA-2018-Bullying-Research/1.0"
    }
)


def wb_get_json(
    url,
    params=None,
    attempts=4,
):

    last_error = None


    for attempt in range(
        attempts
    ):

        try:

            response = SESSION.get(
                url,
                params=params,
                timeout=30,
            )


            response.raise_for_status()


            payload = response.json()


            return payload


        except Exception as e:

            last_error = e


            time.sleep(
                1.5
                *
                (
                    attempt
                    +
                    1
                )
            )


    raise RuntimeError(
        f"World Bank request failed: "
        f"{last_error}"
    )


def fetch_indicator(
    wb_code,
    indicator,
    start_year,
    end_year,
):

    url = (
        "https://api.worldbank.org/v2/"
        f"country/{wb_code}/"
        f"indicator/{indicator}"
    )


    payload = wb_get_json(
        url,
        params={
            "format": "json",
            "date":
                f"{start_year}:{end_year}",
            "per_page": 100,
        },
    )


    if (
        not isinstance(
            payload,
            list,
        )
        or
        len(
            payload
        )
        < 2
        or
        payload[
            1
        ]
        is None
    ):

        return pd.DataFrame(
            columns=[
                "Year",
                "Value",
            ]
        )


    rows = []


    for item in payload[
        1
    ]:

        value = item.get(
            "value"
        )


        year = item.get(
            "date"
        )


        if (
            value is None
            or
            year is None
        ):

            continue


        rows.append(
            {
                "Year":
                    int(
                        year
                    ),

                "Value":
                    float(
                        value
                    ),
            }
        )


    return pd.DataFrame(
        rows
    )


def fetch_country_metadata(
    wb_code,
):

    url = (
        "https://api.worldbank.org/v2/"
        f"country/{wb_code}"
    )


    payload = wb_get_json(
        url,
        params={
            "format": "json",
            "per_page": 10,
        },
    )


    if (
        not isinstance(
            payload,
            list,
        )
        or
        len(
            payload
        )
        < 2
        or
        not payload[
            1
        ]
    ):

        return {
            "WB_Name": None,
            "WB_Region": None,
        }


    record = payload[
        1
    ][
        0
    ]


    return {

        "WB_Name":
            record.get(
                "name"
            ),

        "WB_Region":
            (
                record.get(
                    "region",
                    {}
                )
                .get(
                    "value"
                )
            ),
    }


# ============================================================
# 6. Reconstruct FY2019 World Bank income category
#
# Classification released July 1, 2018.
#
# Based on 2017 Atlas-method GNI per capita.
#
# Thresholds:
#
# Low                 <= 995
# Lower-middle         996–3,895
# Upper-middle       3,896–12,055
# High               > 12,055
#
# NOTE:
# Current WDI historical GNI values can be revised.
# We label this explicitly as a reconstructed category.
#
# If income-group findings become manuscript-critical,
# we will verify against the World Bank historical
# classification workbook before publication.
# ============================================================

def reconstruct_income_group_fy2019(
    gni,
):

    if (
        pd.isna(
            gni
        )
        or
        not np.isfinite(
            gni
        )
    ):

        return np.nan


    if gni <= 995:

        return "Low"


    if gni <= 3895:

        return "Lower-middle"


    if gni <= 12055:

        return "Upper-middle"


    return "High"


# ============================================================
# 7. Retrieve system-level macro data
# ============================================================

metadata_rows = []

fetch_error_rows = []


print("\n")
print("=" * 110)
print("FETCHING WORLD BANK MACRO DATA")
print("=" * 110)


for number, country in enumerate(
    all_systems,
    start=1,
):

    wb_code = (
        SPECIAL_WB_MAPPING[
            country
        ]
        if country in SPECIAL_WB_MAPPING
        else country
    )


    print(
        f"[{number}/{len(all_systems)}] "
        f"{country}",
        end=" ... ",
    )


    if wb_code is None:

        metadata_rows.append(
            {
                "Country":
                    country,

                "WB_Code":
                    np.nan,

                "WB_Name":
                    np.nan,

                "WB_Region":
                    np.nan,

                "GDPpcPPP_2018":
                    np.nan,

                "LogGDPpcPPP_2018":
                    np.nan,

                "GNIpcAtlas_2017":
                    np.nan,

                "IncomeGroup_FY2019":
                    np.nan,

                "Gini_Nearest2018":
                    np.nan,

                "Gini_Year":
                    np.nan,

                "PrimaryMacroEligible":
                    False,

                "ExclusionReason":
                    SPECIAL_REASON.get(
                        country
                    ),
            }
        )


        print(
            "SPECIAL ENTITY — excluded from primary macro mapping"
        )


        continue


    try:

        country_meta = (
            fetch_country_metadata(
                wb_code
            )
        )


        # ----------------------------------------------------
        # 2018 GDP per capita PPP
        # ----------------------------------------------------

        gdp_data = fetch_indicator(
            wb_code,
            INDICATORS[
                "GDPpcPPP"
            ],
            2018,
            2018,
        )


        if len(
            gdp_data
        ) > 0:

            gdp_2018 = (
                gdp_data[
                    "Value"
                ]
                .iloc[
                    0
                ]
            )


        else:

            gdp_2018 = np.nan


        # ----------------------------------------------------
        # 2017 GNI per capita Atlas
        # ----------------------------------------------------

        gni_data = fetch_indicator(
            wb_code,
            INDICATORS[
                "GNIpcAtlas"
            ],
            2017,
            2017,
        )


        if len(
            gni_data
        ) > 0:

            gni_2017 = (
                gni_data[
                    "Value"
                ]
                .iloc[
                    0
                ]
            )


        else:

            gni_2017 = np.nan


        # ----------------------------------------------------
        # Gini nearest to 2018 within +/- 3 years
        # ----------------------------------------------------

        gini_data = fetch_indicator(
            wb_code,
            INDICATORS[
                "Gini"
            ],
            2015,
            2021,
        )


        if len(
            gini_data
        ) > 0:

            gini_data[
                "DistanceFrom2018"
            ] = np.abs(
                gini_data[
                    "Year"
                ]
                -
                2018
            )


            # Tie -> earlier year preferred.
            gini_data = (
                gini_data
                .sort_values(
                    [
                        "DistanceFrom2018",
                        "Year",
                    ],
                    ascending=[
                        True,
                        True,
                    ],
                )
            )


            gini_value = (
                gini_data[
                    "Value"
                ]
                .iloc[
                    0
                ]
            )


            gini_year = int(
                gini_data[
                    "Year"
                ]
                .iloc[
                    0
                ]
            )


        else:

            gini_value = np.nan

            gini_year = np.nan


        income_group = (
            reconstruct_income_group_fy2019(
                gni_2017
            )
        )


        eligible = bool(
            pd.notna(
                country_meta[
                    "WB_Region"
                ]
            )
        )


        metadata_rows.append(
            {
                "Country":
                    country,

                "WB_Code":
                    wb_code,

                "WB_Name":
                    country_meta[
                        "WB_Name"
                    ],

                "WB_Region":
                    country_meta[
                        "WB_Region"
                    ],

                "GDPpcPPP_2018":
                    gdp_2018,

                "LogGDPpcPPP_2018":
                    (
                        np.log(
                            gdp_2018
                        )
                        if (
                            pd.notna(
                                gdp_2018
                            )
                            and
                            gdp_2018 > 0
                        )
                        else np.nan
                    ),

                "GNIpcAtlas_2017":
                    gni_2017,

                "IncomeGroup_FY2019":
                    income_group,

                "Gini_Nearest2018":
                    gini_value,

                "Gini_Year":
                    gini_year,

                "PrimaryMacroEligible":
                    eligible,

                "ExclusionReason":
                    np.nan,
            }
        )


        print(
            "OK"
        )


    except Exception as e:

        fetch_error_rows.append(
            {
                "Country":
                    country,

                "WB_Code":
                    wb_code,

                "ErrorType":
                    type(
                        e
                    ).__name__,

                "Error":
                    str(
                        e
                    ),
            }
        )


        metadata_rows.append(
            {
                "Country":
                    country,

                "WB_Code":
                    wb_code,

                "WB_Name":
                    np.nan,

                "WB_Region":
                    np.nan,

                "GDPpcPPP_2018":
                    np.nan,

                "LogGDPpcPPP_2018":
                    np.nan,

                "GNIpcAtlas_2017":
                    np.nan,

                "IncomeGroup_FY2019":
                    np.nan,

                "Gini_Nearest2018":
                    np.nan,

                "Gini_Year":
                    np.nan,

                "PrimaryMacroEligible":
                    False,

                "ExclusionReason":
                    (
                        f"API fetch error: {e}"
                    ),
            }
        )


        print(
            f"ERROR: {e}"
        )


    time.sleep(
        0.05
    )


macro_metadata = pd.DataFrame(
    metadata_rows
)


fetch_errors = pd.DataFrame(
    fetch_error_rows,
    columns=[
        "Country",
        "WB_Code",
        "ErrorType",
        "Error",
    ],
)


macro_metadata.to_csv(
    OUT_DIR
    / "system_macro_metadata.csv",
    index=False,
)


fetch_errors.to_csv(
    OUT_DIR
    / "macro_fetch_errors.csv",
    index=False,
)


# ============================================================
# 8. Build CLR attribution matrices
# ============================================================

def build_clr_matrix(
    outcome,
    features,
):

    temp = (
        rankings[
            rankings[
                "Outcome"
            ]
            ==
            outcome
        ]
    )


    systems = sorted(
        temp[
            "Country"
        ]
        .unique()
    )


    matrix = []


    for country in systems:

        vector = (
            temp[
                temp[
                    "Country"
                ]
                ==
                country
            ]
            .set_index(
                "Feature"
            )
            .loc[
                features,
                "ImportanceShare",
            ]
            .to_numpy(
                dtype=float
            )
        )


        vector = (
            vector
            /
            vector.sum()
        )


        matrix.append(
            vector
        )


    matrix = np.vstack(
        matrix
    )


    log_matrix = np.log(
        matrix
    )


    clr = (
        log_matrix
        -
        log_matrix.mean(
            axis=1,
            keepdims=True,
        )
    )


    return (
        systems,
        clr,
    )


# ============================================================
# 9. Multivariate statistics
# ============================================================

def total_ss(
    Y
):

    centered = (
        Y
        -
        Y.mean(
            axis=0
        )
    )


    return float(
        np.sum(
            centered ** 2
        )
    )


def categorical_multivariate_stat(
    Y,
    groups,
):

    Y = np.asarray(
        Y,
        dtype=float,
    )


    groups = np.asarray(
        groups
    )


    unique_groups = np.unique(
        groups
    )


    n = len(
        groups
    )


    g = len(
        unique_groups
    )


    grand = Y.mean(
        axis=0
    )


    ss_total = np.sum(
        (
            Y
            -
            grand
        ) ** 2
    )


    ss_within = 0.0


    for group in unique_groups:

        group_Y = Y[
            groups
            ==
            group
        ]


        centroid = group_Y.mean(
            axis=0
        )


        ss_within += np.sum(
            (
                group_Y
                -
                centroid
            ) ** 2
        )


    ss_between = (
        ss_total
        -
        ss_within
    )


    df_between = (
        g
        -
        1
    )


    df_within = (
        n
        -
        g
    )


    F = (
        (
            ss_between
            /
            df_between
        )
        /
        (
            ss_within
            /
            df_within
        )
    )


    R2 = (
        ss_between
        /
        ss_total
    )


    return (
        float(
            F
        ),
        float(
            R2
        ),
    )


def continuous_multivariate_stat(
    Y,
    x,
):

    Y = np.asarray(
        Y,
        dtype=float,
    )


    x = np.asarray(
        x,
        dtype=float,
    )


    x = (
        x
        -
        x.mean()
    ) / x.std(
        ddof=1
    )


    n = len(
        x
    )


    design = np.column_stack(
        [
            np.ones(
                n
            ),
            x,
        ]
    )


    beta = np.linalg.lstsq(
        design,
        Y,
        rcond=None,
    )[
        0
    ]


    fitted = (
        design
        @
        beta
    )


    residual = (
        Y
        -
        fitted
    )


    grand = Y.mean(
        axis=0
    )


    ss_total = np.sum(
        (
            Y
            -
            grand
        ) ** 2
    )


    ss_residual = np.sum(
        residual ** 2
    )


    ss_model = (
        ss_total
        -
        ss_residual
    )


    df_model = 1

    df_residual = (
        n
        -
        2
    )


    F = (
        (
            ss_model
            /
            df_model
        )
        /
        (
            ss_residual
            /
            df_residual
        )
    )


    R2 = (
        ss_model
        /
        ss_total
    )


    return (
        float(
            F
        ),
        float(
            R2
        ),
    )


# ============================================================
# 10. Permutation tests
# ============================================================

def permutation_test_categorical(
    Y,
    groups,
    seed,
):

    observed_F, observed_R2 = (
        categorical_multivariate_stat(
            Y,
            groups,
        )
    )


    rng = np.random.default_rng(
        seed
    )


    null_F = np.empty(
        N_PERMUTATIONS,
        dtype=float,
    )


    for b in range(
        N_PERMUTATIONS
    ):

        permuted = rng.permutation(
            groups
        )


        null_F[
            b
        ] = (
            categorical_multivariate_stat(
                Y,
                permuted,
            )[
                0
            ]
        )


    p_value = (
        1
        +
        np.sum(
            null_F
            >=
            observed_F
        )
    ) / (
        N_PERMUTATIONS
        +
        1
    )


    return {
        "F":
            observed_F,

        "R2":
            observed_R2,

        "PermutationP":
            float(
                p_value
            ),

        "NullFMean":
            float(
                null_F.mean()
            ),
    }


def permutation_test_continuous(
    Y,
    x,
    seed,
):

    observed_F, observed_R2 = (
        continuous_multivariate_stat(
            Y,
            x,
        )
    )


    rng = np.random.default_rng(
        seed
    )


    null_F = np.empty(
        N_PERMUTATIONS,
        dtype=float,
    )


    for b in range(
        N_PERMUTATIONS
    ):

        permuted = rng.permutation(
            x
        )


        null_F[
            b
        ] = (
            continuous_multivariate_stat(
                Y,
                permuted,
            )[
                0
            ]
        )


    p_value = (
        1
        +
        np.sum(
            null_F
            >=
            observed_F
        )
    ) / (
        N_PERMUTATIONS
        +
        1
    )


    return {
        "F":
            observed_F,

        "R2":
            observed_R2,

        "PermutationP":
            float(
                p_value
            ),

        "NullFMean":
            float(
                null_F.mean()
            ),
    }


# ============================================================
# 11. PERMDISP-style diagnostic
# ============================================================

def dispersion_stat(
    Y,
    groups,
):

    groups = np.asarray(
        groups
    )


    distances = np.empty(
        len(
            groups
        ),
        dtype=float,
    )


    unique_groups = np.unique(
        groups
    )


    for group in unique_groups:

        mask = (
            groups
            ==
            group
        )


        centroid = (
            Y[
                mask
            ]
            .mean(
                axis=0
            )
        )


        distances[
            mask
        ] = np.linalg.norm(
            Y[
                mask
            ]
            -
            centroid,
            axis=1,
        )


    grand = (
        distances.mean()
    )


    ss_between = 0.0

    ss_within = 0.0


    for group in unique_groups:

        values = distances[
            groups
            ==
            group
        ]


        mean_group = (
            values.mean()
        )


        ss_between += (
            len(
                values
            )
            *
            (
                mean_group
                -
                grand
            ) ** 2
        )


        ss_within += np.sum(
            (
                values
                -
                mean_group
            ) ** 2
        )


    df_between = (
        len(
            unique_groups
        )
        -
        1
    )


    df_within = (
        len(
            groups
        )
        -
        len(
            unique_groups
        )
    )


    F = (
        (
            ss_between
            /
            df_between
        )
        /
        (
            ss_within
            /
            df_within
        )
    )


    return float(
        F
    )


def permutation_dispersion_test(
    Y,
    groups,
    seed,
):

    observed = dispersion_stat(
        Y,
        groups,
    )


    rng = np.random.default_rng(
        seed
    )


    null = np.empty(
        N_PERMUTATIONS
    )


    for b in range(
        N_PERMUTATIONS
    ):

        permuted = rng.permutation(
            groups
        )


        null[
            b
        ] = dispersion_stat(
            Y,
            permuted,
        )


    p = (
        1
        +
        np.sum(
            null
            >=
            observed
        )
    ) / (
        N_PERMUTATIONS
        +
        1
    )


    return (
        observed,
        float(
            p
        ),
    )


# ============================================================
# 12. Benjamini-Hochberg FDR
# ============================================================

def bh_fdr(
    p_values,
):

    p = np.asarray(
        p_values,
        dtype=float,
    )


    result = np.full(
        len(
            p
        ),
        np.nan,
    )


    valid = np.isfinite(
        p
    )


    valid_p = p[
        valid
    ]


    if len(
        valid_p
    ) == 0:

        return result


    order = np.argsort(
        valid_p
    )


    ranked = valid_p[
        order
    ]


    m = len(
        ranked
    )


    adjusted = (
        ranked
        *
        m
        /
        np.arange(
            1,
            m + 1
        )
    )


    adjusted = np.minimum.accumulate(
        adjusted[
            ::-1
        ]
    )[
        ::-1
    ]


    adjusted = np.minimum(
        adjusted,
        1.0
    )


    unsorted = np.empty(
        m
    )


    unsorted[
        order
    ] = adjusted


    result[
        valid
    ] = unsorted


    return result


# ============================================================
# 13. Macro metadata coverage
# ============================================================

print("\n")
print("=" * 115)
print("MACRO METADATA COVERAGE")
print("=" * 115)


coverage_rows = []


for variable in [
    "WB_Region",
    "IncomeGroup_FY2019",
    "GDPpcPPP_2018",
    "LogGDPpcPPP_2018",
    "Gini_Nearest2018",
]:

    coverage_rows.append(
        {
            "Variable":
                variable,

            "Available":
                int(
                    macro_metadata[
                        variable
                    ]
                    .notna()
                    .sum()
                ),

            "Missing":
                int(
                    macro_metadata[
                        variable
                    ]
                    .isna()
                    .sum()
                ),

            "PctAvailable":
                100
                *
                macro_metadata[
                    variable
                ]
                .notna()
                .mean(),
        }
    )


coverage = pd.DataFrame(
    coverage_rows
)


print(
    coverage
    .round(
        2
    )
    .to_string(
        index=False
    )
)


coverage.to_csv(
    OUT_DIR
    / "macro_metadata_coverage.csv",
    index=False,
)


print("\nINCOME GROUP COUNTS")

print(
    macro_metadata[
        "IncomeGroup_FY2019"
    ]
    .value_counts(
        dropna=False
    )
    .to_string()
)


print("\nREGION COUNTS")

print(
    macro_metadata[
        "WB_Region"
    ]
    .value_counts(
        dropna=False
    )
    .to_string()
)


# ============================================================
# 14. Main full-composition tests
# ============================================================

global_test_rows = []

dispersion_rows = []


categorical_variables = [
    "IncomeGroup_FY2019",
    "WB_Region",
]


continuous_variables = [
    "LogGDPpcPPP_2018",
    "Gini_Nearest2018",
]


test_counter = 0


for profile_name, features in (
    PROFILE_SPECS.items()
):

    for outcome in OUTCOMES:

        systems, clr = (
            build_clr_matrix(
                outcome,
                features,
            )
        )


        clr_df = pd.DataFrame(
            clr,
            index=systems,
            columns=features,
        )


        # ----------------------------------------------------
        # Categorical context
        # ----------------------------------------------------

        for variable in categorical_variables:

            test_counter += 1


            metadata_temp = (
                macro_metadata[
                    [
                        "Country",
                        variable,
                    ]
                ]
                .dropna()
            )


            common = sorted(
                set(
                    clr_df.index
                )
                &
                set(
                    metadata_temp[
                        "Country"
                    ]
                )
            )


            metadata_temp = (
                metadata_temp
                .set_index(
                    "Country"
                )
                .loc[
                    common
                ]
            )


            Y = (
                clr_df
                .loc[
                    common
                ]
                .to_numpy()
            )


            groups = (
                metadata_temp[
                    variable
                ]
                .astype(
                    str
                )
                .to_numpy()
            )


            group_counts = (
                pd.Series(
                    groups
                )
                .value_counts()
            )


            if (
                len(
                    group_counts
                )
                <
                2
            ):

                continue


            result = (
                permutation_test_categorical(
                    Y,
                    groups,
                    seed=(
                        RANDOM_SEED
                        +
                        test_counter
                        *
                        100
                    ),
                )
            )


            global_test_rows.append(
                {
                    "Profile":
                        profile_name,

                    "Outcome":
                        outcome,

                    "MacroVariable":
                        variable,

                    "VariableType":
                        "Categorical",

                    "N_systems":
                        len(
                            common
                        ),

                    "N_groups":
                        len(
                            group_counts
                        ),

                    "MinimumGroupN":
                        int(
                            group_counts.min()
                        ),

                    **result,
                }
            )


            # -----------------------------------------------
            # Dispersion diagnostic
            # -----------------------------------------------

            if (
                group_counts.min()
                >=
                2
            ):

                disp_F, disp_p = (
                    permutation_dispersion_test(
                        Y,
                        groups,
                        seed=(
                            RANDOM_SEED
                            +
                            test_counter
                            *
                            100
                            +
                            1
                        ),
                    )
                )


            else:

                disp_F = np.nan

                disp_p = np.nan


            dispersion_rows.append(
                {
                    "Profile":
                        profile_name,

                    "Outcome":
                        outcome,

                    "MacroVariable":
                        variable,

                    "N_systems":
                        len(
                            common
                        ),

                    "DispersionF":
                        disp_F,

                    "DispersionPermutationP":
                        disp_p,
                }
            )


        # ----------------------------------------------------
        # Continuous context
        # ----------------------------------------------------

        for variable in continuous_variables:

            test_counter += 1


            metadata_temp = (
                macro_metadata[
                    [
                        "Country",
                        variable,
                    ]
                ]
                .dropna()
            )


            common = sorted(
                set(
                    clr_df.index
                )
                &
                set(
                    metadata_temp[
                        "Country"
                    ]
                )
            )


            metadata_temp = (
                metadata_temp
                .set_index(
                    "Country"
                )
                .loc[
                    common
                ]
            )


            Y = (
                clr_df
                .loc[
                    common
                ]
                .to_numpy()
            )


            x = (
                metadata_temp[
                    variable
                ]
                .to_numpy(
                    dtype=float
                )
            )


            if len(
                common
            ) < 10:

                continue


            result = (
                permutation_test_continuous(
                    Y,
                    x,
                    seed=(
                        RANDOM_SEED
                        +
                        test_counter
                        *
                        100
                    ),
                )
            )


            global_test_rows.append(
                {
                    "Profile":
                        profile_name,

                    "Outcome":
                        outcome,

                    "MacroVariable":
                        variable,

                    "VariableType":
                        "Continuous",

                    "N_systems":
                        len(
                            common
                        ),

                    "N_groups":
                        np.nan,

                    "MinimumGroupN":
                        np.nan,

                    **result,
                }
            )


global_tests = pd.DataFrame(
    global_test_rows
)


dispersion_tests = pd.DataFrame(
    dispersion_rows
)


# ============================================================
# 15. FDR within primary and BELONG-removed analysis families
# ============================================================

global_tests[
    "FDR_Q"
] = np.nan


for profile_name in global_tests[
    "Profile"
].unique():

    mask = (
        global_tests[
            "Profile"
        ]
        ==
        profile_name
    )


    global_tests.loc[
        mask,
        "FDR_Q"
    ] = bh_fdr(
        global_tests.loc[
            mask,
            "PermutationP"
        ]
    )


# ============================================================
# 16. Group compositional centers
# ============================================================

group_center_rows = []

group_center_distance_rows = []


for profile_name, features in (
    PROFILE_SPECS.items()
):

    for outcome in OUTCOMES:

        systems, clr = (
            build_clr_matrix(
                outcome,
                features,
            )
        )


        clr_df = pd.DataFrame(
            clr,
            index=systems,
            columns=features,
        )


        for variable in categorical_variables:

            metadata_temp = (
                macro_metadata[
                    [
                        "Country",
                        variable,
                    ]
                ]
                .dropna()
                .set_index(
                    "Country"
                )
            )


            common = sorted(
                set(
                    clr_df.index
                )
                &
                set(
                    metadata_temp.index
                )
            )


            if len(
                common
            ) == 0:

                continue


            group_labels = (
                metadata_temp
                .loc[
                    common,
                    variable
                ]
                .astype(
                    str
                )
            )


            centers = {}


            for group in sorted(
                group_labels.unique()
            ):

                group_systems = (
                    group_labels[
                        group_labels
                        ==
                        group
                    ]
                    .index
                    .tolist()
                )


                center_clr = (
                    clr_df
                    .loc[
                        group_systems
                    ]
                    .mean(
                        axis=0
                    )
                    .to_numpy()
                )


                centers[
                    group
                ] = center_clr


                center_comp = np.exp(
                    center_clr
                )


                center_comp = (
                    center_comp
                    /
                    center_comp.sum()
                )


                for feature_index, feature in enumerate(
                    features
                ):

                    group_center_rows.append(
                        {
                            "Profile":
                                profile_name,

                            "Outcome":
                                outcome,

                            "MacroVariable":
                                variable,

                            "Group":
                                group,

                            "N_systems":
                                len(
                                    group_systems
                                ),

                            "Feature":
                                feature,

                            "CenterShare":
                                center_comp[
                                    feature_index
                                ],
                        }
                    )


            groups = sorted(
                centers.keys()
            )


            for i in range(
                len(
                    groups
                )
            ):

                for j in range(
                    i + 1,
                    len(
                        groups
                    )
                ):

                    group_a = groups[
                        i
                    ]

                    group_b = groups[
                        j
                    ]


                    distance = np.linalg.norm(
                        centers[
                            group_a
                        ]
                        -
                        centers[
                            group_b
                        ]
                    )


                    group_center_distance_rows.append(
                        {
                            "Profile":
                                profile_name,

                            "Outcome":
                                outcome,

                            "MacroVariable":
                                variable,

                            "Group1":
                                group_a,

                            "Group2":
                                group_b,

                            "CenterAitchisonDistance":
                                float(
                                    distance
                                ),
                        }
                    )


group_centers = pd.DataFrame(
    group_center_rows
)


group_center_distances = pd.DataFrame(
    group_center_distance_rows
)


# ============================================================
# 17. PCA correlations with continuous macro variables
#
# SECONDARY / INTERPRETIVE.
#
# The formal test above uses the complete CLR composition.
# These correlations help explain which PCA gradients relate
# to development/inequality.
# ============================================================

pca_macro_rows = []


for profile_name in [
    "Full14",
    "NoBELONG13",
]:

    for outcome in OUTCOMES:

        temp_scores = (
            pca_scores[
                (
                    pca_scores[
                        "Profile"
                    ]
                    ==
                    profile_name
                )
                &
                (
                    pca_scores[
                        "Outcome"
                    ]
                    ==
                    outcome
                )
            ]
        )


        merged = (
            temp_scores
            .merge(
                macro_metadata,
                on="Country",
                how="left",
                validate="one_to_one",
            )
        )


        for variable in continuous_variables:

            for pc in [
                "PC1",
                "PC2",
                "PC3",
            ]:

                valid = (
                    merged[
                        variable
                    ]
                    .notna()
                    &
                    merged[
                        pc
                    ]
                    .notna()
                )


                if valid.sum() < 10:

                    continue


                rho, p = spearmanr(
                    merged.loc[
                        valid,
                        variable
                    ],
                    merged.loc[
                        valid,
                        pc
                    ],
                )


                pca_macro_rows.append(
                    {
                        "Profile":
                            profile_name,

                        "Outcome":
                            outcome,

                        "MacroVariable":
                            variable,

                        "PC":
                            pc,

                        "N_systems":
                            int(
                                valid.sum()
                            ),

                        "SpearmanRho":
                            rho,

                        "P":
                            p,
                    }
                )


pca_macro = pd.DataFrame(
    pca_macro_rows
)


pca_macro[
    "FDR_Q"
] = bh_fdr(
    pca_macro[
        "P"
    ]
)


# ============================================================
# 18. Distinctiveness vs macro context
# ============================================================

distinctiveness_macro_rows = []


merged_distinctiveness = (
    distinctiveness
    .merge(
        macro_metadata,
        on="Country",
        how="left",
        validate="many_to_one",
    )
)


for profile_name in [
    "Full14",
    "NoBELONG13",
]:

    for outcome in OUTCOMES:

        temp = (
            merged_distinctiveness[
                (
                    merged_distinctiveness[
                        "Profile"
                    ]
                    ==
                    profile_name
                )
                &
                (
                    merged_distinctiveness[
                        "Outcome"
                    ]
                    ==
                    outcome
                )
            ]
        )


        for variable in continuous_variables:

            valid = (
                temp[
                    variable
                ]
                .notna()
                &
                temp[
                    "AitchisonDistanceToCenter"
                ]
                .notna()
            )


            if valid.sum() < 10:

                continue


            rho, p = spearmanr(
                temp.loc[
                    valid,
                    variable
                ],
                temp.loc[
                    valid,
                    "AitchisonDistanceToCenter"
                ],
            )


            distinctiveness_macro_rows.append(
                {
                    "Profile":
                        profile_name,

                    "Outcome":
                        outcome,

                    "MacroVariable":
                        variable,

                    "N_systems":
                        int(
                            valid.sum()
                        ),

                    "SpearmanRho":
                        rho,

                    "P":
                        p,
                }
            )


distinctiveness_macro = pd.DataFrame(
    distinctiveness_macro_rows
)


distinctiveness_macro[
    "FDR_Q"
] = bh_fdr(
    distinctiveness_macro[
        "P"
    ]
)


# ============================================================
# 19. Save outputs
# ============================================================

global_tests.to_csv(
    OUT_DIR
    / "macro_multivariate_tests.csv",
    index=False,
)


dispersion_tests.to_csv(
    OUT_DIR
    / "macro_dispersion_tests.csv",
    index=False,
)


group_centers.to_csv(
    OUT_DIR
    / "macro_group_compositional_centers.csv",
    index=False,
)


group_center_distances.to_csv(
    OUT_DIR
    / "macro_group_center_distances.csv",
    index=False,
)


pca_macro.to_csv(
    OUT_DIR
    / "pca_macro_correlations.csv",
    index=False,
)


distinctiveness_macro.to_csv(
    OUT_DIR
    / "distinctiveness_macro_correlations.csv",
    index=False,
)


# ============================================================
# 20. Print primary results
# ============================================================

print("\n")
print("=" * 120)
print("PRIMARY FULL14 MULTIVARIATE MACRO TESTS")
print("=" * 120)


primary_tests = (
    global_tests[
        global_tests[
            "Profile"
        ]
        ==
        "Full14"
    ]
    .copy()
)


print(
    primary_tests[
        [
            "Outcome",
            "MacroVariable",
            "VariableType",
            "N_systems",
            "N_groups",
            "MinimumGroupN",
            "F",
            "R2",
            "PermutationP",
            "FDR_Q",
        ]
    ]
    .sort_values(
        [
            "MacroVariable",
            "Outcome",
        ]
    )
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 21. Print BELONG-removed sensitivity
# ============================================================

print("\n")
print("=" * 120)
print("NO-BELONG13 MULTIVARIATE MACRO TESTS")
print("=" * 120)


no_belong_tests = (
    global_tests[
        global_tests[
            "Profile"
        ]
        ==
        "NoBELONG13"
    ]
)


print(
    no_belong_tests[
        [
            "Outcome",
            "MacroVariable",
            "VariableType",
            "N_systems",
            "N_groups",
            "MinimumGroupN",
            "F",
            "R2",
            "PermutationP",
            "FDR_Q",
        ]
    ]
    .sort_values(
        [
            "MacroVariable",
            "Outcome",
        ]
    )
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 22. Dispersion diagnostics
# ============================================================

print("\n")
print("=" * 120)
print("CATEGORICAL DISPERSION DIAGNOSTICS")
print("=" * 120)


print(
    dispersion_tests
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 23. PCA macro relationships
# ============================================================

print("\n")
print("=" * 120)
print("STRONGEST PCA-MACRO CORRELATIONS")
print("=" * 120)


print(
    pca_macro
    .assign(
        AbsRho=lambda x:
            x[
                "SpearmanRho"
            ].abs()
    )
    .sort_values(
        "AbsRho",
        ascending=False,
    )
    .head(
        30
    )[
        [
            "Profile",
            "Outcome",
            "MacroVariable",
            "PC",
            "N_systems",
            "SpearmanRho",
            "P",
            "FDR_Q",
        ]
    ]
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 24. Group-center differences
# ============================================================

print("\n")
print("=" * 120)
print("INCOME-GROUP CENTER DISTANCES — FULL14")
print("=" * 120)


print(
    group_center_distances[
        (
            group_center_distances[
                "Profile"
            ]
            ==
            "Full14"
        )
        &
        (
            group_center_distances[
                "MacroVariable"
            ]
            ==
            "IncomeGroup_FY2019"
        )
    ]
    .sort_values(
        [
            "Outcome",
            "CenterAitchisonDistance",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 25. Final status
# ============================================================

print("\n")
print("=" * 120)
print("MACRO-CONTEXT STAGE COMPLETE")
print("=" * 120)


print(
    "Macro systems:",
    len(
        macro_metadata
    )
)


print(
    "API errors:",
    len(
        fetch_errors
    )
)


print(
    "Full-composition tests:",
    len(
        global_tests
    )
)


print(
    "IMPORTANT: do not interpret clusters yet."
)



FETCHING WORLD BANK MACRO DATA
[1/75] ALB ... OK
[2/75] ARE ... OK
[3/75] ARG ... OK
[4/75] AUS ... OK
[5/75] AUT ... OK
[6/75] BEL ... OK
[7/75] BGR ... OK
[8/75] BIH ... OK
[9/75] BLR ... OK
[10/75] BRA ... OK
[11/75] BRN ... OK
[12/75] CAN ... OK
[13/75] CHE ... OK
[14/75] CHL ... OK
[15/75] COL ... OK
[16/75] CRI ... OK
[17/75] CZE ... OK
[18/75] DEU ... OK
[19/75] DNK ... OK
[20/75] DOM ... OK
[21/75] ESP ... OK
[22/75] EST ... OK
[23/75] FIN ... OK
[24/75] FRA ... OK
[25/75] GBR ... OK
[26/75] GEO ... OK
[27/75] GRC ... OK
[28/75] HKG ... OK
[29/75] HRV ... OK
[30/75] HUN ... OK
[31/75] IDN ... OK
[32/75] IRL ... OK
[33/75] ISL ... OK
[34/75] ITA ... OK
[35/75] JOR ... OK
[36/75] JPN ... OK
[37/75] KAZ ... OK
[38/75] KOR ... OK
[39/75] KSV ... OK
[40/75] LTU ... OK
[41/75] LUX ... OK
[42/75] LVA ... OK
[43/75] MAC ... OK
[44/75] MAR ... OK
[45/75] MDA ... OK
[46/75] MEX ... OK
[47/75] MLT ... OK
[48/75] MNE ... OK
[49/75] MYS ... OK
[50/75] NLD ... OK
[51/75] NOR ... OK
[52/75]

In [1]:
# ============================================================
# INTERNATIONAL HETEROGENEITY V2
# MACRO DECOMPOSITION + UNSUPERVISED CLUSTER DISCOVERY
# ============================================================
#
# PURPOSE
# -------
# Part A:
#   Determine WHICH predictors drive the previously detected
#   GDP / income / inequality / regional differences.
#
# Part B:
#   Determine whether education systems form genuine,
#   reproducible architecture clusters.
#
# Part C:
#   Validate any discovered clusters against EXTERNAL
#   macro-level characteristics:
#
#       - World Bank income group
#       - geographic region
#       - GDP per-capita tertiles
#       - GDP per-capita quartiles
#       - Gini tertiles
#       - high-income vs middle-income
#       - GDP x inequality quadrants
#       - continuous GDP
#       - continuous Gini
#
#
# CLUSTERING IS UNSUPERVISED.
#
# Macro variables DO NOT enter the clustering algorithm.
# They are used only AFTERWARD as external validators.
#
#
# CLUSTER REPRESENTATIONS
# -----------------------
# 1. BEINGBULLIED architecture
# 2. BULLY_VERBAL architecture
# 3. BULLY_THREAT architecture
# 4. BULLY_PHYSICAL architecture
# 5. BULLY_RELATIONAL architecture
# 6. JOINT FIVE-OUTCOME architecture
#
#
# CLUSTER ALGORITHMS
# ------------------
# 1. K-means
# 2. Ward hierarchical clustering
# 3. Gaussian mixture model
#
#
# NUMBER OF CLUSTERS
# ------------------
# k = 2, 3, 4, 5, 6
#
#
# EVALUATION
# ----------
# - Silhouette score
# - Calinski-Harabasz
# - Davies-Bouldin
# - Minimum cluster size
# - Subsample stability (Adjusted Rand Index)
# - Cross-algorithm agreement
#
#
# GEOMETRY
# --------
# Clustering is performed in ILR coordinates.
#
# ILR preserves Aitchison geometry while avoiding the
# singularity of CLR coordinates.
#
#
# IMPORTANT
# ---------
# This is exploratory structure discovery.
# We DO NOT force clusters to exist.
#
# ============================================================


from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

from scipy.linalg import helmert
from scipy.stats import (
    chi2_contingency,
    spearmanr,
)

from sklearn.cluster import (
    KMeans,
    AgglomerativeClustering,
)

from sklearn.mixture import (
    GaussianMixture,
)

from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    adjusted_rand_score,
)

from sklearn.decomposition import PCA


# ============================================================
# 1. Configuration
# ============================================================

RANKING_PATH = Path(
    "../results/primary_attribution/"
    "oof_shap_rankings.csv"
)


MACRO_PATH = Path(
    "../results/macro_context/"
    "system_macro_metadata.csv"
)


OUT_DIR = Path(
    "../results/cluster_discovery"
)


OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


FEATURES = [
    "ADAPTIVITY",
    "AGE",
    "ATTLNACT",
    "BELONG",
    "DISCLIMA",
    "EDUSHORT",
    "ESCS",
    "JOYREAD",
    "MASTGOAL",
    "METASPAM",
    "METASUM",
    "ST004D01T",
    "STIMREAD",
    "UNDREM",
]


OUTCOMES = [
    "BEINGBULLIED",
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


K_VALUES = [
    2,
    3,
    4,
    5,
    6,
]


ALGORITHMS = [
    "KMeans",
    "Ward",
    "GMM",
]


N_STABILITY_REPEATS = 200

STABILITY_SUBSAMPLE_FRACTION = 0.85

N_PERMUTATIONS = 5000

RANDOM_SEED = 42


# ============================================================
# 2. Load data
# ============================================================

rankings = pd.read_csv(
    RANKING_PATH
)


macro = pd.read_csv(
    MACRO_PATH
)


rankings[
    "ImportanceShare"
] = pd.to_numeric(
    rankings[
        "ImportanceShare"
    ],
    errors="coerce",
)


# ============================================================
# 3. QA
# ============================================================

if (
    rankings[
        "ImportanceShare"
    ]
    <=
    0
).any():

    raise RuntimeError(
        "All compositions must be strictly positive."
    )


if rankings[
    "ImportanceShare"
].isna().any():

    raise RuntimeError(
        "Missing SHAP shares detected."
    )


# ============================================================
# 4. Composition helpers
# ============================================================

def close_composition(
    matrix,
):

    matrix = np.asarray(
        matrix,
        dtype=float,
    )


    return (
        matrix
        /
        matrix.sum(
            axis=1,
            keepdims=True,
        )
    )


def clr_transform(
    matrix,
):

    matrix = close_composition(
        matrix
    )


    logs = np.log(
        matrix
    )


    return (
        logs
        -
        logs.mean(
            axis=1,
            keepdims=True,
        )
    )


def ilr_transform(
    matrix,
):

    """
    Isometric log-ratio transformation.

    Euclidean distance in ILR space =
    Aitchison distance in composition space.
    """

    clr = clr_transform(
        matrix
    )


    basis = helmert(
        matrix.shape[
            1
        ],
        full=False,
    )


    return (
        clr
        @
        basis.T
    )


# ============================================================
# 5. Build architecture matrix
# ============================================================

def build_composition(
    outcome,
):

    temp = (
        rankings[
            rankings[
                "Outcome"
            ]
            ==
            outcome
        ]
    )


    systems = sorted(
        temp[
            "Country"
        ]
        .unique()
    )


    rows = []


    for country in systems:

        vector = (
            temp[
                temp[
                    "Country"
                ]
                ==
                country
            ]
            .set_index(
                "Feature"
            )
            .loc[
                FEATURES,
                "ImportanceShare",
            ]
            .to_numpy(
                dtype=float
            )
        )


        vector = (
            vector
            /
            vector.sum()
        )


        rows.append(
            vector
        )


    return (
        systems,
        np.vstack(
            rows
        ),
    )


# ============================================================
# 6. Build all outcome representations
# ============================================================

representation_store = {}


for outcome in OUTCOMES:

    systems, composition = (
        build_composition(
            outcome
        )
    )


    ilr = ilr_transform(
        composition
    )


    representation_store[
        outcome
    ] = {
        "systems":
            systems,

        "composition":
            composition,

        "ilr":
            ilr,
    }


# ============================================================
# 7. JOINT five-outcome representation
#
# Only systems observed for all five outcomes.
#
# Each outcome block is normalized by its total variance
# before concatenation so that one bullying type does not
# dominate simply because its ILR coordinates have greater
# dispersion.
# ============================================================

common_systems = sorted(
    set.intersection(
        *[
            set(
                representation_store[
                    outcome
                ][
                    "systems"
                ]
            )
            for outcome
            in OUTCOMES
        ]
    )
)


joint_blocks = []


for outcome in OUTCOMES:

    result = representation_store[
        outcome
    ]


    index_map = {
        country: i
        for i, country
        in enumerate(
            result[
                "systems"
            ]
        )
    }


    rows = np.vstack(
        [
            result[
                "ilr"
            ][
                index_map[
                    country
                ]
            ]
            for country
            in common_systems
        ]
    )


    total_variance = np.sum(
        np.var(
            rows,
            axis=0,
            ddof=1,
        )
    )


    if total_variance <= 0:

        raise RuntimeError(
            f"Zero ILR variance for {outcome}"
        )


    rows_scaled = (
        rows
        /
        np.sqrt(
            total_variance
        )
    )


    joint_blocks.append(
        rows_scaled
    )


joint_matrix = np.hstack(
    joint_blocks
)


representation_store[
    "JOINT5"
] = {
    "systems":
        common_systems,

    "ilr":
        joint_matrix,
}


# ============================================================
# 8. Macro strata / validators
# ============================================================

macro[
    "GDP_Tertile"
] = pd.qcut(
    macro[
        "GDPpcPPP_2018"
    ],
    q=3,
    labels=[
        "GDP_Lower",
        "GDP_Middle",
        "GDP_Upper",
    ],
    duplicates="drop",
)


macro[
    "GDP_Quartile"
] = pd.qcut(
    macro[
        "GDPpcPPP_2018"
    ],
    q=4,
    labels=[
        "GDP_Q1",
        "GDP_Q2",
        "GDP_Q3",
        "GDP_Q4",
    ],
    duplicates="drop",
)


macro[
    "Gini_Tertile"
] = pd.qcut(
    macro[
        "Gini_Nearest2018"
    ],
    q=3,
    labels=[
        "Inequality_Lower",
        "Inequality_Middle",
        "Inequality_Upper",
    ],
    duplicates="drop",
)


macro[
    "IncomeBroad"
] = (
    macro[
        "IncomeGroup_FY2019"
    ]
    .replace(
        {
            "Lower-middle":
                "MiddleIncome",

            "Upper-middle":
                "MiddleIncome",

            "High":
                "HighIncome",
        }
    )
)


# ------------------------------------------------------------
# GDP × inequality quadrants
# ------------------------------------------------------------

gdp_median = (
    macro[
        "GDPpcPPP_2018"
    ]
    .median()
)


gini_median = (
    macro[
        "Gini_Nearest2018"
    ]
    .median()
)


def macro_quadrant(
    row,
):

    gdp = row[
        "GDPpcPPP_2018"
    ]


    gini = row[
        "Gini_Nearest2018"
    ]


    if (
        pd.isna(
            gdp
        )
        or
        pd.isna(
            gini
        )
    ):

        return np.nan


    development = (
        "HigherGDP"
        if gdp >= gdp_median
        else "LowerGDP"
    )


    inequality = (
        "HigherInequality"
        if gini >= gini_median
        else "LowerInequality"
    )


    return (
        development
        +
        "_"
        +
        inequality
    )


macro[
    "GDP_Gini_Quadrant"
] = macro.apply(
    macro_quadrant,
    axis=1,
)


# ============================================================
# 9. Print validator counts
# ============================================================

print("\n")
print("=" * 115)
print("EXTERNAL MACRO STRATA")
print("=" * 115)


for variable in [

    "IncomeGroup_FY2019",
    "IncomeBroad",
    "WB_Region",
    "GDP_Tertile",
    "GDP_Quartile",
    "Gini_Tertile",
    "GDP_Gini_Quadrant",

]:

    print("\n")
    print(
        variable
    )

    print("-" * 70)


    print(
        macro[
            variable
        ]
        .value_counts(
            dropna=False
        )
        .to_string()
    )


# ============================================================
# PART A
# MACRO-EFFECT DECOMPOSITION
# ============================================================


# ============================================================
# 10. Benjamini-Hochberg FDR
# ============================================================

def bh_fdr(
    p_values,
):

    p = np.asarray(
        p_values,
        dtype=float,
    )


    result = np.full(
        len(
            p
        ),
        np.nan,
    )


    valid = np.isfinite(
        p
    )


    pv = p[
        valid
    ]


    if len(
        pv
    ) == 0:

        return result


    order = np.argsort(
        pv
    )


    ranked = pv[
        order
    ]


    m = len(
        ranked
    )


    adjusted = (
        ranked
        *
        m
        /
        np.arange(
            1,
            m
            +
            1
        )
    )


    adjusted = np.minimum.accumulate(
        adjusted[
            ::-1
        ]
    )[
        ::-1
    ]


    adjusted = np.minimum(
        adjusted,
        1
    )


    unsorted = np.empty(
        m
    )


    unsorted[
        order
    ] = adjusted


    result[
        valid
    ] = unsorted


    return result


# ============================================================
# 11. Continuous predictor decomposition
#
# Each feature CLR coordinate is regressed on standardized
# macro variable.
#
# Slope > 0:
#   feature receives relatively MORE attribution as the
#   macro variable increases.
#
# Slope < 0:
#   relatively LESS attribution.
#
# These are RELATIVE compositional changes.
# ============================================================

def continuous_feature_test(
    y,
    x,
    seed,
):

    y = np.asarray(
        y,
        dtype=float,
    )


    x = np.asarray(
        x,
        dtype=float,
    )


    xz = (
        x
        -
        x.mean()
    ) / x.std(
        ddof=1
    )


    slope = np.sum(
        xz
        *
        (
            y
            -
            y.mean()
        )
    ) / np.sum(
        xz ** 2
    )


    rho, _ = spearmanr(
        x,
        y,
    )


    rng = np.random.default_rng(
        seed
    )


    null = np.empty(
        N_PERMUTATIONS
    )


    for permutation in range(
        N_PERMUTATIONS
    ):

        xp = rng.permutation(
            xz
        )


        null[
            permutation
        ] = np.sum(
            xp
            *
            (
                y
                -
                y.mean()
            )
        ) / np.sum(
            xp ** 2
        )


    p = (
        1
        +
        np.sum(
            np.abs(
                null
            )
            >=
            abs(
                slope
            )
        )
    ) / (
        N_PERMUTATIONS
        +
        1
    )


    return (
        float(
            slope
        ),
        float(
            rho
        ),
        float(
            p
        ),
    )


# ============================================================
# 12. Categorical predictor decomposition
# ============================================================

def categorical_feature_stat(
    y,
    groups,
):

    y = np.asarray(
        y,
        dtype=float,
    )


    groups = np.asarray(
        groups
    )


    grand = y.mean()


    unique_groups = np.unique(
        groups
    )


    ss_total = np.sum(
        (
            y
            -
            grand
        ) ** 2
    )


    ss_between = 0


    for group in unique_groups:

        vals = y[
            groups
            ==
            group
        ]


        ss_between += (
            len(
                vals
            )
            *
            (
                vals.mean()
                -
                grand
            ) ** 2
        )


    eta2 = (
        ss_between
        /
        ss_total
        if ss_total > 0
        else np.nan
    )


    return float(
        eta2
    )


def categorical_feature_test(
    y,
    groups,
    seed,
):

    observed = (
        categorical_feature_stat(
            y,
            groups,
        )
    )


    rng = np.random.default_rng(
        seed
    )


    null = np.empty(
        N_PERMUTATIONS
    )


    for permutation in range(
        N_PERMUTATIONS
    ):

        gp = rng.permutation(
            groups
        )


        null[
            permutation
        ] = (
            categorical_feature_stat(
                y,
                gp,
            )
        )


    p = (
        1
        +
        np.sum(
            null
            >=
            observed
        )
    ) / (
        N_PERMUTATIONS
        +
        1
    )


    return (
        observed,
        float(
            p
        ),
    )


# ============================================================
# 13. Run macro decomposition
# ============================================================

continuous_macro = [
    "LogGDPpcPPP_2018",
    "Gini_Nearest2018",
]


categorical_macro = [
    "IncomeGroup_FY2019",
    "WB_Region",
]


continuous_rows = []

categorical_rows = []

group_feature_center_rows = []


test_seed_counter = 0


for outcome in OUTCOMES:

    systems, composition = (
        build_composition(
            outcome
        )
    )


    clr = clr_transform(
        composition
    )


    clr_df = pd.DataFrame(
        clr,
        index=systems,
        columns=FEATURES,
    )


    # --------------------------------------------------------
    # Continuous macro variables
    # --------------------------------------------------------

    for variable in continuous_macro:

        meta = (
            macro[
                [
                    "Country",
                    variable,
                ]
            ]
            .dropna()
            .set_index(
                "Country"
            )
        )


        common = sorted(
            set(
                clr_df.index
            )
            &
            set(
                meta.index
            )
        )


        x = (
            meta
            .loc[
                common,
                variable
            ]
            .to_numpy(
                dtype=float
            )
        )


        family_indices = []


        for feature in FEATURES:

            test_seed_counter += 1


            y = (
                clr_df
                .loc[
                    common,
                    feature
                ]
                .to_numpy(
                    dtype=float
                )
            )


            slope, rho, p = (
                continuous_feature_test(
                    y,
                    x,
                    RANDOM_SEED
                    +
                    test_seed_counter,
                )
            )


            family_indices.append(
                len(
                    continuous_rows
                )
            )


            continuous_rows.append(
                {
                    "Outcome":
                        outcome,

                    "MacroVariable":
                        variable,

                    "Feature":
                        feature,

                    "N_systems":
                        len(
                            common
                        ),

                    "StandardizedMacroSlope":
                        slope,

                    "SpearmanRho":
                        rho,

                    "PermutationP":
                        p,
                }
            )


        family_p = [
            continuous_rows[
                i
            ][
                "PermutationP"
            ]
            for i in family_indices
        ]


        family_q = bh_fdr(
            family_p
        )


        for i, q in zip(
            family_indices,
            family_q,
        ):

            continuous_rows[
                i
            ][
                "FDR_Q"
            ] = q


    # --------------------------------------------------------
    # Categorical macro variables
    # --------------------------------------------------------

    for variable in categorical_macro:

        meta = (
            macro[
                [
                    "Country",
                    variable,
                ]
            ]
            .dropna()
            .set_index(
                "Country"
            )
        )


        common = sorted(
            set(
                clr_df.index
            )
            &
            set(
                meta.index
            )
        )


        groups = (
            meta
            .loc[
                common,
                variable
            ]
            .astype(
                str
            )
            .to_numpy()
        )


        family_indices = []


        for feature in FEATURES:

            test_seed_counter += 1


            y = (
                clr_df
                .loc[
                    common,
                    feature
                ]
                .to_numpy(
                    dtype=float
                )
            )


            eta2, p = (
                categorical_feature_test(
                    y,
                    groups,
                    RANDOM_SEED
                    +
                    test_seed_counter,
                )
            )


            family_indices.append(
                len(
                    categorical_rows
                )
            )


            categorical_rows.append(
                {
                    "Outcome":
                        outcome,

                    "MacroVariable":
                        variable,

                    "Feature":
                        feature,

                    "N_systems":
                        len(
                            common
                        ),

                    "EtaSquared":
                        eta2,

                    "PermutationP":
                        p,
                }
            )


        family_p = [
            categorical_rows[
                i
            ][
                "PermutationP"
            ]
            for i in family_indices
        ]


        family_q = bh_fdr(
            family_p
        )


        for i, q in zip(
            family_indices,
            family_q,
        ):

            categorical_rows[
                i
            ][
                "FDR_Q"
            ] = q


        # ----------------------------------------------------
        # Group-specific compositional center
        # ----------------------------------------------------

        group_series = (
            meta
            .loc[
                common,
                variable
            ]
            .astype(
                str
            )
        )


        for group in sorted(
            group_series.unique()
        ):

            group_systems = (
                group_series[
                    group_series
                    ==
                    group
                ]
                .index
                .tolist()
            )


            center_clr = (
                clr_df
                .loc[
                    group_systems
                ]
                .mean(
                    axis=0
                )
                .to_numpy()
            )


            comp = np.exp(
                center_clr
            )


            comp = (
                comp
                /
                comp.sum()
            )


            for feature_index, feature in enumerate(
                FEATURES
            ):

                group_feature_center_rows.append(
                    {
                        "Outcome":
                            outcome,

                        "MacroVariable":
                            variable,

                        "Group":
                            group,

                        "N_systems":
                            len(
                                group_systems
                            ),

                        "Feature":
                            feature,

                        "CenterShare":
                            comp[
                                feature_index
                            ],

                        "CenterCLR":
                            center_clr[
                                feature_index
                            ],
                    }
                )


continuous_decomposition = pd.DataFrame(
    continuous_rows
)


categorical_decomposition = pd.DataFrame(
    categorical_rows
)


group_feature_centers = pd.DataFrame(
    group_feature_center_rows
)


# ============================================================
# PART B
# CLUSTER DISCOVERY
# ============================================================


# ============================================================
# 14. Cluster fitting
# ============================================================

def fit_clusters(
    X,
    algorithm,
    k,
    seed,
):

    if algorithm == "KMeans":

        model = KMeans(
            n_clusters=k,
            n_init=50,
            random_state=seed,
        )


        labels = model.fit_predict(
            X
        )


    elif algorithm == "Ward":

        model = AgglomerativeClustering(
            n_clusters=k,
            linkage="ward",
        )


        labels = model.fit_predict(
            X
        )


    elif algorithm == "GMM":

        model = GaussianMixture(
            n_components=k,
            covariance_type="full",
            n_init=10,
            reg_covar=1e-6,
            random_state=seed,
        )


        labels = model.fit_predict(
            X
        )


    else:

        raise ValueError(
            algorithm
        )


    return labels


# ============================================================
# 15. Stability by repeated subsampling
#
# Compare clustering on an 85% subset against the labels from
# the full-data solution restricted to those same systems.
#
# ARI is label-permutation invariant.
# ============================================================

def cluster_stability(
    X,
    algorithm,
    k,
    full_labels,
    seed,
):

    rng = np.random.default_rng(
        seed
    )


    n = X.shape[
        0
    ]


    subset_n = max(
        int(
            np.floor(
                STABILITY_SUBSAMPLE_FRACTION
                *
                n
            )
        ),
        k
        *
        3,
    )


    ari_values = []


    for repeat in range(
        N_STABILITY_REPEATS
    ):

        subset = np.sort(
            rng.choice(
                n,
                size=subset_n,
                replace=False,
            )
        )


        try:

            subset_labels = fit_clusters(
                X[
                    subset
                ],
                algorithm,
                k,
                seed
                +
                repeat
                +
                1000,
            )


            reference_labels = (
                full_labels[
                    subset
                ]
            )


            ari = adjusted_rand_score(
                reference_labels,
                subset_labels,
            )


            ari_values.append(
                ari
            )


        except Exception:

            continue


    if len(
        ari_values
    ) == 0:

        return (
            np.nan,
            np.nan,
        )


    return (
        float(
            np.mean(
                ari_values
            )
        ),
        float(
            np.median(
                ari_values
            )
        ),
    )


# ============================================================
# 16. Evaluate clustering candidates
# ============================================================

cluster_diagnostic_rows = []

cluster_membership_rows = []

cluster_solution_store = {}


solution_counter = 0


for representation_name, result in (
    representation_store.items()
):

    systems = result[
        "systems"
    ]


    X = result[
        "ilr"
    ]


    for algorithm in ALGORITHMS:

        for k in K_VALUES:

            if k >= len(
                systems
            ):

                continue


            solution_counter += 1


            labels = fit_clusters(
                X,
                algorithm,
                k,
                RANDOM_SEED
                +
                solution_counter,
            )


            counts = np.bincount(
                labels
            )


            min_cluster_size = int(
                counts.min()
            )


            silhouette = silhouette_score(
                X,
                labels,
                metric="euclidean",
            )


            calinski = (
                calinski_harabasz_score(
                    X,
                    labels,
                )
            )


            davies = (
                davies_bouldin_score(
                    X,
                    labels,
                )
            )


            stability_mean, stability_median = (
                cluster_stability(
                    X,
                    algorithm,
                    k,
                    labels,
                    RANDOM_SEED
                    +
                    10000
                    +
                    solution_counter
                    *
                    10,
                )
            )


            solution_id = (
                f"{representation_name}"
                f"__{algorithm}"
                f"__K{k}"
            )


            cluster_solution_store[
                solution_id
            ] = {
                "Representation":
                    representation_name,

                "Algorithm":
                    algorithm,

                "K":
                    k,

                "Systems":
                    systems,

                "X":
                    X,

                "Labels":
                    labels,
            }


            cluster_diagnostic_rows.append(
                {
                    "SolutionID":
                        solution_id,

                    "Representation":
                        representation_name,

                    "Algorithm":
                        algorithm,

                    "K":
                        k,

                    "N_systems":
                        len(
                            systems
                        ),

                    "MinimumClusterSize":
                        min_cluster_size,

                    "Silhouette":
                        silhouette,

                    "CalinskiHarabasz":
                        calinski,

                    "DaviesBouldin":
                        davies,

                    "StabilityMeanARI":
                        stability_mean,

                    "StabilityMedianARI":
                        stability_median,
                }
            )


            for country, label in zip(
                systems,
                labels,
            ):

                cluster_membership_rows.append(
                    {
                        "SolutionID":
                            solution_id,

                        "Representation":
                            representation_name,

                        "Algorithm":
                            algorithm,

                        "K":
                            k,

                        "Country":
                            country,

                        "Cluster":
                            int(
                                label
                            )
                            +
                            1,
                    }
                )


cluster_diagnostics = pd.DataFrame(
    cluster_diagnostic_rows
)


cluster_membership = pd.DataFrame(
    cluster_membership_rows
)


# ============================================================
# 17. Within-representation diagnostic ranks
#
# No hard "winner" is automatically selected.
#
# Composite rank is only a convenient summary.
# ============================================================

cluster_diagnostics[
    "CompositeRank"
] = np.nan


for representation in (
    cluster_diagnostics[
        "Representation"
    ]
    .unique()
):

    mask = (
        cluster_diagnostics[
            "Representation"
        ]
        ==
        representation
    )


    temp = (
        cluster_diagnostics
        .loc[
            mask
        ]
        .copy()
    )


    temp[
        "RankSilhouette"
    ] = (
        temp[
            "Silhouette"
        ]
        .rank(
            ascending=False
        )
    )


    temp[
        "RankStability"
    ] = (
        temp[
            "StabilityMeanARI"
        ]
        .rank(
            ascending=False
        )
    )


    temp[
        "RankDB"
    ] = (
        temp[
            "DaviesBouldin"
        ]
        .rank(
            ascending=True
        )
    )


    composite = (
        temp[
            [
                "RankSilhouette",
                "RankStability",
                "RankDB",
            ]
        ]
        .mean(
            axis=1
        )
    )


    cluster_diagnostics.loc[
        temp.index,
        "CompositeRank"
    ] = composite


# ============================================================
# 18. Cross-algorithm agreement
#
# For the same representation and same k:
# do KMeans, Ward and GMM find similar partitions?
# ============================================================

algorithm_agreement_rows = []


for representation in (
    cluster_diagnostics[
        "Representation"
    ]
    .unique()
):

    for k in K_VALUES:

        relevant = {
            algorithm:
                cluster_solution_store.get(
                    f"{representation}"
                    f"__{algorithm}"
                    f"__K{k}"
                )
            for algorithm
            in ALGORITHMS
        }


        if any(
            value is None
            for value
            in relevant.values()
        ):

            continue


        for algorithm_a, algorithm_b in combinations(
            ALGORITHMS,
            2,
        ):

            labels_a = relevant[
                algorithm_a
            ][
                "Labels"
            ]


            labels_b = relevant[
                algorithm_b
            ][
                "Labels"
            ]


            ari = adjusted_rand_score(
                labels_a,
                labels_b,
            )


            algorithm_agreement_rows.append(
                {
                    "Representation":
                        representation,

                    "K":
                        k,

                    "Algorithm1":
                        algorithm_a,

                    "Algorithm2":
                        algorithm_b,

                    "AdjustedRand":
                        ari,
                }
            )


algorithm_agreement = pd.DataFrame(
    algorithm_agreement_rows
)


agreement_summary = (
    algorithm_agreement
    .groupby(
        [
            "Representation",
            "K",
        ],
        as_index=False,
    )[
        "AdjustedRand"
    ]
    .mean()
    .rename(
        columns={
            "AdjustedRand":
                "MeanCrossAlgorithmARI"
        }
    )
)


cluster_diagnostics = (
    cluster_diagnostics
    .merge(
        agreement_summary,
        on=[
            "Representation",
            "K",
        ],
        how="left",
        validate="many_to_one",
    )
)


# ============================================================
# PART C
# EXTERNAL VALIDATION
# ============================================================


# ============================================================
# 19. Permutation association:
# categorical macro variable vs cluster
# ============================================================

def cramers_v(
    cluster_labels,
    macro_labels,
):

    table = pd.crosstab(
        cluster_labels,
        macro_labels,
    )


    if (
        table.shape[
            0
        ]
        <
        2
        or
        table.shape[
            1
        ]
        <
        2
    ):

        return np.nan


    chi2 = chi2_contingency(
        table,
        correction=False,
    )[
        0
    ]


    n = table.values.sum()


    denominator = (
        min(
            table.shape[
                0
            ]
            -
            1,
            table.shape[
                1
            ]
            -
            1,
        )
    )


    if denominator <= 0:

        return np.nan


    return float(
        np.sqrt(
            (
                chi2
                /
                n
            )
            /
            denominator
        )
    )


def categorical_cluster_permutation(
    clusters,
    macro_labels,
    seed,
):

    observed = cramers_v(
        clusters,
        macro_labels,
    )


    if not np.isfinite(
        observed
    ):

        return (
            np.nan,
            np.nan,
        )


    rng = np.random.default_rng(
        seed
    )


    null = np.empty(
        N_PERMUTATIONS
    )


    macro_labels = np.asarray(
        macro_labels
    )


    for b in range(
        N_PERMUTATIONS
    ):

        permuted = rng.permutation(
            macro_labels
        )


        null[
            b
        ] = cramers_v(
            clusters,
            permuted,
        )


    p = (
        1
        +
        np.sum(
            null
            >=
            observed
        )
    ) / (
        N_PERMUTATIONS
        +
        1
    )


    return (
        float(
            observed
        ),
        float(
            p
        ),
    )


# ============================================================
# 20. Continuous macro variable vs cluster
#
# Permutation ANOVA-style R²:
# how much macro-variable variation is explained by
# cluster membership?
# ============================================================

def continuous_cluster_r2(
    values,
    clusters,
):

    values = np.asarray(
        values,
        dtype=float,
    )


    clusters = np.asarray(
        clusters
    )


    grand = values.mean()


    ss_total = np.sum(
        (
            values
            -
            grand
        ) ** 2
    )


    ss_between = 0


    for cluster in np.unique(
        clusters
    ):

        vals = values[
            clusters
            ==
            cluster
        ]


        ss_between += (
            len(
                vals
            )
            *
            (
                vals.mean()
                -
                grand
            ) ** 2
        )


    return float(
        ss_between
        /
        ss_total
        if ss_total > 0
        else np.nan
    )


def continuous_cluster_permutation(
    clusters,
    values,
    seed,
):

    observed = (
        continuous_cluster_r2(
            values,
            clusters,
        )
    )


    rng = np.random.default_rng(
        seed
    )


    null = np.empty(
        N_PERMUTATIONS
    )


    values = np.asarray(
        values,
        dtype=float,
    )


    for b in range(
        N_PERMUTATIONS
    ):

        permuted = rng.permutation(
            values
        )


        null[
            b
        ] = (
            continuous_cluster_r2(
                permuted,
                clusters,
            )
        )


    p = (
        1
        +
        np.sum(
            null
            >=
            observed
        )
    ) / (
        N_PERMUTATIONS
        +
        1
    )


    return (
        float(
            observed
        ),
        float(
            p
        ),
    )


# ============================================================
# 21. Define macro validators
# ============================================================

categorical_validators = [

    "IncomeGroup_FY2019",

    "IncomeBroad",

    "WB_Region",

    "GDP_Tertile",

    "GDP_Quartile",

    "Gini_Tertile",

    "GDP_Gini_Quadrant",

]


continuous_validators = [

    "LogGDPpcPPP_2018",

    "Gini_Nearest2018",

]


# ============================================================
# 22. Only externally validate reasonably stable solutions
#
# This does NOT claim they are "true clusters."
#
# We simply avoid spending inference on obviously unstable
# partitions.
# ============================================================

candidate_solutions = (
    cluster_diagnostics[
        (
            cluster_diagnostics[
                "MinimumClusterSize"
            ]
            >=
            5
        )
        &
        (
            cluster_diagnostics[
                "StabilityMeanARI"
            ]
            >=
            0.60
        )
    ]
    .copy()
)


external_validation_rows = []


validation_counter = 0


for _, solution_row in (
    candidate_solutions
    .iterrows()
):

    solution_id = (
        solution_row[
            "SolutionID"
        ]
    )


    membership = (
        cluster_membership[
            cluster_membership[
                "SolutionID"
            ]
            ==
            solution_id
        ][
            [
                "Country",
                "Cluster",
            ]
        ]
    )


    joined = (
        membership
        .merge(
            macro,
            on="Country",
            how="left",
            validate="one_to_one",
        )
    )


    # --------------------------------------------------------
    # Categorical validators
    # --------------------------------------------------------

    for variable in categorical_validators:

        valid = (
            joined[
                variable
            ]
            .notna()
        )


        temp = (
            joined.loc[
                valid
            ]
        )


        if (
            temp[
                variable
            ]
            .nunique()
            <
            2
        ):

            continue


        validation_counter += 1


        effect, p = (
            categorical_cluster_permutation(
                temp[
                    "Cluster"
                ]
                .to_numpy(),
                temp[
                    variable
                ]
                .astype(
                    str
                )
                .to_numpy(),
                RANDOM_SEED
                +
                validation_counter,
            )
        )


        external_validation_rows.append(
            {
                "SolutionID":
                    solution_id,

                "Representation":
                    solution_row[
                        "Representation"
                    ],

                "Algorithm":
                    solution_row[
                        "Algorithm"
                    ],

                "K":
                    solution_row[
                        "K"
                    ],

                "Validator":
                    variable,

                "ValidatorType":
                    "Categorical",

                "N_systems":
                    len(
                        temp
                    ),

                "Effect":
                    effect,

                "EffectType":
                    "CramersV",

                "PermutationP":
                    p,
            }
        )


    # --------------------------------------------------------
    # Continuous validators
    # --------------------------------------------------------

    for variable in continuous_validators:

        valid = (
            joined[
                variable
            ]
            .notna()
        )


        temp = (
            joined.loc[
                valid
            ]
        )


        if len(
            temp
        ) < 10:

            continue


        validation_counter += 1


        effect, p = (
            continuous_cluster_permutation(
                temp[
                    "Cluster"
                ]
                .to_numpy(),
                temp[
                    variable
                ]
                .to_numpy(
                    dtype=float
                ),
                RANDOM_SEED
                +
                validation_counter,
            )
        )


        external_validation_rows.append(
            {
                "SolutionID":
                    solution_id,

                "Representation":
                    solution_row[
                        "Representation"
                    ],

                "Algorithm":
                    solution_row[
                        "Algorithm"
                    ],

                "K":
                    solution_row[
                        "K"
                    ],

                "Validator":
                    variable,

                "ValidatorType":
                    "Continuous",

                "N_systems":
                    len(
                        temp
                    ),

                "Effect":
                    effect,

                "EffectType":
                    "ClusterExplainedR2",

                "PermutationP":
                    p,
            }
        )


external_validation = pd.DataFrame(
    external_validation_rows
)


# ============================================================
# 23. FDR for external validation
# ============================================================

if len(
    external_validation
) > 0:

    external_validation[
        "FDR_Q"
    ] = bh_fdr(
        external_validation[
            "PermutationP"
        ]
    )


# ============================================================
# 24. Cluster compositional profiles
#
# For interpretable per-outcome cluster solutions.
# JOINT5 profiles are intentionally not reduced to one
# 14-feature composition here because each outcome is a
# separate block.
# ============================================================

cluster_profile_rows = []


for solution_id, solution in (
    cluster_solution_store.items()
):

    representation = (
        solution[
            "Representation"
        ]
    )


    if representation == "JOINT5":

        continue


    systems = (
        solution[
            "Systems"
        ]
    )


    labels = (
        solution[
            "Labels"
        ]
    )


    original = (
        representation_store[
            representation
        ][
            "composition"
        ]
    )


    for cluster_label in np.unique(
        labels
    ):

        mask = (
            labels
            ==
            cluster_label
        )


        cluster_comp = (
            original[
                mask
            ]
        )


        # Compositional center
        clr = clr_transform(
            cluster_comp
        )


        center_clr = (
            clr.mean(
                axis=0
            )
        )


        center = np.exp(
            center_clr
        )


        center = (
            center
            /
            center.sum()
        )


        for feature_index, feature in enumerate(
            FEATURES
        ):

            cluster_profile_rows.append(
                {
                    "SolutionID":
                        solution_id,

                    "Representation":
                        representation,

                    "Algorithm":
                        solution[
                            "Algorithm"
                        ],

                    "K":
                        solution[
                            "K"
                        ],

                    "Cluster":
                        int(
                            cluster_label
                        )
                        +
                        1,

                    "N_systems":
                        int(
                            mask.sum()
                        ),

                    "Feature":
                        feature,

                    "CenterShare":
                        center[
                            feature_index
                        ],
                }
            )


cluster_profiles = pd.DataFrame(
    cluster_profile_rows
)


# ============================================================
# 25. Save all outputs
# ============================================================

continuous_decomposition.to_csv(
    OUT_DIR
    / "continuous_macro_feature_decomposition.csv",
    index=False,
)


categorical_decomposition.to_csv(
    OUT_DIR
    / "categorical_macro_feature_decomposition.csv",
    index=False,
)


group_feature_centers.to_csv(
    OUT_DIR
    / "macro_group_feature_centers.csv",
    index=False,
)


cluster_diagnostics.to_csv(
    OUT_DIR
    / "cluster_diagnostics.csv",
    index=False,
)


cluster_membership.to_csv(
    OUT_DIR
    / "cluster_memberships.csv",
    index=False,
)


algorithm_agreement.to_csv(
    OUT_DIR
    / "cluster_cross_algorithm_agreement.csv",
    index=False,
)


candidate_solutions.to_csv(
    OUT_DIR
    / "cluster_candidate_solutions.csv",
    index=False,
)


external_validation.to_csv(
    OUT_DIR
    / "cluster_external_validation.csv",
    index=False,
)


cluster_profiles.to_csv(
    OUT_DIR
    / "cluster_compositional_profiles.csv",
    index=False,
)


macro.to_csv(
    OUT_DIR
    / "macro_validators_with_derived_groups.csv",
    index=False,
)


# ============================================================
# 26. Print strongest continuous macro drivers
# ============================================================

print("\n")
print("=" * 120)
print("STRONGEST GDP / GINI FEATURE GRADIENTS")
print("=" * 120)


print(
    continuous_decomposition
    .assign(
        AbsSlope=lambda x:
            x[
                "StandardizedMacroSlope"
            ].abs()
    )
    .sort_values(
        [
            "MacroVariable",
            "AbsSlope",
        ],
        ascending=[
            True,
            False,
        ]
    )
    .groupby(
        [
            "Outcome",
            "MacroVariable",
        ],
        as_index=False,
        group_keys=False,
    )
    .head(
        5
    )[
        [
            "Outcome",
            "MacroVariable",
            "Feature",
            "StandardizedMacroSlope",
            "SpearmanRho",
            "PermutationP",
            "FDR_Q",
        ]
    ]
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 27. Strongest income / region feature decomposition
# ============================================================

print("\n")
print("=" * 120)
print("STRONGEST INCOME / REGION FEATURE DIFFERENCES")
print("=" * 120)


print(
    categorical_decomposition
    .sort_values(
        [
            "MacroVariable",
            "EtaSquared",
        ],
        ascending=[
            True,
            False,
        ]
    )
    .groupby(
        [
            "Outcome",
            "MacroVariable",
        ],
        as_index=False,
        group_keys=False,
    )
    .head(
        5
    )[
        [
            "Outcome",
            "MacroVariable",
            "Feature",
            "EtaSquared",
            "PermutationP",
            "FDR_Q",
        ]
    ]
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 28. Cluster diagnostics
# ============================================================

print("\n")
print("=" * 120)
print("TOP CLUSTER SOLUTIONS BY REPRESENTATION")
print("=" * 120)


for representation in (
    cluster_diagnostics[
        "Representation"
    ]
    .unique()
):

    print("\n")
    print(
        representation
    )

    print("-" * 100)


    temp = (
        cluster_diagnostics[
            cluster_diagnostics[
                "Representation"
            ]
            ==
            representation
        ]
        .sort_values(
            [
                "CompositeRank",
                "StabilityMeanARI",
                "Silhouette",
            ],
            ascending=[
                True,
                False,
                False,
            ]
        )
        .head(
            10
        )
    )


    print(
        temp[
            [
                "SolutionID",
                "Algorithm",
                "K",
                "MinimumClusterSize",
                "Silhouette",
                "DaviesBouldin",
                "StabilityMeanARI",
                "StabilityMedianARI",
                "MeanCrossAlgorithmARI",
                "CompositeRank",
            ]
        ]
        .round(
            4
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 29. Cross-algorithm agreement
# ============================================================

print("\n")
print("=" * 120)
print("CROSS-ALGORITHM CLUSTER AGREEMENT")
print("=" * 120)


print(
    agreement_summary
    .sort_values(
        [
            "Representation",
            "MeanCrossAlgorithmARI",
        ],
        ascending=[
            True,
            False,
        ]
    )
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 30. Stable candidate solutions
# ============================================================

print("\n")
print("=" * 120)
print("REASONABLY STABLE CLUSTER CANDIDATES")
print("=" * 120)


print(
    candidate_solutions[
        [
            "SolutionID",
            "Representation",
            "Algorithm",
            "K",
            "N_systems",
            "MinimumClusterSize",
            "Silhouette",
            "DaviesBouldin",
            "StabilityMeanARI",
            "MeanCrossAlgorithmARI",
            "CompositeRank",
        ]
    ]
    .sort_values(
        [
            "Representation",
            "CompositeRank",
        ]
    )
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 31. Strongest external cluster validation
# ============================================================

print("\n")
print("=" * 120)
print("STRONGEST EXTERNAL VALIDATION OF CLUSTERS")
print("=" * 120)


if len(
    external_validation
) == 0:

    print(
        "No sufficiently stable cluster solutions "
        "qualified for external validation."
    )


else:

    print(
        external_validation
        .sort_values(
            [
                "FDR_Q",
                "Effect",
            ],
            ascending=[
                True,
                False,
            ]
        )
        .head(
            50
        )[
            [
                "SolutionID",
                "Representation",
                "Validator",
                "ValidatorType",
                "N_systems",
                "Effect",
                "EffectType",
                "PermutationP",
                "FDR_Q",
            ]
        ]
        .round(
            5
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 32. Final output
# ============================================================

print("\n")
print("=" * 120)
print("CLUSTER-DISCOVERY STAGE COMPLETE")
print("=" * 120)


print(
    "Total cluster solutions tested:",
    len(
        cluster_diagnostics
    )
)


print(
    "Reasonably stable candidates:",
    len(
        candidate_solutions
    )
)


print(
    "Joint five-outcome systems:",
    len(
        common_systems
    )
)


print(
    "\nDo NOT choose the final cluster solution yet."
)


print(
    "We will interpret stability, geometry, "
    "macro validation, and cluster profiles together."
)



EXTERNAL MACRO STRATA


IncomeGroup_FY2019
----------------------------------------------------------------------
IncomeGroup_FY2019
High            45
Upper-middle    21
Lower-middle     6
NaN              3


IncomeBroad
----------------------------------------------------------------------
IncomeBroad
HighIncome      45
MiddleIncome    27
NaN              3


WB_Region
----------------------------------------------------------------------
WB_Region
Europe & Central Asia                                41
East Asia & Pacific                                  13
Latin America & Caribbean                            10
Middle East, North Africa, Afghanistan & Pakistan     6
NaN                                                   3
North America                                         2


GDP_Tertile
----------------------------------------------------------------------
GDP_Tertile
GDP_Lower     24
GDP_Middle    24
GDP_Upper     24
NaN            3


GDP_Quartile
--------------------------

In [2]:
# ============================================================
# CLUSTER INTERPRETATION + ROBUSTNESS
# ============================================================
#
# PURPOSE
# -------
# We have detected several potentially meaningful cluster
# structures, but we have NOT yet declared any of them to be
# genuine biological/social categories.
#
# This stage asks:
#
# 1. Which education systems belong to each candidate group?
# 2. What predictor architecture defines each group?
# 3. Which macro characteristics characterize the groups?
# 4. How confidently does each system belong to a cluster?
# 5. Are similar partitions recovered across algorithms?
# 6. Which systems are "core" members vs boundary cases?
#
#
# FOCAL SOLUTIONS
# ---------------
# A. Relational bullying KMeans K=2
#    -> strongest discrete structure
#
# B. Joint five-outcome KMeans K=3
#    -> strongest interpretable multi-outcome candidate
#
# C. Joint five-outcome KMeans K=4
#    -> competing multi-outcome candidate
#
# D. Overall bullying KMeans K=3
#    -> stable economic-development-related candidate
#
# E. Physical bullying KMeans K=2
#    -> inequality-associated candidate
#
#
# IMPORTANT
# ---------
# We are CHARACTERIZING candidate clusters.
# We are not yet making them primary manuscript claims.
# ============================================================


from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from scipy.optimize import linear_sum_assignment

from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score


# ============================================================
# 1. Paths
# ============================================================

RESULT_ROOT = Path(
    "../results"
)


CLUSTER_DIR = (
    RESULT_ROOT
    / "cluster_discovery"
)


OUT_DIR = (
    RESULT_ROOT
    / "cluster_interpretation"
)


OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


RANKING_PATH = (
    RESULT_ROOT
    / "primary_attribution"
    / "oof_shap_rankings.csv"
)


MACRO_PATH = (
    CLUSTER_DIR
    / "macro_validators_with_derived_groups.csv"
)


MEMBERSHIP_PATH = (
    CLUSTER_DIR
    / "cluster_memberships.csv"
)


PROFILE_PATH = (
    CLUSTER_DIR
    / "cluster_compositional_profiles.csv"
)


# ============================================================
# 2. Configuration
# ============================================================

FEATURES = [
    "ADAPTIVITY",
    "AGE",
    "ATTLNACT",
    "BELONG",
    "DISCLIMA",
    "EDUSHORT",
    "ESCS",
    "JOYREAD",
    "MASTGOAL",
    "METASPAM",
    "METASUM",
    "ST004D01T",
    "STIMREAD",
    "UNDREM",
]


OUTCOMES = [
    "BEINGBULLIED",
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


FOCAL_SOLUTIONS = [

    "BULLY_RELATIONAL__KMeans__K2",

    "JOINT5__KMeans__K3",

    "JOINT5__KMeans__K4",

    "BEINGBULLIED__KMeans__K3",

    "BULLY_PHYSICAL__KMeans__K2",
]


N_BOOTSTRAPS = 1000

BOOTSTRAP_FRACTION = 0.90

RANDOM_SEED = 42


# ============================================================
# 3. Load data
# ============================================================

rankings = pd.read_csv(
    RANKING_PATH
)


macro = pd.read_csv(
    MACRO_PATH
)


memberships = pd.read_csv(
    MEMBERSHIP_PATH
)


cluster_profiles_existing = pd.read_csv(
    PROFILE_PATH
)


# ============================================================
# 4. Composition helpers
# ============================================================

def close_composition(
    matrix
):

    matrix = np.asarray(
        matrix,
        dtype=float
    )


    return (
        matrix
        /
        matrix.sum(
            axis=1,
            keepdims=True
        )
    )


def clr_transform(
    matrix
):

    matrix = close_composition(
        matrix
    )


    logs = np.log(
        matrix
    )


    return (
        logs
        -
        logs.mean(
            axis=1,
            keepdims=True
        )
    )


# ============================================================
# 5. Helmert ILR transform
# ============================================================

from scipy.linalg import helmert


def ilr_transform(
    matrix
):

    matrix = close_composition(
        matrix
    )


    clr = clr_transform(
        matrix
    )


    basis = helmert(
        matrix.shape[1],
        full=False
    )


    return (
        clr
        @
        basis.T
    )


# ============================================================
# 6. Build outcome composition
# ============================================================

def build_outcome_composition(
    outcome
):

    temp = (
        rankings[
            rankings[
                "Outcome"
            ]
            ==
            outcome
        ]
    )


    systems = sorted(
        temp[
            "Country"
        ]
        .unique()
    )


    rows = []


    for country in systems:

        vector = (
            temp[
                temp[
                    "Country"
                ]
                ==
                country
            ]
            .set_index(
                "Feature"
            )
            .loc[
                FEATURES,
                "ImportanceShare"
            ]
            .to_numpy(
                dtype=float
            )
        )


        vector = (
            vector
            /
            vector.sum()
        )


        rows.append(
            vector
        )


    return (
        systems,
        np.vstack(
            rows
        )
    )


# ============================================================
# 7. Rebuild individual + JOINT representations
# ============================================================

representation_store = {}


for outcome in OUTCOMES:

    systems, comp = (
        build_outcome_composition(
            outcome
        )
    )


    representation_store[
        outcome
    ] = {
        "systems":
            systems,

        "composition":
            comp,

        "ilr":
            ilr_transform(
                comp
            ),
    }


common_systems = sorted(
    set.intersection(
        *[
            set(
                representation_store[
                    outcome
                ][
                    "systems"
                ]
            )
            for outcome
            in OUTCOMES
        ]
    )
)


joint_blocks = []


for outcome in OUTCOMES:

    result = (
        representation_store[
            outcome
        ]
    )


    index = {
        country: i
        for i, country
        in enumerate(
            result[
                "systems"
            ]
        )
    }


    block = np.vstack(
        [
            result[
                "ilr"
            ][
                index[
                    country
                ]
            ]
            for country
            in common_systems
        ]
    )


    total_variance = np.sum(
        np.var(
            block,
            axis=0,
            ddof=1
        )
    )


    block = (
        block
        /
        np.sqrt(
            total_variance
        )
    )


    joint_blocks.append(
        block
    )


representation_store[
    "JOINT5"
] = {
    "systems":
        common_systems,

    "ilr":
        np.hstack(
            joint_blocks
        ),
}


# ============================================================
# 8. Helper: parse solution
# ============================================================

def parse_solution(
    solution_id
):

    parts = solution_id.split(
        "__"
    )


    representation = parts[
        0
    ]


    algorithm = parts[
        1
    ]


    k = int(
        parts[
            2
        ].replace(
            "K",
            ""
        )
    )


    return (
        representation,
        algorithm,
        k
    )


# ============================================================
# 9. Match arbitrary bootstrap labels to reference labels
#
# Cluster labels have no inherent numeric meaning.
#
# Hungarian matching finds the label permutation that
# maximizes agreement with reference clustering.
# ============================================================

def align_labels(
    reference,
    candidate
):

    reference = np.asarray(
        reference
    )


    candidate = np.asarray(
        candidate
    )


    ref_values = np.unique(
        reference
    )


    cand_values = np.unique(
        candidate
    )


    matrix = np.zeros(
        (
            len(
                ref_values
            ),
            len(
                cand_values
            )
        ),
        dtype=int
    )


    for i, ref in enumerate(
        ref_values
    ):

        for j, cand in enumerate(
            cand_values
        ):

            matrix[
                i,
                j
            ] = np.sum(
                (
                    reference
                    ==
                    ref
                )
                &
                (
                    candidate
                    ==
                    cand
                )
            )


    row_ind, col_ind = (
        linear_sum_assignment(
            -matrix
        )
    )


    mapping = {}


    for r, c in zip(
        row_ind,
        col_ind
    ):

        mapping[
            cand_values[
                c
            ]
        ] = (
            ref_values[
                r
            ]
        )


    aligned = np.array(
        [
            mapping.get(
                value,
                value
            )
            for value
            in candidate
        ]
    )


    return aligned


# ============================================================
# 10. Bootstrap membership stability
#
# Each iteration:
#
# - sample 90% of systems
# - fit KMeans to subset
# - compare subset labels to reference labels
# - align arbitrary cluster numbers
#
# For every system we calculate:
#
#     membership probability
#
# = proportion of bootstrap appearances in which the system
#   returns to its original cluster.
#
# ============================================================

def bootstrap_membership_stability(
    X,
    systems,
    reference_labels,
    k,
    seed
):

    rng = np.random.default_rng(
        seed
    )


    n = len(
        systems
    )


    subset_n = int(
        np.floor(
            BOOTSTRAP_FRACTION
            *
            n
        )
    )


    appearances = np.zeros(
        n,
        dtype=int
    )


    matches = np.zeros(
        n,
        dtype=int
    )


    bootstrap_ari = []


    for repeat in range(
        N_BOOTSTRAPS
    ):

        subset = np.sort(
            rng.choice(
                n,
                size=subset_n,
                replace=False
            )
        )


        model = KMeans(
            n_clusters=k,
            n_init=50,
            random_state=(
                seed
                +
                repeat
                +
                100
            )
        )


        candidate = model.fit_predict(
            X[
                subset
            ]
        )


        reference_subset = (
            reference_labels[
                subset
            ]
        )


        candidate_aligned = (
            align_labels(
                reference_subset,
                candidate
            )
        )


        ari = adjusted_rand_score(
            reference_subset,
            candidate_aligned
        )


        bootstrap_ari.append(
            ari
        )


        appearances[
            subset
        ] += 1


        matches[
            subset
        ] += (
            candidate_aligned
            ==
            reference_subset
        )


    probability = (
        matches
        /
        appearances
    )


    return (
        probability,
        np.asarray(
            bootstrap_ari
        )
    )


# ============================================================
# 11. Focal solution characterization
# ============================================================

membership_confidence_rows = []

cluster_macro_rows = []

cluster_predictor_rows = []

cluster_member_rows = []

solution_summary_rows = []


seed_counter = 0


for solution_id in FOCAL_SOLUTIONS:

    (
        representation,
        algorithm,
        k
    ) = parse_solution(
        solution_id
    )


    if algorithm != "KMeans":

        raise RuntimeError(
            "Focal robustness stage expects KMeans."
        )


    stored_membership = (
        memberships[
            memberships[
                "SolutionID"
            ]
            ==
            solution_id
        ][
            [
                "Country",
                "Cluster"
            ]
        ]
        .copy()
    )


    systems = (
        representation_store[
            representation
        ][
            "systems"
        ]
    )


    X = (
        representation_store[
            representation
        ][
            "ilr"
        ]
    )


    membership_map = (
        stored_membership
        .set_index(
            "Country"
        )[
            "Cluster"
        ]
        .to_dict()
    )


    reference_labels = np.array(
        [
            membership_map[
                country
            ]
            -
            1
            for country
            in systems
        ]
    )


    seed_counter += 1


    (
        membership_probability,
        bootstrap_ari
    ) = bootstrap_membership_stability(

        X=X,

        systems=systems,

        reference_labels=reference_labels,

        k=k,

        seed=(
            RANDOM_SEED
            +
            seed_counter
            *
            10000
        ),
    )


    # --------------------------------------------------------
    # Confidence categories
    # --------------------------------------------------------

    for index, country in enumerate(
        systems
    ):

        probability = (
            membership_probability[
                index
            ]
        )


        if probability >= 0.90:

            confidence = (
                "Core"
            )


        elif probability >= 0.75:

            confidence = (
                "Stable"
            )


        elif probability >= 0.60:

            confidence = (
                "Intermediate"
            )


        else:

            confidence = (
                "Boundary"
            )


        cluster = (
            reference_labels[
                index
            ]
            +
            1
        )


        membership_confidence_rows.append(
            {
                "SolutionID":
                    solution_id,

                "Representation":
                    representation,

                "Country":
                    country,

                "Cluster":
                    cluster,

                "MembershipProbability":
                    probability,

                "Confidence":
                    confidence,
            }
        )


        cluster_member_rows.append(
            {
                "SolutionID":
                    solution_id,

                "Representation":
                    representation,

                "Cluster":
                    cluster,

                "Country":
                    country,

                "MembershipProbability":
                    probability,

                "Confidence":
                    confidence,
            }
        )


    # --------------------------------------------------------
    # Solution-level robustness
    # --------------------------------------------------------

    solution_summary_rows.append(
        {
            "SolutionID":
                solution_id,

            "Representation":
                representation,

            "K":
                k,

            "N_systems":
                len(
                    systems
                ),

            "BootstrapMeanARI":
                bootstrap_ari.mean(),

            "BootstrapMedianARI":
                np.median(
                    bootstrap_ari
                ),

            "BootstrapARI_Q025":
                np.quantile(
                    bootstrap_ari,
                    0.025
                ),

            "BootstrapARI_Q975":
                np.quantile(
                    bootstrap_ari,
                    0.975
                ),

            "MeanMembershipProbability":
                np.mean(
                    membership_probability
                ),

            "PctCore":
                100
                *
                np.mean(
                    membership_probability
                    >=
                    0.90
                ),

            "PctStableOrCore":
                100
                *
                np.mean(
                    membership_probability
                    >=
                    0.75
                ),

            "PctBoundary":
                100
                *
                np.mean(
                    membership_probability
                    <
                    0.60
                ),
        }
    )


    # --------------------------------------------------------
    # Macro characteristics by cluster
    # --------------------------------------------------------

    temp_membership = (
        pd.DataFrame(
            {
                "Country":
                    systems,

                "Cluster":
                    reference_labels
                    +
                    1,
            }
        )
        .merge(
            macro,
            on="Country",
            how="left",
            validate="one_to_one"
        )
    )


    for cluster in sorted(
        temp_membership[
            "Cluster"
        ]
        .unique()
    ):

        group = (
            temp_membership[
                temp_membership[
                    "Cluster"
                ]
                ==
                cluster
            ]
        )


        # Continuous summaries
        for variable in [
            "GDPpcPPP_2018",
            "LogGDPpcPPP_2018",
            "Gini_Nearest2018",
        ]:

            values = (
                group[
                    variable
                ]
                .dropna()
            )


            cluster_macro_rows.append(
                {
                    "SolutionID":
                        solution_id,

                    "Representation":
                        representation,

                    "Cluster":
                        cluster,

                    "MacroVariable":
                        variable,

                    "Category":
                        "Continuous",

                    "Level":
                        np.nan,

                    "N":
                        len(
                            values
                        ),

                    "Mean":
                        (
                            values.mean()
                            if len(
                                values
                            )
                            >
                            0
                            else np.nan
                        ),

                    "Median":
                        (
                            values.median()
                            if len(
                                values
                            )
                            >
                            0
                            else np.nan
                        ),

                    "Proportion":
                        np.nan,
                }
            )


        # Categorical composition
        for variable in [
            "IncomeGroup_FY2019",
            "IncomeBroad",
            "WB_Region",
            "GDP_Tertile",
            "GDP_Quartile",
            "Gini_Tertile",
            "GDP_Gini_Quadrant",
        ]:

            counts = (
                group[
                    variable
                ]
                .value_counts(
                    dropna=False
                )
            )


            denominator = len(
                group
            )


            for level, count in counts.items():

                cluster_macro_rows.append(
                    {
                        "SolutionID":
                            solution_id,

                        "Representation":
                            representation,

                        "Cluster":
                            cluster,

                        "MacroVariable":
                            variable,

                        "Category":
                            "Categorical",

                        "Level":
                            str(
                                level
                            ),

                        "N":
                            count,

                        "Mean":
                            np.nan,

                        "Median":
                            np.nan,

                        "Proportion":
                            count
                            /
                            denominator,
                    }
                )


    # --------------------------------------------------------
    # Predictor architecture by cluster
    #
    # For JOINT5:
    # characterize each bullying outcome separately.
    # --------------------------------------------------------

    outcomes_to_characterize = (
        OUTCOMES
        if representation
        ==
        "JOINT5"
        else [
            representation
        ]
    )


    for outcome in outcomes_to_characterize:

        outcome_result = (
            representation_store[
                outcome
            ]
        )


        outcome_index = {
            country: i
            for i, country
            in enumerate(
                outcome_result[
                    "systems"
                ]
            )
        }


        composition = (
            outcome_result[
                "composition"
            ]
        )


        for cluster in sorted(
            np.unique(
                reference_labels
            )
        ):

            cluster_systems = [
                systems[
                    i
                ]
                for i
                in range(
                    len(
                        systems
                    )
                )
                if reference_labels[
                    i
                ]
                ==
                cluster
            ]


            row_indices = [
                outcome_index[
                    country
                ]
                for country
                in cluster_systems
                if country
                in outcome_index
            ]


            cluster_comp = (
                composition[
                    row_indices
                ]
            )


            cluster_clr = (
                clr_transform(
                    cluster_comp
                )
            )


            center_clr = (
                cluster_clr.mean(
                    axis=0
                )
            )


            center_comp = np.exp(
                center_clr
            )


            center_comp = (
                center_comp
                /
                center_comp.sum()
            )


            # International outcome center
            all_clr = clr_transform(
                composition
            )


            international_clr = (
                all_clr.mean(
                    axis=0
                )
            )


            deviation = (
                center_clr
                -
                international_clr
            )


            for feature_index, feature in enumerate(
                FEATURES
            ):

                cluster_predictor_rows.append(
                    {
                        "SolutionID":
                            solution_id,

                        "Representation":
                            representation,

                        "Outcome":
                            outcome,

                        "Cluster":
                            cluster
                            +
                            1,

                        "N_systems":
                            len(
                                cluster_systems
                            ),

                        "Feature":
                            feature,

                        "ClusterCenterShare":
                            center_comp[
                                feature_index
                            ],

                        "CLRDeviationFromInternationalCenter":
                            deviation[
                                feature_index
                            ],
                    }
                )


# ============================================================
# 12. Convert outputs
# ============================================================

membership_confidence = pd.DataFrame(
    membership_confidence_rows
)


cluster_macro_summary = pd.DataFrame(
    cluster_macro_rows
)


cluster_predictor_profiles = pd.DataFrame(
    cluster_predictor_rows
)


cluster_members = pd.DataFrame(
    cluster_member_rows
)


solution_robustness = pd.DataFrame(
    solution_summary_rows
)


# ============================================================
# 13. Cross-solution membership agreement
#
# Compare focal partitions where system universes overlap.
# ============================================================

cross_solution_rows = []


for solution_a, solution_b in combinations(
    FOCAL_SOLUTIONS,
    2
):

    a = (
        cluster_members[
            cluster_members[
                "SolutionID"
            ]
            ==
            solution_a
        ][
            [
                "Country",
                "Cluster"
            ]
        ]
        .rename(
            columns={
                "Cluster":
                    "ClusterA"
            }
        )
    )


    b = (
        cluster_members[
            cluster_members[
                "SolutionID"
            ]
            ==
            solution_b
        ][
            [
                "Country",
                "Cluster"
            ]
        ]
        .rename(
            columns={
                "Cluster":
                    "ClusterB"
            }
        )
    )


    merged = a.merge(
        b,
        on="Country",
        how="inner"
    )


    ari = adjusted_rand_score(
        merged[
            "ClusterA"
        ],
        merged[
            "ClusterB"
        ]
    )


    cross_solution_rows.append(
        {
            "Solution1":
                solution_a,

            "Solution2":
                solution_b,

            "N_common_systems":
                len(
                    merged
                ),

            "AdjustedRand":
                ari,
        }
    )


cross_solution_agreement = pd.DataFrame(
    cross_solution_rows
)


# ============================================================
# 14. Identify strongest cluster-defining predictors
# ============================================================

cluster_driver_rows = []


for (
    solution_id,
    outcome,
    cluster
), group in (
    cluster_predictor_profiles.groupby(
        [
            "SolutionID",
            "Outcome",
            "Cluster"
        ]
    )
):

    temp = (
        group
        .copy()
    )


    temp[
        "AbsDeviation"
    ] = (
        temp[
            "CLRDeviationFromInternationalCenter"
        ]
        .abs()
    )


    temp = (
        temp
        .sort_values(
            "AbsDeviation",
            ascending=False
        )
        .head(
            5
        )
    )


    for rank, (_, row) in enumerate(
        temp.iterrows(),
        start=1
    ):

        cluster_driver_rows.append(
            {
                "SolutionID":
                    solution_id,

                "Outcome":
                    outcome,

                "Cluster":
                    cluster,

                "DriverRank":
                    rank,

                "Feature":
                    row[
                        "Feature"
                    ],

                "CLRDeviation":
                    row[
                        "CLRDeviationFromInternationalCenter"
                    ],

                "Direction":
                    (
                        "RelativeOveremphasis"
                        if row[
                            "CLRDeviationFromInternationalCenter"
                        ]
                        >
                        0
                        else
                        "RelativeUnderemphasis"
                    ),
            }
        )


cluster_drivers = pd.DataFrame(
    cluster_driver_rows
)


# ============================================================
# 15. Save outputs
# ============================================================

solution_robustness.to_csv(
    OUT_DIR
    / "focal_cluster_solution_robustness.csv",
    index=False
)


membership_confidence.to_csv(
    OUT_DIR
    / "focal_cluster_membership_confidence.csv",
    index=False
)


cluster_members.to_csv(
    OUT_DIR
    / "focal_cluster_members.csv",
    index=False
)


cluster_macro_summary.to_csv(
    OUT_DIR
    / "focal_cluster_macro_summary.csv",
    index=False
)


cluster_predictor_profiles.to_csv(
    OUT_DIR
    / "focal_cluster_predictor_profiles.csv",
    index=False
)


cluster_drivers.to_csv(
    OUT_DIR
    / "focal_cluster_top_drivers.csv",
    index=False
)


cross_solution_agreement.to_csv(
    OUT_DIR
    / "focal_cluster_cross_solution_agreement.csv",
    index=False
)


# ============================================================
# 16. Print robustness
# ============================================================

print("\n")
print("=" * 120)
print("FOCAL CLUSTER ROBUSTNESS")
print("=" * 120)


print(
    solution_robustness
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 17. Print memberships
# ============================================================

print("\n")
print("=" * 120)
print("CLUSTER MEMBERSHIP + CONFIDENCE")
print("=" * 120)


for solution_id in FOCAL_SOLUTIONS:

    print("\n")
    print(
        solution_id
    )

    print("-" * 110)


    temp = (
        cluster_members[
            cluster_members[
                "SolutionID"
            ]
            ==
            solution_id
        ]
        .sort_values(
            [
                "Cluster",
                "MembershipProbability"
            ],
            ascending=[
                True,
                False
            ]
        )
    )


    print(
        temp[
            [
                "Cluster",
                "Country",
                "MembershipProbability",
                "Confidence"
            ]
        ]
        .round(
            4
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 18. Print cluster predictor drivers
# ============================================================

print("\n")
print("=" * 120)
print("TOP PREDICTOR DRIVERS OF EACH CLUSTER")
print("=" * 120)


print(
    cluster_drivers
    .sort_values(
        [
            "SolutionID",
            "Outcome",
            "Cluster",
            "DriverRank"
        ]
    )
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 19. Print macro summaries
# ============================================================

print("\n")
print("=" * 120)
print("CLUSTER MACRO CHARACTERISTICS")
print("=" * 120)


for solution_id in FOCAL_SOLUTIONS:

    print("\n")
    print(
        solution_id
    )

    print("-" * 110)


    # Continuous
    continuous = (
        cluster_macro_summary[
            (
                cluster_macro_summary[
                    "SolutionID"
                ]
                ==
                solution_id
            )
            &
            (
                cluster_macro_summary[
                    "Category"
                ]
                ==
                "Continuous"
            )
        ]
    )


    print("\nCONTINUOUS")

    print(
        continuous[
            [
                "Cluster",
                "MacroVariable",
                "N",
                "Mean",
                "Median"
            ]
        ]
        .round(
            2
        )
        .to_string(
            index=False
        )
    )


    # Region
    region = (
        cluster_macro_summary[
            (
                cluster_macro_summary[
                    "SolutionID"
                ]
                ==
                solution_id
            )
            &
            (
                cluster_macro_summary[
                    "MacroVariable"
                ]
                ==
                "WB_Region"
            )
        ]
        .sort_values(
            [
                "Cluster",
                "Proportion"
            ],
            ascending=[
                True,
                False
            ]
        )
    )


    print("\nREGION")

    print(
        region[
            [
                "Cluster",
                "Level",
                "N",
                "Proportion"
            ]
        ]
        .round(
            3
        )
        .to_string(
            index=False
        )
    )


    # Income
    income = (
        cluster_macro_summary[
            (
                cluster_macro_summary[
                    "SolutionID"
                ]
                ==
                solution_id
            )
            &
            (
                cluster_macro_summary[
                    "MacroVariable"
                ]
                ==
                "IncomeGroup_FY2019"
            )
        ]
        .sort_values(
            [
                "Cluster",
                "Proportion"
            ],
            ascending=[
                True,
                False
            ]
        )
    )


    print("\nINCOME GROUP")

    print(
        income[
            [
                "Cluster",
                "Level",
                "N",
                "Proportion"
            ]
        ]
        .round(
            3
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 20. Print cross-solution agreement
# ============================================================

print("\n")
print("=" * 120)
print("CROSS-SOLUTION CLUSTER AGREEMENT")
print("=" * 120)


print(
    cross_solution_agreement
    .sort_values(
        "AdjustedRand",
        ascending=False
    )
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 21. Boundary systems
# ============================================================

print("\n")
print("=" * 120)
print("BOUNDARY / UNCERTAIN SYSTEMS")
print("=" * 120)


boundary = (
    membership_confidence[
        membership_confidence[
            "MembershipProbability"
        ]
        <
        0.60
    ]
    .sort_values(
        [
            "SolutionID",
            "MembershipProbability"
        ]
    )
)


if len(
    boundary
) == 0:

    print(
        "No systems below 0.60 membership probability."
    )


else:

    print(
        boundary[
            [
                "SolutionID",
                "Country",
                "Cluster",
                "MembershipProbability"
            ]
        ]
        .round(
            4
        )
        .to_string(
            index=False
        )
    )


print("\n")
print("=" * 120)
print("CLUSTER INTERPRETATION STAGE COMPLETE")
print("=" * 120)

print(
    "Do not select manuscript clusters until "
    "these outputs are interpreted together."
)



FOCAL CLUSTER ROBUSTNESS
                  SolutionID   Representation  K  N_systems  BootstrapMeanARI  BootstrapMedianARI  BootstrapARI_Q025  BootstrapARI_Q975  MeanMembershipProbability  PctCore  PctStableOrCore  PctBoundary
BULLY_RELATIONAL__KMeans__K2 BULLY_RELATIONAL  2         75            0.9462              1.0000             0.7720             1.0000                     0.9862  93.3333          98.6667       0.0000
          JOINT5__KMeans__K3           JOINT5  3         74            0.8292              0.9027             0.2317             1.0000                     0.9345  83.7838          91.8919       2.7027
          JOINT5__KMeans__K4           JOINT5  4         74            0.6373              0.6407             0.3309             0.9558                     0.8435  59.4595          71.6216      18.9189
    BEINGBULLIED__KMeans__K3     BEINGBULLIED  3         74            0.7705              0.8215             0.3150             0.9707                     0.8963  6

In [3]:
# ============================================================
# CROSS-SUBTYPE REWEIGHTING SIGNATURES
# ============================================================
#
# PURPOSE
# -------
# Identify systematic international patterns in HOW predictor
# architecture changes across bullying subtypes.
#
#
# Instead of asking:
#
#   "What does the predictor architecture look like?"
#
# we now ask:
#
#   "How does the architecture CHANGE when the outcome moves
#    from verbal -> threat -> physical -> relational bullying?"
#
#
# THIS ANALYSIS TESTS:
#
# 1. Overall subtype-reweighting intensity for every system
#
# 2. Continuous reweighting axes using PCA
#
# 3. Whether GDP, inequality, income, region, and previously
#    discovered JOINT5 architecture groups explain these
#    reweighting signatures
#
# 4. Which PREDICTORS drive those macro differences
#
# 5. Specific theoretically interpretable subtype contrasts:
#
#       Threat + Physical
#              vs
#       Verbal + Relational
#
#       Relational
#              vs
#       other three subtypes
#
#       Physical
#              vs
#       other three subtypes
#
# 6. Whether there are reproducible unsupervised
#    "reweighting signature" clusters
#
#
# IMPORTANT
# ---------
# - NO XGBoost fitting.
# - Uses frozen OOF SHAP compositions.
# - Uses CLR/Aitchison geometry.
# - Clusters are discovered WITHOUT macro variables.
#
# ============================================================


from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

from scipy.stats import (
    spearmanr,
    chi2_contingency,
)

from sklearn.decomposition import PCA

from sklearn.cluster import (
    KMeans,
    AgglomerativeClustering,
)

from sklearn.mixture import GaussianMixture

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
)


# ============================================================
# 1. Configuration
# ============================================================

RESULT_ROOT = Path(
    "../results"
)


RANKING_PATH = (
    RESULT_ROOT
    / "primary_attribution"
    / "oof_shap_rankings.csv"
)


MACRO_PATH = (
    RESULT_ROOT
    / "cluster_discovery"
    / "macro_validators_with_derived_groups.csv"
)


CLUSTER_MEMBERSHIP_PATH = (
    RESULT_ROOT
    / "cluster_discovery"
    / "cluster_memberships.csv"
)


OUT_DIR = (
    RESULT_ROOT
    / "subtype_reweighting"
)


OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


FEATURES = [
    "ADAPTIVITY",
    "AGE",
    "ATTLNACT",
    "BELONG",
    "DISCLIMA",
    "EDUSHORT",
    "ESCS",
    "JOYREAD",
    "MASTGOAL",
    "METASPAM",
    "METASUM",
    "ST004D01T",
    "STIMREAD",
    "UNDREM",
]


SUBTYPES = [
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


K_VALUES = [
    2,
    3,
    4,
    5,
]


ALGORITHMS = [
    "KMeans",
    "Ward",
    "GMM",
]


# Exploratory screening.
# Important finalists can later be confirmed with 10,000.
N_PERMUTATIONS = 5_000


N_STABILITY_REPEATS = 200

STABILITY_SUBSAMPLE = 0.85

RANDOM_SEED = 42


# ============================================================
# 2. Load data
# ============================================================

rankings = pd.read_csv(
    RANKING_PATH
)


macro = pd.read_csv(
    MACRO_PATH
)


cluster_memberships = pd.read_csv(
    CLUSTER_MEMBERSHIP_PATH
)


rankings[
    "ImportanceShare"
] = pd.to_numeric(
    rankings[
        "ImportanceShare"
    ],
    errors="coerce",
)


# ============================================================
# 3. Add existing JOINT5 K=3 architecture group
#
# This is NOT used to construct the new subtype signatures.
#
# It is only an external validator:
#
# Do systems belonging to our broad architecture groups also
# differ systematically in how their architecture changes
# across bullying forms?
# ============================================================

joint_k3 = (
    cluster_memberships[
        cluster_memberships[
            "SolutionID"
        ]
        ==
        "JOINT5__KMeans__K3"
    ][
        [
            "Country",
            "Cluster",
        ]
    ]
    .rename(
        columns={
            "Cluster":
                "JointArchitectureK3"
        }
    )
)


macro = (
    macro
    .merge(
        joint_k3,
        on="Country",
        how="left",
        validate="one_to_one",
    )
)


macro[
    "JointArchitectureK3"
] = (
    macro[
        "JointArchitectureK3"
    ]
    .astype(
        "Int64"
    )
    .astype(
        "string"
    )
)


# ============================================================
# 4. QA
# ============================================================

if (
    rankings[
        "ImportanceShare"
    ]
    <=
    0
).any():

    raise RuntimeError(
        "CLR requires strictly positive shares."
    )


if rankings[
    "ImportanceShare"
].isna().any():

    raise RuntimeError(
        "Missing SHAP importance shares."
    )


# ============================================================
# 5. Composition helpers
# ============================================================

def close_composition(
    matrix,
):

    matrix = np.asarray(
        matrix,
        dtype=float,
    )


    return (
        matrix
        /
        matrix.sum(
            axis=1,
            keepdims=True,
        )
    )


def clr_transform(
    matrix,
):

    matrix = close_composition(
        matrix
    )


    log_matrix = np.log(
        matrix
    )


    return (
        log_matrix
        -
        log_matrix.mean(
            axis=1,
            keepdims=True,
        )
    )


# ============================================================
# 6. Build each subtype composition
# ============================================================

subtype_store = {}


for outcome in SUBTYPES:

    temp = (
        rankings[
            rankings[
                "Outcome"
            ]
            ==
            outcome
        ]
    )


    systems = sorted(
        temp[
            "Country"
        ]
        .unique()
    )


    rows = []


    for country in systems:

        vector = (
            temp[
                temp[
                    "Country"
                ]
                ==
                country
            ]
            .set_index(
                "Feature"
            )
            .loc[
                FEATURES,
                "ImportanceShare"
            ]
            .to_numpy(
                dtype=float
            )
        )


        vector = (
            vector
            /
            vector.sum()
        )


        rows.append(
            vector
        )


    composition = np.vstack(
        rows
    )


    clr = clr_transform(
        composition
    )


    subtype_store[
        outcome
    ] = {

        "systems":
            systems,

        "composition":
            composition,

        "clr":
            clr,
    }


# ============================================================
# 7. Common systems across ALL four subtypes
# ============================================================

common_systems = sorted(
    set.intersection(
        *[
            set(
                subtype_store[
                    outcome
                ][
                    "systems"
                ]
            )
            for outcome
            in SUBTYPES
        ]
    )
)


print(
    "Systems with all four subtype profiles:",
    len(
        common_systems
    )
)


# ============================================================
# 8. Build tensor
#
# Shape:
#
#   system x subtype x feature
#
# ============================================================

clr_tensor = np.empty(
    (
        len(
            common_systems
        ),
        len(
            SUBTYPES
        ),
        len(
            FEATURES
        ),
    ),
    dtype=float,
)


composition_tensor = np.empty_like(
    clr_tensor
)


for subtype_index, outcome in enumerate(
    SUBTYPES
):

    result = subtype_store[
        outcome
    ]


    index_map = {
        country: i
        for i, country
        in enumerate(
            result[
                "systems"
            ]
        )
    }


    for system_index, country in enumerate(
        common_systems
    ):

        source_index = (
            index_map[
                country
            ]
        )


        clr_tensor[
            system_index,
            subtype_index,
            :
        ] = (
            result[
                "clr"
            ][
                source_index
            ]
        )


        composition_tensor[
            system_index,
            subtype_index,
            :
        ] = (
            result[
                "composition"
            ][
                source_index
            ]
        )


# ============================================================
# 9. Construct SUBTYPE-REWEIGHTING SIGNATURE
#
# For each system:
#
#   calculate mean CLR architecture across four subtypes
#
#   then subtract it from each subtype profile.
#
#
# This removes the system's general architecture and isolates
# ONLY how that architecture is reweighted by bullying type.
#
#
# Example:
#
# A system may generally emphasize gender.
#
# That alone does NOT produce a reweighting signature.
#
# What matters here is whether gender becomes relatively MORE
# important for physical/threat bullying than for relational/
# verbal bullying within that same system.
# ============================================================

system_mean_clr = (
    clr_tensor.mean(
        axis=1,
        keepdims=True,
    )
)


reweight_tensor = (
    clr_tensor
    -
    system_mean_clr
)


# Flatten:
#
# n systems x (4 subtypes * 14 features)
#
signature_matrix = (
    reweight_tensor
    .reshape(
        len(
            common_systems
        ),
        -1,
    )
)


# ============================================================
# 10. Equal-block sensitivity representation
#
# Primary signature uses the raw CLR contrasts.
#
# Sensitivity:
# normalize each subtype block so every subtype contributes
# equal total cross-system variance.
# ============================================================

equal_block_tensor = (
    reweight_tensor.copy()
)


for subtype_index in range(
    len(
        SUBTYPES
    )
):

    block = (
        equal_block_tensor[
            :,
            subtype_index,
            :
        ]
    )


    total_variance = np.sum(
        np.var(
            block,
            axis=0,
            ddof=1,
        )
    )


    if total_variance > 0:

        equal_block_tensor[
            :,
            subtype_index,
            :
        ] = (
            block
            /
            np.sqrt(
                total_variance
            )
        )


equal_block_signature = (
    equal_block_tensor.reshape(
        len(
            common_systems
        ),
        -1,
    )
)


# ============================================================
# 11. Reweighting intensity
#
# Aitchison-style RMS distance of each subtype architecture
# from that system's four-subtype compositional center.
#
# Larger value:
# predictor structure changes MORE across bullying forms.
#
# Smaller value:
# architecture stays more similar across bullying forms.
# ============================================================

reweighting_intensity = np.sqrt(
    np.mean(
        np.sum(
            reweight_tensor ** 2,
            axis=2,
        ),
        axis=1,
    )
)


intensity_df = pd.DataFrame(
    {
        "Country":
            common_systems,

        "SubtypeReweightingIntensity":
            reweighting_intensity,
    }
)


# ============================================================
# 12. Pairwise within-system subtype distances
# ============================================================

pairwise_subtype_rows = []


for system_index, country in enumerate(
    common_systems
):

    for subtype_a, subtype_b in combinations(
        range(
            len(
                SUBTYPES
            )
        ),
        2,
    ):

        distance = np.linalg.norm(
            clr_tensor[
                system_index,
                subtype_a,
                :
            ]
            -
            clr_tensor[
                system_index,
                subtype_b,
                :
            ]
        )


        pairwise_subtype_rows.append(
            {
                "Country":
                    country,

                "Subtype1":
                    SUBTYPES[
                        subtype_a
                    ],

                "Subtype2":
                    SUBTYPES[
                        subtype_b
                    ],

                "AitchisonDistance":
                    distance,
            }
        )


pairwise_subtype_distances = pd.DataFrame(
    pairwise_subtype_rows
)


# ============================================================
# 13. Feature-specific subtype variability
#
# For every system + predictor:
#
# quantify how much that predictor's RELATIVE attribution
# changes across the four bullying forms.
# ============================================================

feature_variability_rows = []


for system_index, country in enumerate(
    common_systems
):

    for feature_index, feature in enumerate(
        FEATURES
    ):

        values = (
            reweight_tensor[
                system_index,
                :,
                feature_index
            ]
        )


        feature_variability_rows.append(
            {
                "Country":
                    country,

                "Feature":
                    feature,

                "SubtypeCLR_SD":
                    np.std(
                        values,
                        ddof=1,
                    ),

                "SubtypeCLR_Range":
                    (
                        values.max()
                        -
                        values.min()
                    ),
            }
        )


feature_variability = pd.DataFrame(
    feature_variability_rows
)


# ============================================================
# 14. Interpretable subtype contrast scores
#
# CONTRAST A:
#
# Threat + Physical
#        versus
# Verbal + Relational
#
# This is especially relevant to our earlier gender finding.
#
#
# CONTRAST B:
#
# Relational
#     versus
# mean(other three)
#
#
# CONTRAST C:
#
# Physical
#     versus
# mean(other three)
# ============================================================

subtype_index = {
    subtype:
        i
    for i, subtype
    in enumerate(
        SUBTYPES
    )
}


contrast_rows = []


for system_index, country in enumerate(
    common_systems
):

    for feature_index, feature in enumerate(
        FEATURES
    ):

        verbal = clr_tensor[
            system_index,
            subtype_index[
                "BULLY_VERBAL"
            ],
            feature_index
        ]


        threat = clr_tensor[
            system_index,
            subtype_index[
                "BULLY_THREAT"
            ],
            feature_index
        ]


        physical = clr_tensor[
            system_index,
            subtype_index[
                "BULLY_PHYSICAL"
            ],
            feature_index
        ]


        relational = clr_tensor[
            system_index,
            subtype_index[
                "BULLY_RELATIONAL"
            ],
            feature_index
        ]


        tp_vs_vr = (
            (
                threat
                +
                physical
            )
            /
            2
            -
            (
                verbal
                +
                relational
            )
            /
            2
        )


        relational_vs_other = (
            relational
            -
            np.mean(
                [
                    verbal,
                    threat,
                    physical,
                ]
            )
        )


        physical_vs_other = (
            physical
            -
            np.mean(
                [
                    verbal,
                    threat,
                    relational,
                ]
            )
        )


        contrast_rows.extend(
            [
                {
                    "Country":
                        country,

                    "Feature":
                        feature,

                    "Contrast":
                        "ThreatPhysical_vs_VerbalRelational",

                    "Value":
                        tp_vs_vr,
                },

                {
                    "Country":
                        country,

                    "Feature":
                        feature,

                    "Contrast":
                        "Relational_vs_Other3",

                    "Value":
                        relational_vs_other,
                },

                {
                    "Country":
                        country,

                    "Feature":
                        feature,

                    "Contrast":
                        "Physical_vs_Other3",

                    "Value":
                        physical_vs_other,
                },
            ]
        )


contrast_scores = pd.DataFrame(
    contrast_rows
)


# ============================================================
# 15. Attach macro variables
# ============================================================

system_metadata = (
    pd.DataFrame(
        {
            "Country":
                common_systems
        }
    )
    .merge(
        macro,
        on="Country",
        how="left",
        validate="one_to_one",
    )
)


intensity_macro = (
    intensity_df
    .merge(
        system_metadata,
        on="Country",
        how="left",
        validate="one_to_one",
    )
)


# ============================================================
# 16. BH FDR
# ============================================================

def bh_fdr(
    p_values,
):

    p = np.asarray(
        p_values,
        dtype=float,
    )


    output = np.full(
        len(
            p
        ),
        np.nan,
    )


    valid = np.isfinite(
        p
    )


    pv = p[
        valid
    ]


    if len(
        pv
    ) == 0:

        return output


    order = np.argsort(
        pv
    )


    ranked = pv[
        order
    ]


    m = len(
        ranked
    )


    adjusted = (
        ranked
        *
        m
        /
        np.arange(
            1,
            m
            +
            1
        )
    )


    adjusted = np.minimum.accumulate(
        adjusted[
            ::-1
        ]
    )[
        ::-1
    ]


    adjusted = np.minimum(
        adjusted,
        1.0
    )


    unsorted = np.empty(
        m
    )


    unsorted[
        order
    ] = adjusted


    output[
        valid
    ] = unsorted


    return output


# ============================================================
# 17. PCA of subtype-reweighting signatures
# ============================================================

pca = PCA()


pca_scores_array = pca.fit_transform(
    signature_matrix
)


components = (
    pca.components_
    .copy()
)


# ------------------------------------------------------------
# Deterministic sign orientation
# ------------------------------------------------------------

for pc_index in range(
    components.shape[
        0
    ]
):

    strongest = np.argmax(
        np.abs(
            components[
                pc_index
            ]
        )
    )


    if (
        components[
            pc_index,
            strongest
        ]
        <
        0
    ):

        components[
            pc_index
        ] *= -1


        pca_scores_array[
            :,
            pc_index
        ] *= -1


explained = (
    pca.explained_variance_ratio_
)


cumulative = np.cumsum(
    explained
)


components_80 = int(
    np.argmax(
        cumulative
        >=
        0.80
    )
    +
    1
)


components_90 = int(
    np.argmax(
        cumulative
        >=
        0.90
    )
    +
    1
)


pca_score_rows = []


for system_index, country in enumerate(
    common_systems
):

    row = {
        "Country":
            country
    }


    for pc_index in range(
        pca_scores_array.shape[
            1
        ]
    ):

        row[
            f"PC{pc_index + 1}"
        ] = (
            pca_scores_array[
                system_index,
                pc_index
            ]
        )


    pca_score_rows.append(
        row
    )


reweight_pca_scores = pd.DataFrame(
    pca_score_rows
)


# ============================================================
# 18. PCA loading interpretation
#
# Each variable is:
#
#     subtype : predictor
#
# ============================================================

variable_names = []


for subtype in SUBTYPES:

    for feature in FEATURES:

        variable_names.append(
            f"{subtype}:{feature}"
        )


loading_rows = []


for pc_index in range(
    components.shape[
        0
    ]
):

    for variable_index, variable in enumerate(
        variable_names
    ):

        loading_rows.append(
            {
                "PC":
                    pc_index
                    +
                    1,

                "Variable":
                    variable,

                "Coefficient":
                    components[
                        pc_index,
                        variable_index
                    ],
            }
        )


pca_loadings = pd.DataFrame(
    loading_rows
)


# ============================================================
# 19. Multivariate statistics
# ============================================================

def continuous_multivariate_stat(
    Y,
    x,
):

    Y = np.asarray(
        Y,
        dtype=float,
    )


    x = np.asarray(
        x,
        dtype=float,
    )


    x = (
        x
        -
        x.mean()
    ) / x.std(
        ddof=1
    )


    n = len(
        x
    )


    design = np.column_stack(
        [
            np.ones(
                n
            ),
            x,
        ]
    )


    beta = np.linalg.lstsq(
        design,
        Y,
        rcond=None,
    )[
        0
    ]


    fitted = (
        design
        @
        beta
    )


    residual = (
        Y
        -
        fitted
    )


    grand = Y.mean(
        axis=0
    )


    ss_total = np.sum(
        (
            Y
            -
            grand
        ) ** 2
    )


    ss_resid = np.sum(
        residual ** 2
    )


    ss_model = (
        ss_total
        -
        ss_resid
    )


    F = (
        ss_model
        /
        (
            ss_resid
            /
            (
                n
                -
                2
            )
        )
    )


    R2 = (
        ss_model
        /
        ss_total
    )


    return (
        float(
            F
        ),
        float(
            R2
        ),
    )


def categorical_multivariate_stat(
    Y,
    groups,
):

    Y = np.asarray(
        Y,
        dtype=float,
    )


    groups = np.asarray(
        groups
    )


    unique_groups = np.unique(
        groups
    )


    grand = Y.mean(
        axis=0
    )


    ss_total = np.sum(
        (
            Y
            -
            grand
        ) ** 2
    )


    ss_within = 0.0


    for group in unique_groups:

        group_Y = Y[
            groups
            ==
            group
        ]


        center = group_Y.mean(
            axis=0
        )


        ss_within += np.sum(
            (
                group_Y
                -
                center
            ) ** 2
        )


    ss_between = (
        ss_total
        -
        ss_within
    )


    g = len(
        unique_groups
    )


    n = len(
        groups
    )


    F = (
        (
            ss_between
            /
            (
                g
                -
                1
            )
        )
        /
        (
            ss_within
            /
            (
                n
                -
                g
            )
        )
    )


    R2 = (
        ss_between
        /
        ss_total
    )


    return (
        float(
            F
        ),
        float(
            R2
        ),
    )


# ============================================================
# 20. Permutation tests
# ============================================================

def permutation_continuous(
    Y,
    x,
    seed,
):

    observed_F, observed_R2 = (
        continuous_multivariate_stat(
            Y,
            x,
        )
    )


    rng = np.random.default_rng(
        seed
    )


    null = np.empty(
        N_PERMUTATIONS
    )


    for b in range(
        N_PERMUTATIONS
    ):

        permuted = rng.permutation(
            x
        )


        null[
            b
        ] = (
            continuous_multivariate_stat(
                Y,
                permuted,
            )[
                0
            ]
        )


    p = (
        1
        +
        np.sum(
            null
            >=
            observed_F
        )
    ) / (
        N_PERMUTATIONS
        +
        1
    )


    return (
        observed_F,
        observed_R2,
        float(
            p
        ),
    )


def permutation_categorical(
    Y,
    groups,
    seed,
):

    observed_F, observed_R2 = (
        categorical_multivariate_stat(
            Y,
            groups,
        )
    )


    rng = np.random.default_rng(
        seed
    )


    null = np.empty(
        N_PERMUTATIONS
    )


    for b in range(
        N_PERMUTATIONS
    ):

        permuted = rng.permutation(
            groups
        )


        null[
            b
        ] = (
            categorical_multivariate_stat(
                Y,
                permuted,
            )[
                0
            ]
        )


    p = (
        1
        +
        np.sum(
            null
            >=
            observed_F
        )
    ) / (
        N_PERMUTATIONS
        +
        1
    )


    return (
        observed_F,
        observed_R2,
        float(
            p
        ),
    )


# ============================================================
# 21. Global macro tests of COMPLETE reweighting signature
# ============================================================

continuous_validators = [
    "LogGDPpcPPP_2018",
    "Gini_Nearest2018",
]


categorical_validators = [
    "IncomeGroup_FY2019",
    "IncomeBroad",
    "WB_Region",
    "GDP_Tertile",
    "GDP_Quartile",
    "Gini_Tertile",
    "GDP_Gini_Quadrant",
    "JointArchitectureK3",
]


signature_representations = {

    "PrimaryRawCLR":
        signature_matrix,

    "EqualSubtypeBlocks":
        equal_block_signature,
}


global_test_rows = []

test_counter = 0


for representation_name, Y_all in (
    signature_representations.items()
):

    # --------------------------------------------------------
    # Continuous
    # --------------------------------------------------------

    for variable in continuous_validators:

        valid = (
            system_metadata[
                variable
            ]
            .notna()
        )


        indices = np.flatnonzero(
            valid.to_numpy()
        )


        Y = (
            Y_all[
                indices
            ]
        )


        x = (
            system_metadata.loc[
                valid,
                variable
            ]
            .to_numpy(
                dtype=float
            )
        )


        test_counter += 1


        F, R2, p = (
            permutation_continuous(
                Y,
                x,
                RANDOM_SEED
                +
                test_counter,
            )
        )


        global_test_rows.append(
            {
                "Representation":
                    representation_name,

                "Validator":
                    variable,

                "Type":
                    "Continuous",

                "N_systems":
                    len(
                        x
                    ),

                "F":
                    F,

                "R2":
                    R2,

                "PermutationP":
                    p,
            }
        )


    # --------------------------------------------------------
    # Categorical
    # --------------------------------------------------------

    for variable in categorical_validators:

        valid = (
            system_metadata[
                variable
            ]
            .notna()
        )


        temp_meta = (
            system_metadata.loc[
                valid
            ]
            .copy()
        )


        groups = (
            temp_meta[
                variable
            ]
            .astype(
                str
            )
            .to_numpy()
        )


        if len(
            np.unique(
                groups
            )
        ) < 2:

            continue


        indices = np.flatnonzero(
            valid.to_numpy()
        )


        Y = (
            Y_all[
                indices
            ]
        )


        test_counter += 1


        F, R2, p = (
            permutation_categorical(
                Y,
                groups,
                RANDOM_SEED
                +
                test_counter,
            )
        )


        global_test_rows.append(
            {
                "Representation":
                    representation_name,

                "Validator":
                    variable,

                "Type":
                    "Categorical",

                "N_systems":
                    len(
                        groups
                    ),

                "F":
                    F,

                "R2":
                    R2,

                "PermutationP":
                    p,
            }
        )


global_signature_tests = pd.DataFrame(
    global_test_rows
)


for representation in (
    global_signature_tests[
        "Representation"
    ]
    .unique()
):

    mask = (
        global_signature_tests[
            "Representation"
        ]
        ==
        representation
    )


    global_signature_tests.loc[
        mask,
        "FDR_Q"
    ] = bh_fdr(
        global_signature_tests.loc[
            mask,
            "PermutationP"
        ]
    )


# ============================================================
# 22. Feature-level reweighting tests
#
# For each predictor:
#
# response =
# its 4-dimensional subtype-reweighting profile
#
# Question:
#
# Does GDP / region / income / existing architecture group
# systematically change HOW that predictor is distributed
# across bullying types?
# ============================================================

feature_reweight_test_rows = []


feature_validators_cont = [
    "LogGDPpcPPP_2018",
    "Gini_Nearest2018",
]


feature_validators_cat = [
    "IncomeBroad",
    "WB_Region",
    "JointArchitectureK3",
]


feature_test_counter = 0


for feature_index, feature in enumerate(
    FEATURES
):

    feature_Y_all = (
        reweight_tensor[
            :,
            :,
            feature_index
        ]
    )


    # --------------------------------------------------------
    # Continuous
    # --------------------------------------------------------

    for variable in feature_validators_cont:

        valid = (
            system_metadata[
                variable
            ]
            .notna()
        )


        indices = np.flatnonzero(
            valid.to_numpy()
        )


        Y = feature_Y_all[
            indices
        ]


        x = (
            system_metadata.loc[
                valid,
                variable
            ]
            .to_numpy(
                dtype=float
            )
        )


        feature_test_counter += 1


        F, R2, p = (
            permutation_continuous(
                Y,
                x,
                RANDOM_SEED
                +
                10000
                +
                feature_test_counter,
            )
        )


        feature_reweight_test_rows.append(
            {
                "Feature":
                    feature,

                "Validator":
                    variable,

                "Type":
                    "Continuous",

                "N_systems":
                    len(
                        x
                    ),

                "F":
                    F,

                "R2":
                    R2,

                "PermutationP":
                    p,
            }
        )


    # --------------------------------------------------------
    # Categorical
    # --------------------------------------------------------

    for variable in feature_validators_cat:

        valid = (
            system_metadata[
                variable
            ]
            .notna()
        )


        indices = np.flatnonzero(
            valid.to_numpy()
        )


        Y = feature_Y_all[
            indices
        ]


        groups = (
            system_metadata.loc[
                valid,
                variable
            ]
            .astype(
                str
            )
            .to_numpy()
        )


        feature_test_counter += 1


        F, R2, p = (
            permutation_categorical(
                Y,
                groups,
                RANDOM_SEED
                +
                10000
                +
                feature_test_counter,
            )
        )


        feature_reweight_test_rows.append(
            {
                "Feature":
                    feature,

                "Validator":
                    variable,

                "Type":
                    "Categorical",

                "N_systems":
                    len(
                        groups
                    ),

                "F":
                    F,

                "R2":
                    R2,

                "PermutationP":
                    p,
            }
        )


feature_reweight_tests = pd.DataFrame(
    feature_reweight_test_rows
)


# FDR within validator family across 14 predictors

feature_reweight_tests[
    "FDR_Q"
] = np.nan


for validator in (
    feature_reweight_tests[
        "Validator"
    ]
    .unique()
):

    mask = (
        feature_reweight_tests[
            "Validator"
        ]
        ==
        validator
    )


    feature_reweight_tests.loc[
        mask,
        "FDR_Q"
    ] = bh_fdr(
        feature_reweight_tests.loc[
            mask,
            "PermutationP"
        ]
    )


# ============================================================
# 23. Macro relationships with reweighting INTENSITY
# ============================================================

intensity_rows = []


for variable in continuous_validators:

    valid = (
        intensity_macro[
            variable
        ]
        .notna()
    )


    rho, p = spearmanr(
        intensity_macro.loc[
            valid,
            variable
        ],
        intensity_macro.loc[
            valid,
            "SubtypeReweightingIntensity"
        ],
    )


    intensity_rows.append(
        {
            "Validator":
                variable,

            "N_systems":
                int(
                    valid.sum()
                ),

            "SpearmanRho":
                rho,

            "P":
                p,
        }
    )


intensity_correlations = pd.DataFrame(
    intensity_rows
)


intensity_correlations[
    "FDR_Q"
] = bh_fdr(
    intensity_correlations[
        "P"
    ]
)


# ============================================================
# 24. Contrast-direction analysis
#
# This provides INTERPRETABILITY after the multivariate tests.
#
# Example:
#
# positive gender TP_vs_VR score =
#
# gender receives relatively greater attribution for
# threat/physical than verbal/relational bullying.
# ============================================================

contrast_macro = (
    contrast_scores
    .merge(
        system_metadata,
        on="Country",
        how="left",
        validate="many_to_one",
    )
)


contrast_association_rows = []


# ------------------------------------------------------------
# Continuous macro gradients
# ------------------------------------------------------------

for variable in [
    "LogGDPpcPPP_2018",
    "Gini_Nearest2018",
]:

    for contrast in (
        contrast_macro[
            "Contrast"
        ]
        .unique()
    ):

        for feature in FEATURES:

            temp = (
                contrast_macro[
                    (
                        contrast_macro[
                            "Contrast"
                        ]
                        ==
                        contrast
                    )
                    &
                    (
                        contrast_macro[
                            "Feature"
                        ]
                        ==
                        feature
                    )
                ]
            )


            valid = (
                temp[
                    variable
                ]
                .notna()
            )


            if valid.sum() < 10:

                continue


            rho, p = spearmanr(
                temp.loc[
                    valid,
                    variable
                ],
                temp.loc[
                    valid,
                    "Value"
                ],
            )


            contrast_association_rows.append(
                {
                    "Validator":
                        variable,

                    "Type":
                        "Continuous",

                    "Contrast":
                        contrast,

                    "Feature":
                        feature,

                    "N_systems":
                        int(
                            valid.sum()
                        ),

                    "Effect":
                        rho,

                    "EffectType":
                        "SpearmanRho",

                    "P":
                        p,
                }
            )


contrast_associations = pd.DataFrame(
    contrast_association_rows
)


contrast_associations[
    "FDR_Q"
] = np.nan


for (
    validator,
    contrast
), indices in (
    contrast_associations
    .groupby(
        [
            "Validator",
            "Contrast",
        ]
    )
    .groups
    .items()
):

    contrast_associations.loc[
        indices,
        "FDR_Q"
    ] = bh_fdr(
        contrast_associations.loc[
            indices,
            "P"
        ]
    )


# ============================================================
# 25. PCA macro correlations
# ============================================================

pca_macro = (
    reweight_pca_scores
    .merge(
        system_metadata,
        on="Country",
        how="left",
        validate="one_to_one",
    )
)


pca_macro_rows = []


for variable in continuous_validators:

    for pc in [
        "PC1",
        "PC2",
        "PC3",
    ]:

        valid = (
            pca_macro[
                variable
            ]
            .notna()
        )


        rho, p = spearmanr(
            pca_macro.loc[
                valid,
                variable
            ],
            pca_macro.loc[
                valid,
                pc
            ],
        )


        pca_macro_rows.append(
            {
                "Validator":
                    variable,

                "PC":
                    pc,

                "N_systems":
                    int(
                        valid.sum()
                    ),

                "SpearmanRho":
                    rho,

                "P":
                    p,
            }
        )


pca_macro_correlations = pd.DataFrame(
    pca_macro_rows
)


pca_macro_correlations[
    "FDR_Q"
] = bh_fdr(
    pca_macro_correlations[
        "P"
    ]
)


# ============================================================
# 26. PCA-reduced representation for clustering
#
# Full signature is high dimensional.
#
# Use enough PCs to preserve >=80% of its variance.
# ============================================================

cluster_PCs = (
    pca_scores_array[
        :,
        :
        components_80
    ]
)


# ============================================================
# 27. Cluster helpers
# ============================================================

def fit_clusters(
    X,
    algorithm,
    k,
    seed,
):

    if algorithm == "KMeans":

        model = KMeans(
            n_clusters=k,
            n_init=50,
            random_state=seed,
        )


        labels = model.fit_predict(
            X
        )


    elif algorithm == "Ward":

        labels = (
            AgglomerativeClustering(
                n_clusters=k,
                linkage="ward",
            )
            .fit_predict(
                X
            )
        )


    elif algorithm == "GMM":

        labels = (
            GaussianMixture(
                n_components=k,
                covariance_type="full",
                reg_covar=1e-6,
                n_init=10,
                random_state=seed,
            )
            .fit_predict(
                X
            )
        )


    else:

        raise ValueError(
            algorithm
        )


    return labels


def cluster_stability(
    X,
    algorithm,
    k,
    full_labels,
    seed,
):

    rng = np.random.default_rng(
        seed
    )


    n = len(
        X
    )


    subset_n = int(
        np.floor(
            STABILITY_SUBSAMPLE
            *
            n
        )
    )


    ari_values = []


    for repeat in range(
        N_STABILITY_REPEATS
    ):

        subset = np.sort(
            rng.choice(
                n,
                size=subset_n,
                replace=False,
            )
        )


        try:

            labels = fit_clusters(
                X[
                    subset
                ],
                algorithm,
                k,
                seed
                +
                repeat
                +
                1000,
            )


            ari = adjusted_rand_score(
                full_labels[
                    subset
                ],
                labels,
            )


            ari_values.append(
                ari
            )


        except Exception:

            pass


    if len(
        ari_values
    ) == 0:

        return (
            np.nan,
            np.nan,
        )


    return (
        float(
            np.mean(
                ari_values
            )
        ),
        float(
            np.median(
                ari_values
            )
        ),
    )


# ============================================================
# 28. Reweighting-signature cluster discovery
# ============================================================

cluster_rows = []

cluster_membership_rows = []

cluster_store = {}


solution_counter = 0


for algorithm in ALGORITHMS:

    for k in K_VALUES:

        solution_counter += 1


        labels = fit_clusters(
            cluster_PCs,
            algorithm,
            k,
            RANDOM_SEED
            +
            solution_counter,
        )


        counts = np.bincount(
            labels
        )


        silhouette = silhouette_score(
            cluster_PCs,
            labels,
        )


        db = davies_bouldin_score(
            cluster_PCs,
            labels,
        )


        ch = calinski_harabasz_score(
            cluster_PCs,
            labels,
        )


        stability_mean, stability_median = (
            cluster_stability(
                cluster_PCs,
                algorithm,
                k,
                labels,
                RANDOM_SEED
                +
                20000
                +
                solution_counter
                *
                100,
            )
        )


        solution_id = (
            "REWEIGHT"
            f"__{algorithm}"
            f"__K{k}"
        )


        cluster_store[
            solution_id
        ] = labels


        cluster_rows.append(
            {
                "SolutionID":
                    solution_id,

                "Algorithm":
                    algorithm,

                "K":
                    k,

                "N_systems":
                    len(
                        common_systems
                    ),

                "MinimumClusterSize":
                    int(
                        counts.min()
                    ),

                "Silhouette":
                    silhouette,

                "DaviesBouldin":
                    db,

                "CalinskiHarabasz":
                    ch,

                "StabilityMeanARI":
                    stability_mean,

                "StabilityMedianARI":
                    stability_median,
            }
        )


        for country, cluster in zip(
            common_systems,
            labels,
        ):

            cluster_membership_rows.append(
                {
                    "SolutionID":
                        solution_id,

                    "Country":
                        country,

                    "Cluster":
                        int(
                            cluster
                        )
                        +
                        1,
                }
            )


reweight_cluster_diagnostics = pd.DataFrame(
    cluster_rows
)


reweight_cluster_memberships = pd.DataFrame(
    cluster_membership_rows
)


# ============================================================
# 29. Cross-algorithm cluster agreement
# ============================================================

agreement_rows = []


for k in K_VALUES:

    for algorithm_a, algorithm_b in combinations(
        ALGORITHMS,
        2,
    ):

        id_a = (
            f"REWEIGHT__"
            f"{algorithm_a}"
            f"__K{k}"
        )


        id_b = (
            f"REWEIGHT__"
            f"{algorithm_b}"
            f"__K{k}"
        )


        ari = adjusted_rand_score(
            cluster_store[
                id_a
            ],
            cluster_store[
                id_b
            ],
        )


        agreement_rows.append(
            {
                "K":
                    k,

                "Algorithm1":
                    algorithm_a,

                "Algorithm2":
                    algorithm_b,

                "AdjustedRand":
                    ari,
            }
        )


reweight_algorithm_agreement = pd.DataFrame(
    agreement_rows
)


agreement_summary = (
    reweight_algorithm_agreement
    .groupby(
        "K",
        as_index=False,
    )[
        "AdjustedRand"
    ]
    .mean()
    .rename(
        columns={
            "AdjustedRand":
                "MeanCrossAlgorithmARI"
        }
    )
)


reweight_cluster_diagnostics = (
    reweight_cluster_diagnostics
    .merge(
        agreement_summary,
        on="K",
        how="left",
        validate="many_to_one",
    )
)


# ============================================================
# 30. Compare reweighting clusters to existing JOINT K=3
# ============================================================

existing_joint_labels = (
    system_metadata[
        "JointArchitectureK3"
    ]
)


joint_valid = (
    existing_joint_labels.notna()
)


joint_cluster_comparison_rows = []


for solution_id, labels in (
    cluster_store.items()
):

    ari = adjusted_rand_score(
        existing_joint_labels[
            joint_valid
        ],
        labels[
            joint_valid.to_numpy()
        ],
    )


    joint_cluster_comparison_rows.append(
        {
            "SolutionID":
                solution_id,

            "ARI_with_JOINT5_K3":
                ari,
        }
    )


joint_cluster_comparison = pd.DataFrame(
    joint_cluster_comparison_rows
)


reweight_cluster_diagnostics = (
    reweight_cluster_diagnostics
    .merge(
        joint_cluster_comparison,
        on="SolutionID",
        how="left",
        validate="one_to_one",
    )
)


# ============================================================
# 31. Cluster external-validation helpers
# ============================================================

def cramers_v(
    cluster_labels,
    external_labels,
):

    table = pd.crosstab(
        cluster_labels,
        external_labels,
    )


    if (
        table.shape[
            0
        ]
        <
        2
        or
        table.shape[
            1
        ]
        <
        2
    ):

        return np.nan


    chi2 = chi2_contingency(
        table,
        correction=False,
    )[
        0
    ]


    n = table.values.sum()


    denominator = min(
        table.shape[
            0
        ]
        -
        1,
        table.shape[
            1
        ]
        -
        1,
    )


    return float(
        np.sqrt(
            (
                chi2
                /
                n
            )
            /
            denominator
        )
    )


def continuous_cluster_r2(
    values,
    clusters,
):

    values = np.asarray(
        values,
        dtype=float,
    )


    clusters = np.asarray(
        clusters
    )


    grand = values.mean()


    total = np.sum(
        (
            values
            -
            grand
        ) ** 2
    )


    between = 0.0


    for cluster in np.unique(
        clusters
    ):

        vals = values[
            clusters
            ==
            cluster
        ]


        between += (
            len(
                vals
            )
            *
            (
                vals.mean()
                -
                grand
            ) ** 2
        )


    return float(
        between
        /
        total
    )


# ============================================================
# 32. External validation of reasonably stable signature
# clusters
# ============================================================

candidate_solutions = (
    reweight_cluster_diagnostics[
        (
            reweight_cluster_diagnostics[
                "MinimumClusterSize"
            ]
            >=
            5
        )
        &
        (
            reweight_cluster_diagnostics[
                "StabilityMeanARI"
            ]
            >=
            0.60
        )
    ]
)


external_rows = []


external_counter = 0


for _, solution in (
    candidate_solutions.iterrows()
):

    solution_id = (
        solution[
            "SolutionID"
        ]
    )


    labels = (
        cluster_store[
            solution_id
        ]
    )


    # --------------------------------------------------------
    # Continuous
    # --------------------------------------------------------

    for variable in [
        "LogGDPpcPPP_2018",
        "Gini_Nearest2018",
    ]:

        valid = (
            system_metadata[
                variable
            ]
            .notna()
        )


        values = (
            system_metadata.loc[
                valid,
                variable
            ]
            .to_numpy(
                dtype=float
            )
        )


        clusters_valid = labels[
            valid.to_numpy()
        ]


        observed = (
            continuous_cluster_r2(
                values,
                clusters_valid,
            )
        )


        rng = np.random.default_rng(
            RANDOM_SEED
            +
            30000
            +
            external_counter
        )


        null = np.empty(
            N_PERMUTATIONS
        )


        for b in range(
            N_PERMUTATIONS
        ):

            null[
                b
            ] = (
                continuous_cluster_r2(
                    rng.permutation(
                        values
                    ),
                    clusters_valid,
                )
            )


        p = (
            1
            +
            np.sum(
                null
                >=
                observed
            )
        ) / (
            N_PERMUTATIONS
            +
            1
        )


        external_counter += 1


        external_rows.append(
            {
                "SolutionID":
                    solution_id,

                "Validator":
                    variable,

                "Type":
                    "Continuous",

                "Effect":
                    observed,

                "EffectType":
                    "ClusterExplainedR2",

                "PermutationP":
                    p,
            }
        )


    # --------------------------------------------------------
    # Categorical
    # --------------------------------------------------------

    for variable in [
        "IncomeBroad",
        "WB_Region",
        "GDP_Tertile",
        "Gini_Tertile",
        "GDP_Gini_Quadrant",
        "JointArchitectureK3",
    ]:

        valid = (
            system_metadata[
                variable
            ]
            .notna()
        )


        external_values = (
            system_metadata.loc[
                valid,
                variable
            ]
            .astype(
                str
            )
            .to_numpy()
        )


        clusters_valid = labels[
            valid.to_numpy()
        ]


        observed = cramers_v(
            clusters_valid,
            external_values,
        )


        rng = np.random.default_rng(
            RANDOM_SEED
            +
            40000
            +
            external_counter
        )


        null = np.empty(
            N_PERMUTATIONS
        )


        for b in range(
            N_PERMUTATIONS
        ):

            null[
                b
            ] = cramers_v(
                clusters_valid,
                rng.permutation(
                    external_values
                ),
            )


        p = (
            1
            +
            np.sum(
                null
                >=
                observed
            )
        ) / (
            N_PERMUTATIONS
            +
            1
        )


        external_counter += 1


        external_rows.append(
            {
                "SolutionID":
                    solution_id,

                "Validator":
                    variable,

                "Type":
                    "Categorical",

                "Effect":
                    observed,

                "EffectType":
                    "CramersV",

                "PermutationP":
                    p,
            }
        )


cluster_external_validation = pd.DataFrame(
    external_rows
)


if len(
    cluster_external_validation
) > 0:

    cluster_external_validation[
        "FDR_Q"
    ] = bh_fdr(
        cluster_external_validation[
            "PermutationP"
        ]
    )


# ============================================================
# 33. Save outputs
# ============================================================

intensity_df.to_csv(
    OUT_DIR
    / "system_subtype_reweighting_intensity.csv",
    index=False,
)


pairwise_subtype_distances.to_csv(
    OUT_DIR
    / "system_pairwise_subtype_distances.csv",
    index=False,
)


feature_variability.to_csv(
    OUT_DIR
    / "feature_subtype_variability_by_system.csv",
    index=False,
)


contrast_scores.to_csv(
    OUT_DIR
    / "system_feature_subtype_contrasts.csv",
    index=False,
)


reweight_pca_scores.to_csv(
    OUT_DIR
    / "reweighting_pca_scores.csv",
    index=False,
)


pca_loadings.to_csv(
    OUT_DIR
    / "reweighting_pca_loadings.csv",
    index=False,
)


global_signature_tests.to_csv(
    OUT_DIR
    / "global_reweighting_macro_tests.csv",
    index=False,
)


feature_reweight_tests.to_csv(
    OUT_DIR
    / "feature_reweighting_macro_tests.csv",
    index=False,
)


intensity_correlations.to_csv(
    OUT_DIR
    / "reweighting_intensity_macro_correlations.csv",
    index=False,
)


contrast_associations.to_csv(
    OUT_DIR
    / "subtype_contrast_macro_correlations.csv",
    index=False,
)


pca_macro_correlations.to_csv(
    OUT_DIR
    / "reweighting_pca_macro_correlations.csv",
    index=False,
)


reweight_cluster_diagnostics.to_csv(
    OUT_DIR
    / "reweighting_cluster_diagnostics.csv",
    index=False,
)


reweight_cluster_memberships.to_csv(
    OUT_DIR
    / "reweighting_cluster_memberships.csv",
    index=False,
)


reweight_algorithm_agreement.to_csv(
    OUT_DIR
    / "reweighting_cross_algorithm_agreement.csv",
    index=False,
)


candidate_solutions.to_csv(
    OUT_DIR
    / "reweighting_cluster_candidates.csv",
    index=False,
)


cluster_external_validation.to_csv(
    OUT_DIR
    / "reweighting_cluster_external_validation.csv",
    index=False,
)


# ============================================================
# 34. Print PCA summary
# ============================================================

print("\n")
print("=" * 120)
print("SUBTYPE-REWEIGHTING PCA")
print("=" * 120)


print(
    f"PC1 variance: {explained[0]:.4f}"
)

print(
    f"PC2 variance: {explained[1]:.4f}"
)

print(
    f"PC3 variance: {explained[2]:.4f}"
)

print(
    f"PC1-3 total: {explained[:3].sum():.4f}"
)

print(
    "Components for >=80%:",
    components_80
)

print(
    "Components for >=90%:",
    components_90
)


# ============================================================
# 35. Top PCA loading variables
# ============================================================

print("\n")
print("=" * 120)
print("TOP REWEIGHTING PCA DRIVERS")
print("=" * 120)


for pc in [
    1,
    2,
    3,
]:

    temp = (
        pca_loadings[
            pca_loadings[
                "PC"
            ]
            ==
            pc
        ]
        .copy()
    )


    temp[
        "AbsCoefficient"
    ] = (
        temp[
            "Coefficient"
        ]
        .abs()
    )


    temp = (
        temp
        .sort_values(
            "AbsCoefficient",
            ascending=False,
        )
        .head(
            12
        )
    )


    print("\n")
    print(
        f"PC{pc}"
    )

    print("-" * 90)


    print(
        temp[
            [
                "Variable",
                "Coefficient",
            ]
        ]
        .round(
            4
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 36. Global macro-pattern tests
# ============================================================

print("\n")
print("=" * 120)
print("GLOBAL MACRO ASSOCIATIONS WITH SUBTYPE REWEIGHTING")
print("=" * 120)


print(
    global_signature_tests[
        [
            "Representation",
            "Validator",
            "Type",
            "N_systems",
            "F",
            "R2",
            "PermutationP",
            "FDR_Q",
        ]
    ]
    .sort_values(
        [
            "Representation",
            "FDR_Q",
        ]
    )
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 37. Predictor-specific macro effects
# ============================================================

print("\n")
print("=" * 120)
print("PREDICTORS WHOSE SUBTYPE REWEIGHTING VARIES BY MACRO CONTEXT")
print("=" * 120)


print(
    feature_reweight_tests
    .sort_values(
        [
            "Validator",
            "FDR_Q",
            "R2",
        ],
        ascending=[
            True,
            True,
            False,
        ]
    )
    .groupby(
        "Validator",
        group_keys=False,
    )
    .head(
        8
    )[
        [
            "Validator",
            "Feature",
            "Type",
            "N_systems",
            "R2",
            "PermutationP",
            "FDR_Q",
        ]
    ]
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 38. Strongest directional subtype contrasts
# ============================================================

print("\n")
print("=" * 120)
print("STRONGEST GDP / GINI SUBTYPE-CONTRAST PATTERNS")
print("=" * 120)


print(
    contrast_associations
    .assign(
        AbsEffect=lambda x:
            x[
                "Effect"
            ].abs()
    )
    .sort_values(
        [
            "FDR_Q",
            "AbsEffect",
        ],
        ascending=[
            True,
            False,
        ]
    )
    .head(
        40
    )[
        [
            "Validator",
            "Contrast",
            "Feature",
            "N_systems",
            "Effect",
            "P",
            "FDR_Q",
        ]
    ]
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 39. Reweighting intensity
# ============================================================

print("\n")
print("=" * 120)
print("MACRO CORRELATES OF OVERALL SUBTYPE-REWEIGHTING INTENSITY")
print("=" * 120)


print(
    intensity_correlations
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 40. Most / least subtype-sensitive systems
# ============================================================

print("\n")
print("=" * 120)
print("MOST SUBTYPE-SENSITIVE SYSTEMS")
print("=" * 120)


print(
    intensity_df
    .sort_values(
        "SubtypeReweightingIntensity",
        ascending=False,
    )
    .head(
        15
    )
    .round(
        4
    )
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 120)
print("LEAST SUBTYPE-SENSITIVE SYSTEMS")
print("=" * 120)


print(
    intensity_df
    .sort_values(
        "SubtypeReweightingIntensity",
        ascending=True,
    )
    .head(
        15
    )
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 41. New reweighting-signature cluster results
# ============================================================

print("\n")
print("=" * 120)
print("REWEIGHTING-SIGNATURE CLUSTER DIAGNOSTICS")
print("=" * 120)


print(
    reweight_cluster_diagnostics
    .sort_values(
        [
            "StabilityMeanARI",
            "Silhouette",
        ],
        ascending=[
            False,
            False,
        ]
    )
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 42. Reasonably stable candidates
# ============================================================

print("\n")
print("=" * 120)
print("STABLE REWEIGHTING-SIGNATURE CLUSTER CANDIDATES")
print("=" * 120)


if len(
    candidate_solutions
) == 0:

    print(
        "No cluster solutions met the stability criteria."
    )


else:

    print(
        candidate_solutions
        .sort_values(
            [
                "StabilityMeanARI",
                "Silhouette",
            ],
            ascending=[
                False,
                False,
            ]
        )
        .round(
            4
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 43. External validation
# ============================================================

print("\n")
print("=" * 120)
print("EXTERNAL VALIDATION OF REWEIGHTING CLUSTERS")
print("=" * 120)


if len(
    cluster_external_validation
) == 0:

    print(
        "No sufficiently stable clusters available "
        "for external validation."
    )


else:

    print(
        cluster_external_validation
        .sort_values(
            [
                "FDR_Q",
                "Effect",
            ],
            ascending=[
                True,
                False,
            ]
        )
        .head(
            50
        )
        .round(
            5
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 44. Final status
# ============================================================

print("\n")
print("=" * 120)
print("CROSS-SUBTYPE REWEIGHTING STAGE COMPLETE")
print("=" * 120)


print(
    "Systems:",
    len(
        common_systems
    )
)


print(
    "Signature dimensions:",
    signature_matrix.shape[
        1
    ]
)


print(
    "PCA dimensions retaining >=80% variance:",
    components_80
)


print(
    "Stable cluster candidates:",
    len(
        candidate_solutions
    )
)


print(
    "\nDo not interpret cluster numbers until "
    "we inspect the results."
)

Systems with all four subtype profiles: 75


SUBTYPE-REWEIGHTING PCA
PC1 variance: 0.2292
PC2 variance: 0.0854
PC3 variance: 0.0608
PC1-3 total: 0.3755
Components for >=80%: 16
Components for >=90%: 23


TOP REWEIGHTING PCA DRIVERS


PC1
------------------------------------------------------------------------------------------
                  Variable  Coefficient
BULLY_RELATIONAL:ST004D01T       0.8024
  BULLY_PHYSICAL:ST004D01T      -0.2757
    BULLY_VERBAL:ST004D01T      -0.2670
    BULLY_THREAT:ST004D01T      -0.2597
 BULLY_RELATIONAL:METASPAM      -0.1303
   BULLY_RELATIONAL:UNDREM      -0.1060
  BULLY_RELATIONAL:METASUM      -0.1047
     BULLY_VERBAL:METASPAM       0.0917
      BULLY_THREAT:JOYREAD       0.0778
       BULLY_VERBAL:UNDREM       0.0776
     BULLY_PHYSICAL:BELONG       0.0756
   BULLY_VERBAL:ADAPTIVITY       0.0748


PC2
------------------------------------------------------------------------------------------
              Variable  Coefficient
BULLY_VERBAL:ST004

In [4]:
# ============================================================
# FINAL CHARACTERIZATION:
# SUBTYPE-REWEIGHTING K=2 ARCHITECTURE GROUPS
# ============================================================
#
# PURPOSE
# -------
# Fully characterize the strongest subtype-reweighting
# clustering result:
#
#       REWEIGHT__KMeans__K2
#
#
# QUESTIONS
# ---------
# 1. Which education systems are in each group?
#
# 2. How stable is each system's membership?
#
# 3. Which subtype x predictor contrasts separate the groups?
#
# 4. How much of the separation is driven by:
#       - relational gender?
#       - gender across all subtypes?
#       - relational bullying generally?
#
# 5. What do the groups look like economically/geographically?
#
# 6. How do these groups overlap with our previous
#    JOINT5 K=3 architecture groups?
#
#
# IMPORTANT
# ---------
# Cluster numbers are arbitrary.
#
# This cell does NOT refit any XGBoost models.
# ============================================================


from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

from scipy.optimize import linear_sum_assignment

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score


# ============================================================
# 1. Paths
# ============================================================

RESULT_ROOT = Path(
    "../results"
)


RANKING_PATH = (
    RESULT_ROOT
    / "primary_attribution"
    / "oof_shap_rankings.csv"
)


MACRO_PATH = (
    RESULT_ROOT
    / "cluster_discovery"
    / "macro_validators_with_derived_groups.csv"
)


REWEIGHT_MEMBERSHIP_PATH = (
    RESULT_ROOT
    / "subtype_reweighting"
    / "reweighting_cluster_memberships.csv"
)


JOINT_MEMBERSHIP_PATH = (
    RESULT_ROOT
    / "cluster_discovery"
    / "cluster_memberships.csv"
)


OUT_DIR = (
    RESULT_ROOT
    / "subtype_reweighting_k2_final"
)


OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. Configuration
# ============================================================

FEATURES = [
    "ADAPTIVITY",
    "AGE",
    "ATTLNACT",
    "BELONG",
    "DISCLIMA",
    "EDUSHORT",
    "ESCS",
    "JOYREAD",
    "MASTGOAL",
    "METASPAM",
    "METASUM",
    "ST004D01T",
    "STIMREAD",
    "UNDREM",
]


SUBTYPES = [
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


TARGET_SOLUTION = (
    "REWEIGHT__KMeans__K2"
)


JOINT_SOLUTION = (
    "JOINT5__KMeans__K3"
)


N_BOOTSTRAPS = 1000

BOOTSTRAP_FRACTION = 0.90

RANDOM_SEED = 42


# ============================================================
# 3. Load files
# ============================================================

rankings = pd.read_csv(
    RANKING_PATH
)


macro = pd.read_csv(
    MACRO_PATH
)


reweight_memberships = pd.read_csv(
    REWEIGHT_MEMBERSHIP_PATH
)


joint_memberships = pd.read_csv(
    JOINT_MEMBERSHIP_PATH
)


rankings[
    "ImportanceShare"
] = pd.to_numeric(
    rankings[
        "ImportanceShare"
    ],
    errors="coerce",
)


# ============================================================
# 4. Composition helpers
# ============================================================

def close_composition(
    matrix
):

    matrix = np.asarray(
        matrix,
        dtype=float
    )


    return (
        matrix
        /
        matrix.sum(
            axis=1,
            keepdims=True
        )
    )


def clr_transform(
    matrix
):

    matrix = close_composition(
        matrix
    )


    logs = np.log(
        matrix
    )


    return (
        logs
        -
        logs.mean(
            axis=1,
            keepdims=True
        )
    )


# ============================================================
# 5. Build subtype CLR matrices
# ============================================================

subtype_store = {}


for outcome in SUBTYPES:

    temp = (
        rankings[
            rankings[
                "Outcome"
            ]
            ==
            outcome
        ]
    )


    systems = sorted(
        temp[
            "Country"
        ]
        .unique()
    )


    rows = []


    for country in systems:

        vector = (
            temp[
                temp[
                    "Country"
                ]
                ==
                country
            ]
            .set_index(
                "Feature"
            )
            .loc[
                FEATURES,
                "ImportanceShare"
            ]
            .to_numpy(
                dtype=float
            )
        )


        vector = (
            vector
            /
            vector.sum()
        )


        rows.append(
            vector
        )


    composition = np.vstack(
        rows
    )


    subtype_store[
        outcome
    ] = {

        "systems":
            systems,

        "composition":
            composition,

        "clr":
            clr_transform(
                composition
            ),
    }


# ============================================================
# 6. Common systems
# ============================================================

common_systems = sorted(
    set.intersection(
        *[
            set(
                subtype_store[
                    outcome
                ][
                    "systems"
                ]
            )
            for outcome
            in SUBTYPES
        ]
    )
)


print(
    "Common systems:",
    len(
        common_systems
    )
)


# ============================================================
# 7. Build system x subtype x feature tensor
# ============================================================

clr_tensor = np.empty(
    (
        len(
            common_systems
        ),
        len(
            SUBTYPES
        ),
        len(
            FEATURES
        ),
    )
)


for subtype_index, outcome in enumerate(
    SUBTYPES
):

    result = (
        subtype_store[
            outcome
        ]
    )


    index_map = {
        country: i
        for i, country
        in enumerate(
            result[
                "systems"
            ]
        )
    }


    for system_index, country in enumerate(
        common_systems
    ):

        clr_tensor[
            system_index,
            subtype_index,
            :
        ] = (
            result[
                "clr"
            ][
                index_map[
                    country
                ]
            ]
        )


# ============================================================
# 8. Reweighting signature
#
# Remove each system's mean architecture across subtypes.
# ============================================================

system_mean_clr = (
    clr_tensor.mean(
        axis=1,
        keepdims=True
    )
)


reweight_tensor = (
    clr_tensor
    -
    system_mean_clr
)


signature_matrix = (
    reweight_tensor.reshape(
        len(
            common_systems
        ),
        -1
    )
)


# ============================================================
# 9. Rebuild PCA representation used for clustering
# ============================================================

pca = PCA()


pca_scores = pca.fit_transform(
    signature_matrix
)


cumulative = np.cumsum(
    pca.explained_variance_ratio_
)


components_80 = int(
    np.argmax(
        cumulative
        >=
        0.80
    )
    +
    1
)


cluster_X = (
    pca_scores[
        :,
        :
        components_80
    ]
)


print(
    "PCA components used:",
    components_80
)


# ============================================================
# 10. Load frozen K=2 membership
# ============================================================

target_membership = (
    reweight_memberships[
        reweight_memberships[
            "SolutionID"
        ]
        ==
        TARGET_SOLUTION
    ][
        [
            "Country",
            "Cluster"
        ]
    ]
    .copy()
)


membership_map = (
    target_membership
    .set_index(
        "Country"
    )[
        "Cluster"
    ]
    .to_dict()
)


reference_labels = np.array(
    [
        membership_map[
            country
        ]
        -
        1
        for country
        in common_systems
    ]
)


# ============================================================
# 11. Label-alignment helper
# ============================================================

def align_labels(
    reference,
    candidate
):

    reference = np.asarray(
        reference
    )


    candidate = np.asarray(
        candidate
    )


    ref_values = np.unique(
        reference
    )


    cand_values = np.unique(
        candidate
    )


    matrix = np.zeros(
        (
            len(
                ref_values
            ),
            len(
                cand_values
            )
        ),
        dtype=int
    )


    for i, ref in enumerate(
        ref_values
    ):

        for j, cand in enumerate(
            cand_values
        ):

            matrix[
                i,
                j
            ] = np.sum(
                (
                    reference
                    ==
                    ref
                )
                &
                (
                    candidate
                    ==
                    cand
                )
            )


    rows, cols = (
        linear_sum_assignment(
            -matrix
        )
    )


    mapping = {
        cand_values[
            col
        ]:
        ref_values[
            row
        ]
        for row, col
        in zip(
            rows,
            cols
        )
    }


    return np.array(
        [
            mapping[
                label
            ]
            for label
            in candidate
        ]
    )


# ============================================================
# 12. Bootstrap membership stability
# ============================================================

rng = np.random.default_rng(
    RANDOM_SEED
)


n_systems = len(
    common_systems
)


subset_n = int(
    np.floor(
        BOOTSTRAP_FRACTION
        *
        n_systems
    )
)


appearances = np.zeros(
    n_systems,
    dtype=int
)


matches = np.zeros(
    n_systems,
    dtype=int
)


bootstrap_ari = []


for bootstrap in range(
    N_BOOTSTRAPS
):

    subset = np.sort(
        rng.choice(
            n_systems,
            size=subset_n,
            replace=False
        )
    )


    model = KMeans(
        n_clusters=2,
        n_init=50,
        random_state=(
            RANDOM_SEED
            +
            bootstrap
            +
            1000
        )
    )


    candidate_labels = (
        model.fit_predict(
            cluster_X[
                subset
            ]
        )
    )


    ref_subset = (
        reference_labels[
            subset
        ]
    )


    aligned = align_labels(
        ref_subset,
        candidate_labels
    )


    appearances[
        subset
    ] += 1


    matches[
        subset
    ] += (
        aligned
        ==
        ref_subset
    )


    bootstrap_ari.append(
        adjusted_rand_score(
            ref_subset,
            aligned
        )
    )


membership_probability = (
    matches
    /
    appearances
)


bootstrap_ari = np.asarray(
    bootstrap_ari
)


# ============================================================
# 13. Membership table
# ============================================================

membership_rows = []


for index, country in enumerate(
    common_systems
):

    probability = (
        membership_probability[
            index
        ]
    )


    if probability >= 0.90:

        confidence = (
            "Core"
        )


    elif probability >= 0.75:

        confidence = (
            "Stable"
        )


    elif probability >= 0.60:

        confidence = (
            "Intermediate"
        )


    else:

        confidence = (
            "Boundary"
        )


    membership_rows.append(
        {
            "Country":
                country,

            "Cluster":
                reference_labels[
                    index
                ]
                +
                1,

            "MembershipProbability":
                probability,

            "Confidence":
                confidence,
        }
    )


membership_confidence = pd.DataFrame(
    membership_rows
)


# ============================================================
# 14. Full signature variable names
# ============================================================

signature_names = []


for subtype in SUBTYPES:

    for feature in FEATURES:

        signature_names.append(
            f"{subtype}:{feature}"
        )


# ============================================================
# 15. Cluster centroids in full reweighting space
# ============================================================

cluster_centroids = {}


for cluster in [
    0,
    1,
]:

    cluster_centroids[
        cluster
    ] = (
        signature_matrix[
            reference_labels
            ==
            cluster
        ]
        .mean(
            axis=0
        )
    )


centroid_difference = (
    cluster_centroids[
        1
    ]
    -
    cluster_centroids[
        0
    ]
)


centroid_driver_rows = []


for index, variable in enumerate(
    signature_names
):

    subtype, feature = (
        variable.split(
            ":"
        )
    )


    centroid_driver_rows.append(
        {
            "Subtype":
                subtype,

            "Feature":
                feature,

            "Variable":
                variable,

            "Cluster1Mean":
                cluster_centroids[
                    0
                ][
                    index
                ],

            "Cluster2Mean":
                cluster_centroids[
                    1
                ][
                    index
                ],

            "Cluster2MinusCluster1":
                centroid_difference[
                    index
                ],

            "AbsDifference":
                abs(
                    centroid_difference[
                        index
                    ]
                ),

            "SquaredDifference":
                centroid_difference[
                    index
                ] ** 2,
        }
    )


centroid_drivers = pd.DataFrame(
    centroid_driver_rows
)


# ============================================================
# 16. Contribution to total cluster separation
# ============================================================

total_squared_separation = np.sum(
    centroid_difference ** 2
)


centroid_drivers[
    "PctOfSquaredSeparation"
] = (
    100
    *
    centroid_drivers[
        "SquaredDifference"
    ]
    /
    total_squared_separation
)


# ============================================================
# 17. Aggregate separation by FEATURE
# ============================================================

feature_separation = (
    centroid_drivers
    .groupby(
        "Feature",
        as_index=False
    )
    .agg(
        SquaredSeparation=(
            "SquaredDifference",
            "sum"
        )
    )
)


feature_separation[
    "PctOfTotalSeparation"
] = (
    100
    *
    feature_separation[
        "SquaredSeparation"
    ]
    /
    total_squared_separation
)


feature_separation = (
    feature_separation
    .sort_values(
        "PctOfTotalSeparation",
        ascending=False
    )
)


# ============================================================
# 18. Aggregate separation by SUBTYPE
# ============================================================

subtype_separation = (
    centroid_drivers
    .groupby(
        "Subtype",
        as_index=False
    )
    .agg(
        SquaredSeparation=(
            "SquaredDifference",
            "sum"
        )
    )
)


subtype_separation[
    "PctOfTotalSeparation"
] = (
    100
    *
    subtype_separation[
        "SquaredSeparation"
    ]
    /
    total_squared_separation
)


subtype_separation = (
    subtype_separation
    .sort_values(
        "PctOfTotalSeparation",
        ascending=False
    )
)


# ============================================================
# 19. Specific contribution of relational gender
# ============================================================

relational_gender = (
    centroid_drivers[
        (
            centroid_drivers[
                "Subtype"
            ]
            ==
            "BULLY_RELATIONAL"
        )
        &
        (
            centroid_drivers[
                "Feature"
            ]
            ==
            "ST004D01T"
        )
    ]
    .iloc[
        0
    ]
)


gender_total = (
    centroid_drivers[
        centroid_drivers[
            "Feature"
        ]
        ==
        "ST004D01T"
    ][
        "SquaredDifference"
    ]
    .sum()
)


relational_total = (
    centroid_drivers[
        centroid_drivers[
            "Subtype"
        ]
        ==
        "BULLY_RELATIONAL"
    ][
        "SquaredDifference"
    ]
    .sum()
)


separation_summary = pd.DataFrame(
    [
        {
            "Component":
                "Relational gender coordinate",

            "PctOfTotalSquaredSeparation":
                100
                *
                relational_gender[
                    "SquaredDifference"
                ]
                /
                total_squared_separation,
        },

        {
            "Component":
                "Gender across all four subtypes",

            "PctOfTotalSquaredSeparation":
                100
                *
                gender_total
                /
                total_squared_separation,
        },

        {
            "Component":
                "Relational subtype block",

            "PctOfTotalSquaredSeparation":
                100
                *
                relational_total
                /
                total_squared_separation,
        },
    ]
)


# ============================================================
# 20. Interpretable subtype contrasts by group
# ============================================================

subtype_index = {
    subtype:
        i
    for i, subtype
    in enumerate(
        SUBTYPES
    )
}


contrast_rows = []


for system_index, country in enumerate(
    common_systems
):

    for feature_index, feature in enumerate(
        FEATURES
    ):

        verbal = clr_tensor[
            system_index,
            subtype_index[
                "BULLY_VERBAL"
            ],
            feature_index
        ]


        threat = clr_tensor[
            system_index,
            subtype_index[
                "BULLY_THREAT"
            ],
            feature_index
        ]


        physical = clr_tensor[
            system_index,
            subtype_index[
                "BULLY_PHYSICAL"
            ],
            feature_index
        ]


        relational = clr_tensor[
            system_index,
            subtype_index[
                "BULLY_RELATIONAL"
            ],
            feature_index
        ]


        contrast_rows.extend(
            [
                {
                    "Country":
                        country,

                    "Cluster":
                        reference_labels[
                            system_index
                        ]
                        +
                        1,

                    "Feature":
                        feature,

                    "Contrast":
                        "Relational_vs_Other3",

                    "Value":
                        relational
                        -
                        np.mean(
                            [
                                verbal,
                                threat,
                                physical
                            ]
                        ),
                },

                {
                    "Country":
                        country,

                    "Cluster":
                        reference_labels[
                            system_index
                        ]
                        +
                        1,

                    "Feature":
                        feature,

                    "Contrast":
                        "ThreatPhysical_vs_VerbalRelational",

                    "Value":
                        (
                            threat
                            +
                            physical
                        )
                        /
                        2
                        -
                        (
                            verbal
                            +
                            relational
                        )
                        /
                        2,
                },

                {
                    "Country":
                        country,

                    "Cluster":
                        reference_labels[
                            system_index
                        ]
                        +
                        1,

                    "Feature":
                        feature,

                    "Contrast":
                        "Physical_vs_Other3",

                    "Value":
                        physical
                        -
                        np.mean(
                            [
                                verbal,
                                threat,
                                relational
                            ]
                        ),
                },
            ]
        )


contrast_df = pd.DataFrame(
    contrast_rows
)


contrast_group_summary = (
    contrast_df
    .groupby(
        [
            "Cluster",
            "Contrast",
            "Feature"
        ],
        as_index=False
    )
    .agg(
        MeanContrast=(
            "Value",
            "mean"
        ),
        MedianContrast=(
            "Value",
            "median"
        )
    )
)


# ============================================================
# 21. Direct group-difference in contrast scores
# ============================================================

contrast_group_difference_rows = []


for contrast in (
    contrast_df[
        "Contrast"
    ]
    .unique()
):

    for feature in FEATURES:

        temp = (
            contrast_df[
                (
                    contrast_df[
                        "Contrast"
                    ]
                    ==
                    contrast
                )
                &
                (
                    contrast_df[
                        "Feature"
                    ]
                    ==
                    feature
                )
            ]
        )


        c1 = (
            temp[
                temp[
                    "Cluster"
                ]
                ==
                1
            ][
                "Value"
            ]
            .mean()
        )


        c2 = (
            temp[
                temp[
                    "Cluster"
                ]
                ==
                2
            ][
                "Value"
            ]
            .mean()
        )


        contrast_group_difference_rows.append(
            {
                "Contrast":
                    contrast,

                "Feature":
                    feature,

                "Cluster1Mean":
                    c1,

                "Cluster2Mean":
                    c2,

                "Cluster2MinusCluster1":
                    (
                        c2
                        -
                        c1
                    ),

                "AbsDifference":
                    abs(
                        c2
                        -
                        c1
                    ),
            }
        )


contrast_group_differences = pd.DataFrame(
    contrast_group_difference_rows
)


# ============================================================
# 22. Attach macro data
# ============================================================

membership_macro = (
    membership_confidence
    .merge(
        macro,
        on="Country",
        how="left",
        validate="one_to_one"
    )
)


# ============================================================
# 23. Continuous macro summaries
# ============================================================

continuous_macro_rows = []


for cluster in [
    1,
    2
]:

    group = (
        membership_macro[
            membership_macro[
                "Cluster"
            ]
            ==
            cluster
        ]
    )


    for variable in [
        "GDPpcPPP_2018",
        "LogGDPpcPPP_2018",
        "Gini_Nearest2018"
    ]:

        values = (
            group[
                variable
            ]
            .dropna()
        )


        continuous_macro_rows.append(
            {
                "Cluster":
                    cluster,

                "Variable":
                    variable,

                "N":
                    len(
                        values
                    ),

                "Mean":
                    values.mean(),

                "Median":
                    values.median(),

                "SD":
                    values.std(
                        ddof=1
                    ),
            }
        )


continuous_macro_summary = pd.DataFrame(
    continuous_macro_rows
)


# ============================================================
# 24. Categorical macro summaries
# ============================================================

categorical_macro_rows = []


for cluster in [
    1,
    2
]:

    group = (
        membership_macro[
            membership_macro[
                "Cluster"
            ]
            ==
            cluster
        ]
    )


    for variable in [
        "IncomeGroup_FY2019",
        "IncomeBroad",
        "WB_Region",
        "GDP_Tertile",
        "GDP_Quartile",
        "Gini_Tertile",
        "GDP_Gini_Quadrant"
    ]:

        counts = (
            group[
                variable
            ]
            .value_counts(
                dropna=False
            )
        )


        for level, count in counts.items():

            categorical_macro_rows.append(
                {
                    "Cluster":
                        cluster,

                    "Variable":
                        variable,

                    "Level":
                        str(
                            level
                        ),

                    "Count":
                        int(
                            count
                        ),

                    "Proportion":
                        count
                        /
                        len(
                            group
                        ),
                }
            )


categorical_macro_summary = pd.DataFrame(
    categorical_macro_rows
)


# ============================================================
# 25. Overlap with previous JOINT5 K=3
# ============================================================

joint_k3 = (
    joint_memberships[
        joint_memberships[
            "SolutionID"
        ]
        ==
        JOINT_SOLUTION
    ][
        [
            "Country",
            "Cluster"
        ]
    ]
    .rename(
        columns={
            "Cluster":
                "JointCluster"
        }
    )
)


overlap = (
    membership_confidence
    .merge(
        joint_k3,
        on="Country",
        how="inner",
        validate="one_to_one"
    )
)


overlap_table = pd.crosstab(
    overlap[
        "Cluster"
    ],
    overlap[
        "JointCluster"
    ]
)


overlap_row_proportions = pd.crosstab(
    overlap[
        "Cluster"
    ],
    overlap[
        "JointCluster"
    ],
    normalize="index"
)


overlap_ari = adjusted_rand_score(
    overlap[
        "Cluster"
    ],
    overlap[
        "JointCluster"
    ]
)


# ============================================================
# 26. Save outputs
# ============================================================

membership_confidence.to_csv(
    OUT_DIR
    / "reweight_k2_membership_confidence.csv",
    index=False
)


centroid_drivers.to_csv(
    OUT_DIR
    / "reweight_k2_centroid_drivers.csv",
    index=False
)


feature_separation.to_csv(
    OUT_DIR
    / "reweight_k2_feature_separation.csv",
    index=False
)


subtype_separation.to_csv(
    OUT_DIR
    / "reweight_k2_subtype_separation.csv",
    index=False
)


separation_summary.to_csv(
    OUT_DIR
    / "reweight_k2_separation_summary.csv",
    index=False
)


contrast_group_summary.to_csv(
    OUT_DIR
    / "reweight_k2_contrast_group_summary.csv",
    index=False
)


contrast_group_differences.to_csv(
    OUT_DIR
    / "reweight_k2_contrast_group_differences.csv",
    index=False
)


continuous_macro_summary.to_csv(
    OUT_DIR
    / "reweight_k2_continuous_macro_summary.csv",
    index=False
)


categorical_macro_summary.to_csv(
    OUT_DIR
    / "reweight_k2_categorical_macro_summary.csv",
    index=False
)


overlap.to_csv(
    OUT_DIR
    / "reweight_k2_joint5_overlap_members.csv",
    index=False
)


overlap_table.to_csv(
    OUT_DIR
    / "reweight_k2_joint5_overlap_counts.csv"
)


overlap_row_proportions.to_csv(
    OUT_DIR
    / "reweight_k2_joint5_overlap_row_proportions.csv"
)


# ============================================================
# 27. Print robustness
# ============================================================

print("\n")
print("=" * 120)
print("REWEIGHT K=2 BOOTSTRAP ROBUSTNESS")
print("=" * 120)


print(
    f"Mean bootstrap ARI: "
    f"{bootstrap_ari.mean():.4f}"
)


print(
    f"Median bootstrap ARI: "
    f"{np.median(bootstrap_ari):.4f}"
)


print(
    f"2.5th percentile ARI: "
    f"{np.quantile(bootstrap_ari, 0.025):.4f}"
)


print(
    f"97.5th percentile ARI: "
    f"{np.quantile(bootstrap_ari, 0.975):.4f}"
)


print(
    f"Mean membership probability: "
    f"{membership_probability.mean():.4f}"
)


print(
    f"Core >= .90: "
    f"{np.mean(membership_probability >= 0.90) * 100:.2f}%"
)


print(
    f"Stable/Core >= .75: "
    f"{np.mean(membership_probability >= 0.75) * 100:.2f}%"
)


print(
    f"Boundary < .60: "
    f"{np.mean(membership_probability < 0.60) * 100:.2f}%"
)


# ============================================================
# 28. Print memberships
# ============================================================

print("\n")
print("=" * 120)
print("REWEIGHT K=2 MEMBERSHIP")
print("=" * 120)


for cluster in [
    1,
    2
]:

    print("\n")
    print(
        f"CLUSTER {cluster}"
    )

    print("-" * 90)


    temp = (
        membership_confidence[
            membership_confidence[
                "Cluster"
            ]
            ==
            cluster
        ]
        .sort_values(
            "MembershipProbability",
            ascending=False
        )
    )


    print(
        temp[
            [
                "Country",
                "MembershipProbability",
                "Confidence"
            ]
        ]
        .round(
            4
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 29. Print biggest subtype x predictor differences
# ============================================================

print("\n")
print("=" * 120)
print("TOP SUBTYPE × PREDICTOR DIFFERENCES BETWEEN CLUSTERS")
print("=" * 120)


print(
    centroid_drivers
    .sort_values(
        "AbsDifference",
        ascending=False
    )
    .head(
        25
    )[
        [
            "Subtype",
            "Feature",
            "Cluster1Mean",
            "Cluster2Mean",
            "Cluster2MinusCluster1",
            "PctOfSquaredSeparation"
        ]
    ]
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 30. Feature-level separation
# ============================================================

print("\n")
print("=" * 120)
print("PREDICTORS CONTRIBUTING MOST TO K=2 SEPARATION")
print("=" * 120)


print(
    feature_separation
    .round(
        3
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 31. Subtype-level separation
# ============================================================

print("\n")
print("=" * 120)
print("BULLYING SUBTYPES CONTRIBUTING MOST TO K=2 SEPARATION")
print("=" * 120)


print(
    subtype_separation
    .round(
        3
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 32. Relational-gender dominance
# ============================================================

print("\n")
print("=" * 120)
print("HOW MUCH OF THE K=2 SPLIT IS THE RELATIONAL-GENDER PATTERN?")
print("=" * 120)


print(
    separation_summary
    .round(
        3
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 33. Most different interpretable contrasts
# ============================================================

print("\n")
print("=" * 120)
print("TOP INTERPRETABLE SUBTYPE-CONTRAST DIFFERENCES")
print("=" * 120)


print(
    contrast_group_differences
    .sort_values(
        "AbsDifference",
        ascending=False
    )
    .head(
        30
    )
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 34. Continuous macro profile
# ============================================================

print("\n")
print("=" * 120)
print("K=2 CONTINUOUS MACRO CHARACTERISTICS")
print("=" * 120)


print(
    continuous_macro_summary
    .round(
        2
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 35. Income profile
# ============================================================

print("\n")
print("=" * 120)
print("K=2 INCOME COMPOSITION")
print("=" * 120)


print(
    categorical_macro_summary[
        categorical_macro_summary[
            "Variable"
        ]
        ==
        "IncomeGroup_FY2019"
    ]
    .sort_values(
        [
            "Cluster",
            "Proportion"
        ],
        ascending=[
            True,
            False
        ]
    )
    .round(
        3
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 36. Region profile
# ============================================================

print("\n")
print("=" * 120)
print("K=2 REGIONAL COMPOSITION")
print("=" * 120)


print(
    categorical_macro_summary[
        categorical_macro_summary[
            "Variable"
        ]
        ==
        "WB_Region"
    ]
    .sort_values(
        [
            "Cluster",
            "Proportion"
        ],
        ascending=[
            True,
            False
        ]
    )
    .round(
        3
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 37. GDP / inequality regime composition
# ============================================================

print("\n")
print("=" * 120)
print("K=2 GDP × INEQUALITY COMPOSITION")
print("=" * 120)


print(
    categorical_macro_summary[
        categorical_macro_summary[
            "Variable"
        ]
        ==
        "GDP_Gini_Quadrant"
    ]
    .sort_values(
        [
            "Cluster",
            "Proportion"
        ],
        ascending=[
            True,
            False
        ]
    )
    .round(
        3
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 38. JOINT5 overlap
# ============================================================

print("\n")
print("=" * 120)
print("OVERLAP WITH JOINT5 K=3")
print("=" * 120)


print(
    "\nCounts:"
)


print(
    overlap_table
    .to_string()
)


print(
    "\nRow proportions:"
)


print(
    overlap_row_proportions
    .round(
        3
    )
    .to_string()
)


print(
    f"\nAdjusted Rand Index: "
    f"{overlap_ari:.4f}"
)


# ============================================================
# 39. Boundary systems
# ============================================================

print("\n")
print("=" * 120)
print("UNCERTAIN / BOUNDARY SYSTEMS")
print("=" * 120)


boundary = (
    membership_confidence[
        membership_confidence[
            "MembershipProbability"
        ]
        <
        0.75
    ]
    .sort_values(
        "MembershipProbability"
    )
)


if len(
    boundary
) == 0:

    print(
        "No systems below 0.75 membership probability."
    )


else:

    print(
        boundary
        .round(
            4
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 40. Final status
# ============================================================

print("\n")
print("=" * 120)
print("FINAL REWEIGHTING K=2 CHARACTERIZATION COMPLETE")
print("=" * 120)


print(
    "Cluster sizes:"
)


print(
    membership_confidence[
        "Cluster"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


print(
    "\nResults saved to:"
)


print(
    OUT_DIR
)

Common systems: 75
PCA components used: 16


REWEIGHT K=2 BOOTSTRAP ROBUSTNESS
Mean bootstrap ARI: 0.9982
Median bootstrap ARI: 1.0000
2.5th percentile ARI: 0.9403
97.5th percentile ARI: 1.0000
Mean membership probability: 0.9995
Core >= .90: 100.00%
Stable/Core >= .75: 100.00%
Boundary < .60: 0.00%


REWEIGHT K=2 MEMBERSHIP


CLUSTER 1
------------------------------------------------------------------------------------------
Country  MembershipProbability Confidence
    ARG                 1.0000       Core
    AUS                 1.0000       Core
    AUT                 1.0000       Core
    BIH                 1.0000       Core
    BLR                 1.0000       Core
    CAN                 1.0000       Core
    CHE                 1.0000       Core
    CHL                 1.0000       Core
    CZE                 1.0000       Core
    DEU                 1.0000       Core
    DNK                 1.0000       Core
    DOM                 1.0000       Core
    ESP                 

In [5]:
# ============================================================
# FINAL CONTEXT-PATTERN SCREEN
#
# NEW INTERNATIONAL CONTEXT VARIABLES
# ============================================================
#
# PURPOSE
# -------
# Test whether additional system-level characteristics explain
# heterogeneity in:
#
#   A) baseline bullying-predictor architecture across all
#      five outcomes
#
#   B) subtype-specific reweighting architecture
#
#
# NEW CONTEXT VARIABLES
# ---------------------
# CONTINUOUS
#
#   1. HDI 2018
#   2. Gender Inequality Index (GII) 2018
#   3. Internet users (% population)
#   4. Urban population (% population)
#   5. Government effectiveness
#   6. Education expenditure (% GDP)
#
# CATEGORICAL
#
#   7. OECD member in 2018
#   8. Post-communist / post-socialist Europe-Eurasia
#
#
# OPTIONAL
# --------
# If a system-level PISA performance CSV already exists in the
# repository, this cell will try to detect and include it.
#
#
# TWO LEVELS OF TESTING
# ---------------------
#
# 1. UNADJUSTED:
#
#       architecture ~ context variable
#
#
# 2. PARTIAL / ADJUSTED:
#
#       architecture ~ GDP + region + context variable
#
#
# This is critical.
#
# If HDI is associated with architecture but adds nothing
# after GDP + region, then HDI is probably describing the
# same broad development gradient we already found.
#
# If something such as GII, internet penetration, or
# post-communist status remains associated AFTER adjustment,
# that is genuinely new information.
#
#
# INFERENCE
# ---------
# Freedman-Lane residual permutation for adjusted tests.
#
# 2,000 permutations here because this is an exploratory
# screening stage.
#
# Any important finalist should later be confirmed with
# 10,000 permutations.
#
# ============================================================


from pathlib import Path
import warnings
import requests

import numpy as np
import pandas as pd


# ============================================================
# 1. Configuration
# ============================================================

RESULT_ROOT = Path(
    "../results"
)


RANKING_PATH = (
    RESULT_ROOT
    / "primary_attribution"
    / "oof_shap_rankings.csv"
)


EXISTING_MACRO_PATH = (
    RESULT_ROOT
    / "cluster_discovery"
    / "macro_validators_with_derived_groups.csv"
)


OUT_DIR = (
    RESULT_ROOT
    / "extended_context_patterns"
)


OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


FEATURES = [
    "ADAPTIVITY",
    "AGE",
    "ATTLNACT",
    "BELONG",
    "DISCLIMA",
    "EDUSHORT",
    "ESCS",
    "JOYREAD",
    "MASTGOAL",
    "METASPAM",
    "METASUM",
    "ST004D01T",
    "STIMREAD",
    "UNDREM",
]


OUTCOMES = [
    "BEINGBULLIED",
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


SUBTYPES = [
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


N_PERMUTATIONS = 2_000

RANDOM_SEED = 42


UNDP_URL = (
    "https://hdr.undp.org/sites/default/files/"
    "2025_HDR/"
    "HDR25_Composite_indices_complete_time_series.csv"
)


# ============================================================
# 2. Load frozen attribution + existing macro data
# ============================================================

rankings = pd.read_csv(
    RANKING_PATH
)


macro = pd.read_csv(
    EXISTING_MACRO_PATH
)


rankings[
    "ImportanceShare"
] = pd.to_numeric(
    rankings[
        "ImportanceShare"
    ],
    errors="coerce"
)


if (
    rankings[
        "ImportanceShare"
    ]
    <=
    0
).any():

    raise RuntimeError(
        "All attribution shares must be > 0 for CLR."
    )


# ============================================================
# 3. PISA system code -> ISO3 mapping
#
# QAZ = Baku (Azerbaijan)
# QCI = B-S-J-Z (China)
# TAP = Chinese Taipei
#
# We intentionally do NOT assign national macro indicators
# to these special PISA entities.
# ============================================================

SYSTEM_TO_ISO3 = {

    "ALB": "ALB",
    "ARE": "ARE",
    "ARG": "ARG",
    "AUS": "AUS",
    "AUT": "AUT",
    "BEL": "BEL",
    "BGR": "BGR",
    "BIH": "BIH",
    "BLR": "BLR",
    "BRA": "BRA",
    "BRN": "BRN",
    "CAN": "CAN",
    "CHE": "CHE",
    "CHL": "CHL",
    "COL": "COL",
    "CRI": "CRI",
    "CZE": "CZE",
    "DEU": "DEU",
    "DNK": "DNK",
    "DOM": "DOM",
    "ESP": "ESP",
    "EST": "EST",
    "FIN": "FIN",
    "FRA": "FRA",
    "GBR": "GBR",
    "GEO": "GEO",
    "GRC": "GRC",
    "HKG": "HKG",
    "HRV": "HRV",
    "HUN": "HUN",
    "IDN": "IDN",
    "IRL": "IRL",
    "ISL": "ISL",
    "ITA": "ITA",
    "JOR": "JOR",
    "JPN": "JPN",
    "KAZ": "KAZ",
    "KOR": "KOR",
    "KSV": "XKX",
    "LTU": "LTU",
    "LUX": "LUX",
    "LVA": "LVA",
    "MAC": "MAC",
    "MAR": "MAR",
    "MDA": "MDA",
    "MEX": "MEX",
    "MLT": "MLT",
    "MNE": "MNE",
    "MYS": "MYS",
    "NLD": "NLD",
    "NOR": "NOR",
    "NZL": "NZL",
    "PAN": "PAN",
    "PER": "PER",
    "PHL": "PHL",
    "POL": "POL",
    "PRT": "PRT",
    "QAT": "QAT",
    "ROU": "ROU",
    "RUS": "RUS",
    "SAU": "SAU",
    "SGP": "SGP",
    "SRB": "SRB",
    "SVK": "SVK",
    "SVN": "SVN",
    "SWE": "SWE",
    "THA": "THA",
    "TUR": "TUR",
    "UKR": "UKR",
    "URY": "URY",
    "USA": "USA",
    "VNM": "VNM",

    # intentionally no national macro assignment
    "QAZ": None,
    "QCI": None,
    "TAP": None,
}


macro[
    "ISO3"
] = (
    macro[
        "Country"
    ]
    .map(
        SYSTEM_TO_ISO3
    )
)


# ============================================================
# 4. UNDP HDI + GII
#
# Official UNDP time-series file.
#
# Expected columns include:
#
#   iso3
#   hdi_2018
#   gii_2018
#
# We detect columns robustly.
# ============================================================

print(
    "Downloading UNDP human-development data..."
)


try:

    hdr = pd.read_csv(
        UNDP_URL,
        low_memory=False
    )


    hdr.columns = [
        str(
            column
        )
        .strip()
        .lower()
        for column
        in hdr.columns
    ]


    iso_candidates = [
        "iso3",
        "iso3_code",
        "country_code"
    ]


    undp_iso_col = next(
        (
            column
            for column
            in iso_candidates
            if column
            in hdr.columns
        ),
        None
    )


    if undp_iso_col is None:

        raise RuntimeError(
            "Could not identify ISO3 column "
            "in UNDP dataset."
        )


    def find_undp_year_column(
        prefix,
        year
    ):

        exact = (
            f"{prefix}_{year}"
        )


        if exact in hdr.columns:

            return exact


        matches = [
            column
            for column
            in hdr.columns
            if (
                prefix
                in column
                and
                str(
                    year
                )
                in column
                and
                "rank"
                not in column
            )
        ]


        return (
            matches[
                0
            ]
            if len(
                matches
            )
            >
            0
            else None
        )


    hdi_col = find_undp_year_column(
        "hdi",
        2018
    )


    gii_col = find_undp_year_column(
        "gii",
        2018
    )


    print(
        "UNDP HDI column:",
        hdi_col
    )


    print(
        "UNDP GII column:",
        gii_col
    )


    undp_keep = [
        undp_iso_col
    ]


    if hdi_col is not None:

        undp_keep.append(
            hdi_col
        )


    if gii_col is not None:

        undp_keep.append(
            gii_col
        )


    undp = (
        hdr[
            undp_keep
        ]
        .copy()
    )


    rename_map = {
        undp_iso_col:
            "ISO3"
    }


    if hdi_col is not None:

        rename_map[
            hdi_col
        ] = (
            "HDI_2018"
        )


    if gii_col is not None:

        rename_map[
            gii_col
        ] = (
            "GII_2018"
        )


    undp = undp.rename(
        columns=rename_map
    )


    undp[
        "ISO3"
    ] = (
        undp[
            "ISO3"
        ]
        .astype(
            str
        )
        .str.upper()
        .str.strip()
    )


    for column in [
        "HDI_2018",
        "GII_2018"
    ]:

        if column in undp.columns:

            undp[
                column
            ] = pd.to_numeric(
                undp[
                    column
                ],
                errors="coerce"
            )


    macro = (
        macro
        .merge(
            undp,
            on="ISO3",
            how="left",
            validate="many_to_one"
        )
    )


except Exception as error:

    warnings.warn(
        f"UNDP download failed: {error}"
    )


# ============================================================
# 5. World Bank indicator downloader
#
# We use values near 2018.
#
# For indicators that may not be reported every year,
# select the observation closest to 2018 from 2016-2020.
# ============================================================

WB_INDICATORS = {

    "InternetUsersPct":
        "IT.NET.USER.ZS",

    "UrbanPopulationPct":
        "SP.URB.TOTL.IN.ZS",

    "GovernmentEffectiveness":
        "GE.EST",

    "EducationExpenditurePctGDP":
        "SE.XPD.TOTL.GD.ZS",
}


valid_iso_codes = sorted(
    macro[
        "ISO3"
    ]
    .dropna()
    .astype(
        str
    )
    .unique()
)


def fetch_world_bank_indicator(
    iso_codes,
    indicator,
    target_year=2018,
    start_year=2016,
    end_year=2020
):

    codes = ";".join(
        iso_codes
    )


    url = (
        "https://api.worldbank.org/v2/"
        f"country/{codes}/"
        f"indicator/{indicator}"
        f"?format=json"
        f"&date={start_year}:{end_year}"
        f"&per_page=20000"
    )


    response = requests.get(
        url,
        timeout=60
    )


    response.raise_for_status()


    payload = response.json()


    if (
        not isinstance(
            payload,
            list
        )
        or
        len(
            payload
        )
        <
        2
        or
        payload[
            1
        ]
        is None
    ):

        return pd.DataFrame(
            columns=[
                "ISO3",
                "Year",
                "Value"
            ]
        )


    rows = []


    for item in payload[
        1
    ]:

        value = item.get(
            "value"
        )


        if value is None:

            continue


        iso3 = item.get(
            "countryiso3code"
        )


        year = item.get(
            "date"
        )


        try:

            year = int(
                year
            )


            value = float(
                value
            )


        except Exception:

            continue


        rows.append(
            {
                "ISO3":
                    iso3,

                "Year":
                    year,

                "Value":
                    value,

                "YearDistance":
                    abs(
                        year
                        -
                        target_year
                    ),
            }
        )


    frame = pd.DataFrame(
        rows
    )


    if len(
        frame
    ) == 0:

        return frame


    # prefer:
    # 1) closest year to 2018
    # 2) if tied, earlier year
    frame = (
        frame
        .sort_values(
            [
                "ISO3",
                "YearDistance",
                "Year"
            ],
            ascending=[
                True,
                True,
                True
            ]
        )
        .groupby(
            "ISO3",
            as_index=False
        )
        .first()
    )


    return frame


print(
    "\nDownloading World Bank indicators..."
)


for variable, indicator in (
    WB_INDICATORS.items()
):

    print(
        variable,
        indicator
    )


    try:

        frame = (
            fetch_world_bank_indicator(
                valid_iso_codes,
                indicator
            )
        )


        if len(
            frame
        ) == 0:

            print(
                "  no data returned"
            )

            macro[
                variable
            ] = np.nan


            macro[
                variable
                +
                "_Year"
            ] = np.nan


            continue


        frame = frame.rename(
            columns={
                "Value":
                    variable,

                "Year":
                    (
                        variable
                        +
                        "_Year"
                    ),
            }
        )


        frame = frame[
            [
                "ISO3",
                variable,
                variable
                +
                "_Year",
            ]
        ]


        macro = (
            macro
            .merge(
                frame,
                on="ISO3",
                how="left",
                validate="many_to_one"
            )
        )


    except Exception as error:

        warnings.warn(
            f"{variable} download failed: {error}"
        )


        macro[
            variable
        ] = np.nan


        macro[
            variable
            +
            "_Year"
        ] = np.nan


# ============================================================
# 6. OECD membership in 2018
#
# Use year-specific membership rather than today's membership.
# ============================================================

OECD_2018 = {

    "AUS",
    "AUT",
    "BEL",
    "CAN",
    "CHL",
    "CZE",
    "DNK",
    "EST",
    "FIN",
    "FRA",
    "DEU",
    "GRC",
    "HUN",
    "ISL",
    "IRL",
    "ITA",
    "JPN",
    "KOR",
    "LVA",
    "LTU",
    "LUX",
    "MEX",
    "NLD",
    "NZL",
    "NOR",
    "POL",
    "PRT",
    "SVK",
    "SVN",
    "ESP",
    "SWE",
    "CHE",
    "TUR",
    "GBR",
    "USA",
}


macro[
    "OECD_2018"
] = (
    macro[
        "ISO3"
    ]
    .apply(
        lambda x:
            (
                "OECD"
                if x
                in OECD_2018
                else
                (
                    "NonOECD"
                    if pd.notna(
                        x
                    )
                    else np.nan
                )
            )
    )
)


# ============================================================
# 7. Post-communist / post-socialist historical grouping
#
# Restricted to European/Eurasian systems that experienced
# communist/socialist state structures before the transitions
# around 1989-1991.
#
# This is deliberately narrower than:
#
#       communist vs capitalist
#
# because countries such as Vietnam remain socialist states
# and represent a different historical classification.
#
# Germany is not coded as post-communist because the PISA
# system is unified Germany rather than former East Germany.
# ============================================================

POST_COMMUNIST_SYSTEMS = {

    "ALB",
    "BGR",
    "BIH",
    "BLR",
    "CZE",
    "EST",
    "GEO",
    "HRV",
    "HUN",
    "KAZ",
    "KSV",
    "LTU",
    "LVA",
    "MDA",
    "MNE",
    "POL",
    "ROU",
    "RUS",
    "SRB",
    "SVK",
    "SVN",
    "UKR",

    # Baku/Azerbaijan PISA entity
    "QAZ",
}


macro[
    "PostCommunist"
] = (
    macro[
        "Country"
    ]
    .apply(
        lambda x:
            (
                "PostCommunist"
                if x
                in POST_COMMUNIST_SYSTEMS
                else
                "Other"
            )
    )
)


# ============================================================
# 8. HDI descriptive categories
#
# Standard HDI cut-points:
#
# low       < .550
# medium    .550-.699
# high      .700-.799
# very high >= .800
#
# Inference below still uses continuous HDI.
# ============================================================

if (
    "HDI_2018"
    in macro.columns
):

    macro[
        "HDI_Group"
    ] = pd.cut(
        macro[
            "HDI_2018"
        ],
        bins=[
            -np.inf,
            0.550,
            0.700,
            0.800,
            np.inf
        ],
        labels=[
            "Low",
            "Medium",
            "High",
            "VeryHigh"
        ],
        right=False
    )


# ============================================================
# 9. Optional PISA-performance file
#
# If we already created one somewhere in the project, include
# it. Otherwise skip it for now rather than using an
# inconsistent external aggregate.
#
# Expected possibilities:
#
#   Country + PISA_Mean_2018
#
# OR
#
#   Country + ReadingMean + MathMean + ScienceMean
# ============================================================

PISA_FILE_CANDIDATES = [

    RESULT_ROOT
    / "system_pisa_performance_2018.csv",

    RESULT_ROOT
    / "pisa_system_performance_2018.csv",

    RESULT_ROOT
    / "system_achievement_means.csv",
]


pisa_file_used = None


for path in PISA_FILE_CANDIDATES:

    if path.exists():

        try:

            perf = pd.read_csv(
                path
            )


            if (
                "Country"
                not in perf.columns
            ):

                continue


            if (
                "PISA_Mean_2018"
                in perf.columns
            ):

                perf_keep = (
                    perf[
                        [
                            "Country",
                            "PISA_Mean_2018"
                        ]
                    ]
                    .copy()
                )


            else:

                reading_candidates = [
                    "ReadingMean",
                    "READ_Mean",
                    "PVREAD_Mean"
                ]


                math_candidates = [
                    "MathMean",
                    "MATH_Mean",
                    "PVMATH_Mean"
                ]


                science_candidates = [
                    "ScienceMean",
                    "SCIE_Mean",
                    "PVSCIE_Mean"
                ]


                read_col = next(
                    (
                        c
                        for c
                        in reading_candidates
                        if c
                        in perf.columns
                    ),
                    None
                )


                math_col = next(
                    (
                        c
                        for c
                        in math_candidates
                        if c
                        in perf.columns
                    ),
                    None
                )


                science_col = next(
                    (
                        c
                        for c
                        in science_candidates
                        if c
                        in perf.columns
                    ),
                    None
                )


                if (
                    read_col is None
                    or
                    math_col is None
                    or
                    science_col is None
                ):

                    continue


                perf_keep = (
                    perf[
                        [
                            "Country",
                            read_col,
                            math_col,
                            science_col
                        ]
                    ]
                    .copy()
                )


                perf_keep[
                    "PISA_Mean_2018"
                ] = (
                    perf_keep[
                        [
                            read_col,
                            math_col,
                            science_col
                        ]
                    ]
                    .mean(
                        axis=1
                    )
                )


                perf_keep = perf_keep[
                    [
                        "Country",
                        "PISA_Mean_2018"
                    ]
                ]


            macro = (
                macro
                .merge(
                    perf_keep,
                    on="Country",
                    how="left",
                    validate="one_to_one"
                )
            )


            pisa_file_used = path


            break


        except Exception:

            pass


if pisa_file_used is None:

    print(
        "\nPISA performance file not found."
    )

    print(
        "Skipping PISA-performance validator "
        "in this screening cell."
    )


else:

    print(
        "\nUsing PISA performance file:",
        pisa_file_used
    )


# ============================================================
# 10. Save enriched metadata
# ============================================================

macro.to_csv(
    OUT_DIR
    / "extended_context_metadata.csv",
    index=False
)


# ============================================================
# 11. CLR helper
# ============================================================

def clr_transform(
    matrix
):

    matrix = np.asarray(
        matrix,
        dtype=float
    )


    matrix = (
        matrix
        /
        matrix.sum(
            axis=1,
            keepdims=True
        )
    )


    logs = np.log(
        matrix
    )


    return (
        logs
        -
        logs.mean(
            axis=1,
            keepdims=True
        )
    )


# ============================================================
# 12. Build outcome CLR matrices
# ============================================================

outcome_store = {}


for outcome in OUTCOMES:

    temp = (
        rankings[
            rankings[
                "Outcome"
            ]
            ==
            outcome
        ]
    )


    systems = sorted(
        temp[
            "Country"
        ]
        .unique()
    )


    rows = []


    for country in systems:

        vector = (
            temp[
                temp[
                    "Country"
                ]
                ==
                country
            ]
            .set_index(
                "Feature"
            )
            .loc[
                FEATURES,
                "ImportanceShare"
            ]
            .to_numpy(
                dtype=float
            )
        )


        vector = (
            vector
            /
            vector.sum()
        )


        rows.append(
            vector
        )


    composition = np.vstack(
        rows
    )


    outcome_store[
        outcome
    ] = {

        "systems":
            systems,

        "clr":
            clr_transform(
                composition
            )
    }


# ============================================================
# 13. RESPONSE A:
# Five-outcome baseline architecture
#
# Only systems with all 5 outcome profiles.
#
# Each outcome CLR block is divided by the square root of its
# total variance so every outcome contributes equally.
# ============================================================

baseline_systems = sorted(
    set.intersection(
        *[
            set(
                outcome_store[
                    outcome
                ][
                    "systems"
                ]
            )
            for outcome
            in OUTCOMES
        ]
    )
)


baseline_blocks = []


for outcome in OUTCOMES:

    result = (
        outcome_store[
            outcome
        ]
    )


    index_map = {
        country: i
        for i, country
        in enumerate(
            result[
                "systems"
            ]
        )
    }


    block = np.vstack(
        [
            result[
                "clr"
            ][
                index_map[
                    country
                ]
            ]
            for country
            in baseline_systems
        ]
    )


    total_variance = np.sum(
        np.var(
            block,
            axis=0,
            ddof=1
        )
    )


    block = (
        block
        /
        np.sqrt(
            total_variance
        )
    )


    baseline_blocks.append(
        block
    )


baseline_joint = np.hstack(
    baseline_blocks
)


# ============================================================
# 14. RESPONSE B:
# Four-subtype REWEIGHTING architecture
#
# Remove each system's mean architecture across subtypes.
# ============================================================

reweight_systems = sorted(
    set.intersection(
        *[
            set(
                outcome_store[
                    subtype
                ][
                    "systems"
                ]
            )
            for subtype
            in SUBTYPES
        ]
    )
)


reweight_tensor = np.empty(
    (
        len(
            reweight_systems
        ),
        len(
            SUBTYPES
        ),
        len(
            FEATURES
        )
    )
)


for subtype_index, subtype in enumerate(
    SUBTYPES
):

    result = (
        outcome_store[
            subtype
        ]
    )


    index_map = {
        country: i
        for i, country
        in enumerate(
            result[
                "systems"
            ]
        )
    }


    reweight_tensor[
        :,
        subtype_index,
        :
    ] = np.vstack(
        [
            result[
                "clr"
            ][
                index_map[
                    country
                ]
            ]
            for country
            in reweight_systems
        ]
    )


system_average = (
    reweight_tensor.mean(
        axis=1,
        keepdims=True
    )
)


reweight_tensor = (
    reweight_tensor
    -
    system_average
)


reweight_signature = (
    reweight_tensor.reshape(
        len(
            reweight_systems
        ),
        -1
    )
)


RESPONSES = {

    "BaselineJoint5": {
        "systems":
            baseline_systems,

        "Y":
            baseline_joint,
    },

    "SubtypeReweighting": {
        "systems":
            reweight_systems,

        "Y":
            reweight_signature,
    },
}


# ============================================================
# 15. Candidate validators
# ============================================================

CONTINUOUS_VALIDATORS = [

    "HDI_2018",

    "GII_2018",

    "InternetUsersPct",

    "UrbanPopulationPct",

    "GovernmentEffectiveness",

    "EducationExpenditurePctGDP",
]


if (
    "PISA_Mean_2018"
    in macro.columns
):

    CONTINUOUS_VALIDATORS.append(
        "PISA_Mean_2018"
    )


CATEGORICAL_VALIDATORS = [

    "OECD_2018",

    "PostCommunist",
]


# Remove variables that failed to download

CONTINUOUS_VALIDATORS = [
    variable
    for variable
    in CONTINUOUS_VALIDATORS
    if variable
    in macro.columns
]


CATEGORICAL_VALIDATORS = [
    variable
    for variable
    in CATEGORICAL_VALIDATORS
    if variable
    in macro.columns
]


# ============================================================
# 16. Coverage summary
# ============================================================

coverage_rows = []


for variable in (
    CONTINUOUS_VALIDATORS
    +
    CATEGORICAL_VALIDATORS
):

    coverage_rows.append(
        {
            "Variable":
                variable,

            "N_available":
                int(
                    macro[
                        variable
                    ]
                    .notna()
                    .sum()
                ),

            "Pct_available":
                100
                *
                macro[
                    variable
                ]
                .notna()
                .mean(),

            "N_unique":
                macro[
                    variable
                ]
                .nunique(
                    dropna=True
                ),
        }
    )


validator_coverage = pd.DataFrame(
    coverage_rows
)


# ============================================================
# 17. Descriptive correlation among continuous validators
#
# This helps identify redundant "development proxies".
# ============================================================

correlation_columns = [

    "LogGDPpcPPP_2018"
]


for variable in CONTINUOUS_VALIDATORS:

    if variable not in correlation_columns:

        correlation_columns.append(
            variable
        )


correlation_columns = [
    column
    for column
    in correlation_columns
    if column
    in macro.columns
]


validator_correlations = (
    macro[
        correlation_columns
    ]
    .corr(
        method="spearman"
    )
)


# ============================================================
# 18. Benjamini-Hochberg FDR
# ============================================================

def bh_fdr(
    p_values
):

    p = np.asarray(
        p_values,
        dtype=float
    )


    output = np.full(
        len(
            p
        ),
        np.nan
    )


    valid = np.isfinite(
        p
    )


    pv = p[
        valid
    ]


    if len(
        pv
    ) == 0:

        return output


    order = np.argsort(
        pv
    )


    ranked = pv[
        order
    ]


    m = len(
        ranked
    )


    adjusted = (
        ranked
        *
        m
        /
        np.arange(
            1,
            m
            +
            1
        )
    )


    adjusted = np.minimum.accumulate(
        adjusted[
            ::-1
        ]
    )[
        ::-1
    ]


    adjusted = np.minimum(
        adjusted,
        1
    )


    unsorted = np.empty(
        m
    )


    unsorted[
        order
    ] = adjusted


    output[
        valid
    ] = unsorted


    return output


# ============================================================
# 19. Design-matrix helpers
# ============================================================

def zscore(
    values
):

    values = np.asarray(
        values,
        dtype=float
    )


    sd = values.std(
        ddof=1
    )


    if (
        not np.isfinite(
            sd
        )
        or
        sd
        ==
        0
    ):

        return np.zeros_like(
            values,
            dtype=float
        )


    return (
        values
        -
        values.mean()
    ) / sd


def categorical_dummies(
    values
):

    values = pd.Series(
        values,
        dtype="string"
    )


    dummies = pd.get_dummies(
        values,
        drop_first=True,
        dtype=float
    )


    return dummies.to_numpy(
        dtype=float
    )


def build_base_design(
    frame,
    adjusted
):

    n = len(
        frame
    )


    parts = [
        np.ones(
            (
                n,
                1
            )
        )
    ]


    if adjusted:

        gdp = zscore(
            frame[
                "LogGDPpcPPP_2018"
            ]
            .to_numpy(
                dtype=float
            )
        )


        parts.append(
            gdp.reshape(
                -1,
                1
            )
        )


        region = categorical_dummies(
            frame[
                "WB_Region"
            ]
        )


        if region.shape[
            1
        ] > 0:

            parts.append(
                region
            )


    return np.hstack(
        parts
    )


def build_term_matrix(
    frame,
    variable,
    variable_type
):

    if variable_type == "Continuous":

        values = zscore(
            frame[
                variable
            ]
            .to_numpy(
                dtype=float
            )
        )


        return values.reshape(
            -1,
            1
        )


    elif variable_type == "Categorical":

        return categorical_dummies(
            frame[
                variable
            ]
        )


    else:

        raise ValueError(
            variable_type
        )


# ============================================================
# 20. Multivariate SSE helper
# ============================================================

def fit_sse(
    X,
    Y
):

    beta = np.linalg.lstsq(
        X,
        Y,
        rcond=None
    )[
        0
    ]


    fitted = (
        X
        @
        beta
    )


    residual = (
        Y
        -
        fitted
    )


    sse = float(
        np.sum(
            residual ** 2
        )
    )


    return (
        fitted,
        residual,
        sse
    )


# ============================================================
# 21. Freedman-Lane partial permutation test
#
# H0:
#
# the new context variable adds no multivariate information
# beyond the base model.
#
#
# Unadjusted:
#
#   base = intercept
#
#
# Adjusted:
#
#   base = intercept + log GDP + region
#
# ============================================================

def freedman_lane_test(
    Y,
    frame,
    variable,
    variable_type,
    adjusted,
    n_permutations,
    seed
):

    X_base = build_base_design(
        frame,
        adjusted=adjusted
    )


    X_term = build_term_matrix(
        frame,
        variable,
        variable_type
    )


    if X_term.shape[
        1
    ] == 0:

        return None


    X_full = np.hstack(
        [
            X_base,
            X_term
        ]
    )


    rank_base = np.linalg.matrix_rank(
        X_base
    )


    rank_full = np.linalg.matrix_rank(
        X_full
    )


    df_term = (
        rank_full
        -
        rank_base
    )


    df_resid = (
        len(
            frame
        )
        -
        rank_full
    )


    if (
        df_term
        <=
        0
        or
        df_resid
        <=
        0
    ):

        return None


    (
        fitted_base,
        residual_base,
        sse_base
    ) = fit_sse(
        X_base,
        Y
    )


    (
        _,
        _,
        sse_full
    ) = fit_sse(
        X_full,
        Y
    )


    partial_ss = (
        sse_base
        -
        sse_full
    )


    partial_r2 = (
        partial_ss
        /
        sse_base
        if sse_base
        >
        0
        else np.nan
    )


    observed_F = (
        (
            partial_ss
            /
            df_term
        )
        /
        (
            sse_full
            /
            df_resid
        )
    )


    rng = np.random.default_rng(
        seed
    )


    null_F = np.empty(
        n_permutations
    )


    for permutation in range(
        n_permutations
    ):

        order = rng.permutation(
            len(
                frame
            )
        )


        Y_perm = (
            fitted_base
            +
            residual_base[
                order
            ]
        )


        (
            _,
            _,
            perm_sse_base
        ) = fit_sse(
            X_base,
            Y_perm
        )


        (
            _,
            _,
            perm_sse_full
        ) = fit_sse(
            X_full,
            Y_perm
        )


        perm_partial_ss = (
            perm_sse_base
            -
            perm_sse_full
        )


        null_F[
            permutation
        ] = (
            (
                perm_partial_ss
                /
                df_term
            )
            /
            (
                perm_sse_full
                /
                df_resid
            )
        )


    p = (
        1
        +
        np.sum(
            null_F
            >=
            observed_F
        )
    ) / (
        n_permutations
        +
        1
    )


    return {

        "F":
            observed_F,

        "PartialR2":
            partial_r2,

        "PermutationP":
            p,

        "DF_Term":
            df_term,

        "DF_Residual":
            df_resid,
    }


# ============================================================
# 22. Run context-pattern screening
# ============================================================

result_rows = []

test_counter = 0


validators = []


for variable in CONTINUOUS_VALIDATORS:

    validators.append(
        (
            variable,
            "Continuous"
        )
    )


for variable in CATEGORICAL_VALIDATORS:

    validators.append(
        (
            variable,
            "Categorical"
        )
    )


for response_name, response in (
    RESPONSES.items()
):

    systems = (
        response[
            "systems"
        ]
    )


    Y_all = (
        response[
            "Y"
        ]
    )


    response_index = {
        country: i
        for i, country
        in enumerate(
            systems
        )
    }


    for variable, variable_type in validators:

        # ----------------------------------------------------
        # UNADJUSTED
        # ----------------------------------------------------

        valid_meta = (
            macro[
                [
                    "Country",
                    variable
                ]
            ]
            .dropna()
        )


        common = [
            country
            for country
            in systems
            if country
            in set(
                valid_meta[
                    "Country"
                ]
            )
        ]


        if len(
            common
        ) < 20:

            continue


        frame = (
            macro[
                macro[
                    "Country"
                ]
                .isin(
                    common
                )
            ][
                [
                    "Country",
                    variable
                ]
            ]
            .set_index(
                "Country"
            )
            .loc[
                common
            ]
            .reset_index()
        )


        Y = np.vstack(
            [
                Y_all[
                    response_index[
                        country
                    ]
                ]
                for country
                in common
            ]
        )


        test_counter += 1


        result = freedman_lane_test(

            Y=Y,

            frame=frame,

            variable=variable,

            variable_type=variable_type,

            adjusted=False,

            n_permutations=N_PERMUTATIONS,

            seed=(
                RANDOM_SEED
                +
                test_counter
            )
        )


        if result is not None:

            result_rows.append(
                {
                    "Response":
                        response_name,

                    "Validator":
                        variable,

                    "ValidatorType":
                        variable_type,

                    "Test":
                        "Unadjusted",

                    "N_systems":
                        len(
                            common
                        ),

                    **result
                }
            )


        # ----------------------------------------------------
        # ADJUSTED:
        #
        # context variable beyond GDP + region
        # ----------------------------------------------------

        required_columns = [

            "Country",
            variable,
            "LogGDPpcPPP_2018",
            "WB_Region"
        ]


        adjusted_meta = (
            macro[
                required_columns
            ]
            .dropna()
        )


        common_adjusted = [
            country
            for country
            in systems
            if country
            in set(
                adjusted_meta[
                    "Country"
                ]
            )
        ]


        if len(
            common_adjusted
        ) < 25:

            continue


        frame_adjusted = (
            macro[
                macro[
                    "Country"
                ]
                .isin(
                    common_adjusted
                )
            ][
                required_columns
            ]
            .set_index(
                "Country"
            )
            .loc[
                common_adjusted
            ]
            .reset_index()
        )


        Y_adjusted = np.vstack(
            [
                Y_all[
                    response_index[
                        country
                    ]
                ]
                for country
                in common_adjusted
            ]
        )


        test_counter += 1


        result_adjusted = freedman_lane_test(

            Y=Y_adjusted,

            frame=frame_adjusted,

            variable=variable,

            variable_type=variable_type,

            adjusted=True,

            n_permutations=N_PERMUTATIONS,

            seed=(
                RANDOM_SEED
                +
                10000
                +
                test_counter
            )
        )


        if result_adjusted is not None:

            result_rows.append(
                {
                    "Response":
                        response_name,

                    "Validator":
                        variable,

                    "ValidatorType":
                        variable_type,

                    "Test":
                        "AdjustedForGDPRegion",

                    "N_systems":
                        len(
                            common_adjusted
                        ),

                    **result_adjusted
                }
            )


context_results = pd.DataFrame(
    result_rows
)


# ============================================================
# 23. FDR
#
# Correct separately within:
#
#   response x test type
#
# so that the family represents:
#
# "Which additional contextual dimensions matter for this
# architecture representation?"
# ============================================================

context_results[
    "FDR_Q"
] = np.nan


for (
    response_name,
    test_name
), indices in (
    context_results
    .groupby(
        [
            "Response",
            "Test"
        ]
    )
    .groups
    .items()
):

    context_results.loc[
        indices,
        "FDR_Q"
    ] = bh_fdr(
        context_results.loc[
            indices,
            "PermutationP"
        ]
    )


# ============================================================
# 24. Identify potential NEW contextual findings
#
# Primary criterion:
#
# q < .05 AFTER GDP + REGION adjustment
# ============================================================

new_context_findings = (
    context_results[
        (
            context_results[
                "Test"
            ]
            ==
            "AdjustedForGDPRegion"
        )
        &
        (
            context_results[
                "FDR_Q"
            ]
            <
            0.05
        )
    ]
    .sort_values(
        [
            "Response",
            "FDR_Q",
            "PartialR2"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)


# ============================================================
# 25. Save outputs
# ============================================================

validator_coverage.to_csv(
    OUT_DIR
    / "validator_coverage.csv",
    index=False
)


validator_correlations.to_csv(
    OUT_DIR
    / "validator_spearman_correlations.csv"
)


context_results.to_csv(
    OUT_DIR
    / "extended_context_global_tests.csv",
    index=False
)


new_context_findings.to_csv(
    OUT_DIR
    / "extended_context_adjusted_significant.csv",
    index=False
)


# ============================================================
# 26. Print coverage
# ============================================================

print("\n")
print("=" * 120)
print("EXTENDED CONTEXT VALIDATOR COVERAGE")
print("=" * 120)


print(
    validator_coverage
    .round(
        2
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 27. Print source-year QA
# ============================================================

print("\n")
print("=" * 120)
print("WORLD BANK SOURCE-YEAR QA")
print("=" * 120)


for variable in (
    WB_INDICATORS.keys()
):

    year_column = (
        variable
        +
        "_Year"
    )


    if year_column not in macro.columns:

        continue


    values = (
        macro[
            year_column
        ]
        .dropna()
    )


    if len(
        values
    ) == 0:

        continue


    print(
        "\n",
        variable
    )


    print(
        values
        .value_counts()
        .sort_index()
        .to_string()
    )


# ============================================================
# 28. Correlation with GDP
# ============================================================

print("\n")
print("=" * 120)
print("CORRELATION OF NEW CONTEXT VARIABLES WITH GDP")
print("=" * 120)


if (
    "LogGDPpcPPP_2018"
    in validator_correlations.index
):

    print(
        validator_correlations[
            "LogGDPpcPPP_2018"
        ]
        .drop(
            labels=[
                "LogGDPpcPPP_2018"
            ],
            errors="ignore"
        )
        .sort_values(
            key=lambda x:
                x.abs(),
            ascending=False
        )
        .round(
            4
        )
        .to_string()
    )


# ============================================================
# 29. UNADJUSTED findings
# ============================================================

print("\n")
print("=" * 120)
print("UNADJUSTED EXTENDED-CONTEXT ASSOCIATIONS")
print("=" * 120)


print(
    context_results[
        context_results[
            "Test"
        ]
        ==
        "Unadjusted"
    ][
        [
            "Response",
            "Validator",
            "ValidatorType",
            "N_systems",
            "PartialR2",
            "F",
            "PermutationP",
            "FDR_Q"
        ]
    ]
    .sort_values(
        [
            "Response",
            "FDR_Q",
            "PartialR2"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 30. Most important table:
#
# DOES THE VARIABLE ADD INFORMATION BEYOND GDP + REGION?
# ============================================================

print("\n")
print("=" * 120)
print("CONTEXT ASSOCIATIONS AFTER ADJUSTING FOR GDP + REGION")
print("=" * 120)


print(
    context_results[
        context_results[
            "Test"
        ]
        ==
        "AdjustedForGDPRegion"
    ][
        [
            "Response",
            "Validator",
            "ValidatorType",
            "N_systems",
            "PartialR2",
            "F",
            "PermutationP",
            "FDR_Q"
        ]
    ]
    .sort_values(
        [
            "Response",
            "FDR_Q",
            "PartialR2"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 31. Finalists
# ============================================================

print("\n")
print("=" * 120)
print("POTENTIALLY NEW CONTEXTUAL PATTERNS")
print("=" * 120)


if len(
    new_context_findings
) == 0:

    print(
        "No new context variable survived FDR after "
        "adjusting for GDP + region."
    )


    print(
        "\nInterpretation if this holds:"
    )


    print(
        "The broader development/geographic gradients already "
        "identified appear sufficient to account for these "
        "additional contextual classifications."
    )


else:

    print(
        new_context_findings[
            [
                "Response",
                "Validator",
                "ValidatorType",
                "N_systems",
                "PartialR2",
                "F",
                "PermutationP",
                "FDR_Q"
            ]
        ]
        .round(
            5
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 32. HDI / income descriptive table
# ============================================================

if (
    "HDI_Group"
    in macro.columns
):

    print("\n")
    print("=" * 120)
    print("HDI GROUP COUNTS")
    print("=" * 120)


    print(
        macro[
            "HDI_Group"
        ]
        .value_counts(
            dropna=False
        )
        .to_string()
    )


# ============================================================
# 33. Historical / institutional group counts
# ============================================================

print("\n")
print("=" * 120)
print("OECD / POST-COMMUNIST GROUP COUNTS")
print("=" * 120)


for variable in [
    "OECD_2018",
    "PostCommunist"
]:

    print(
        "\n",
        variable
    )


    print(
        macro[
            variable
        ]
        .value_counts(
            dropna=False
        )
        .to_string()
    )


# ============================================================
# 34. Final status
# ============================================================

print("\n")
print("=" * 120)
print("EXTENDED CONTEXT SCREEN COMPLETE")
print("=" * 120)


print(
    "Continuous validators tested:",
    CONTINUOUS_VALIDATORS
)


print(
    "Categorical validators tested:",
    CATEGORICAL_VALIDATORS
)


print(
    "\nPrimary decision rule:"
)


print(
    "Prioritize variables with FDR q < .05 in the "
    "GDP + region adjusted analysis."
)


print(
    "\nDo NOT add further country classifications until "
    "these results are interpreted."
)

C:\Users\adhiv\AppData\Local\Temp\ipykernel_15800\3071831965.py:546: UserWarning: UNDP download failed: HTTP Error 403: Forbidden
  warnings.warn(



InternetUsersPct IT.NET.USER.ZS
UrbanPopulationPct SP.URB.TOTL.IN.ZS
GovernmentEffectiveness GE.EST
  no data returned
EducationExpenditurePctGDP SE.XPD.TOTL.GD.ZS

PISA performance file not found.
Skipping PISA-performance validator in this screening cell.


EXTENDED CONTEXT VALIDATOR COVERAGE
                  Variable  N_available  Pct_available  N_unique
          InternetUsersPct           72           96.0        72
        UrbanPopulationPct           72           96.0        70
   GovernmentEffectiveness            0            0.0         0
EducationExpenditurePctGDP           69           92.0        69
                 OECD_2018           72           96.0         2
             PostCommunist           75          100.0         2


WORLD BANK SOURCE-YEAR QA

 InternetUsersPct
InternetUsersPct_Year
2018.0    72

 UrbanPopulationPct
UrbanPopulationPct_Year
2018.0    72

 EducationExpenditurePctGDP
EducationExpenditurePctGDP_Year
2016.0     1
2017.0     2
2018.0    64
2019.0  

In [6]:
# ============================================================
# REPAIR MISSING HIGH-PRIORITY CONTEXT VARIABLES
#
# Adds:
#   - HDI 2018
#   - GII 2018
#   - Government Effectiveness 2018
#
# Then reruns ONLY these variables against:
#
#   1. BaselineJoint5
#   2. SubtypeReweighting
#
# both:
#   - unadjusted
#   - adjusted for GDP + World Bank region
# ============================================================


from io import StringIO
import requests
import numpy as np
import pandas as pd


# ============================================================
# 1. DOWNLOAD UNDP FILE ROBUSTLY
# ============================================================

UNDP_URL = (
    "https://hdr.undp.org/sites/default/files/"
    "2025_HDR/"
    "HDR25_Composite_indices_complete_time_series.csv"
)


headers = {
    "User-Agent":
        "Mozilla/5.0"
}


response = requests.get(
    UNDP_URL,
    headers=headers,
    timeout=120
)


response.raise_for_status()


print(
    "UNDP status:",
    response.status_code
)

print(
    "UNDP bytes:",
    len(
        response.content
    )
)


hdr = pd.read_csv(
    StringIO(
        response.text
    ),
    low_memory=False
)


hdr.columns = [
    str(
        column
    )
    .strip()
    .lower()
    for column
    in hdr.columns
]


print(
    "\nFirst UNDP columns:"
)

print(
    hdr.columns[
        :25
    ].tolist()
)


# ============================================================
# 2. FIND ISO / HDI / GII COLUMNS
# ============================================================

iso_candidates = [
    "iso3",
    "iso3_code",
    "country_code"
]


iso_col = next(
    (
        column
        for column
        in iso_candidates
        if column
        in hdr.columns
    ),
    None
)


if iso_col is None:

    raise RuntimeError(
        "Could not find ISO3 column."
    )


def find_indicator_year_column(
    columns,
    indicator,
    year
):

    indicator = (
        indicator
        .lower()
    )


    year = str(
        year
    )


    # exact preferred
    exact = (
        f"{indicator}_{year}"
    )


    if exact in columns:

        return exact


    candidates = [
        column
        for column
        in columns
        if (
            indicator
            in column
            and
            year
            in column
            and
            "rank"
            not in column
        )
    ]


    if len(
        candidates
    ) == 0:

        return None


    return candidates[
        0
    ]


hdi_col = find_indicator_year_column(
    hdr.columns,
    "hdi",
    2018
)


gii_col = find_indicator_year_column(
    hdr.columns,
    "gii",
    2018
)


print(
    "\nHDI column:",
    hdi_col
)


print(
    "GII column:",
    gii_col
)


if hdi_col is None:

    raise RuntimeError(
        "HDI 2018 column not found."
    )


if gii_col is None:

    raise RuntimeError(
        "GII 2018 column not found."
    )


undp = (
    hdr[
        [
            iso_col,
            hdi_col,
            gii_col
        ]
    ]
    .copy()
)


undp = undp.rename(
    columns={
        iso_col:
            "ISO3",

        hdi_col:
            "HDI_2018",

        gii_col:
            "GII_2018",
    }
)


undp[
    "ISO3"
] = (
    undp[
        "ISO3"
    ]
    .astype(
        str
    )
    .str.strip()
    .str.upper()
)


for column in [
    "HDI_2018",
    "GII_2018"
]:

    undp[
        column
    ] = pd.to_numeric(
        undp[
            column
        ],
        errors="coerce"
    )


undp = (
    undp
    .drop_duplicates(
        subset=[
            "ISO3"
        ]
    )
)


# ============================================================
# 3. REPLACE ANY OLD FAILED HDI/GII COLUMNS
# ============================================================

for column in [
    "HDI_2018",
    "GII_2018"
]:

    if column in macro.columns:

        macro = macro.drop(
            columns=[
                column
            ]
        )


macro = (
    macro
    .merge(
        undp,
        on="ISO3",
        how="left",
        validate="many_to_one"
    )
)


# ============================================================
# 4. GOVERNMENT EFFECTIVENESS
#
# Try current World Bank governance code first.
# Fall back to legacy GE.EST.
# ============================================================

GOVERNANCE_CODES = [
    "GOV_WGI_GE.EST",
    "GE.EST",
]


def fetch_single_wb_indicator(
    iso_codes,
    indicator,
    target_year=2018
):

    codes = ";".join(
        iso_codes
    )


    url = (
        "https://api.worldbank.org/v2/"
        f"country/{codes}/"
        f"indicator/{indicator}"
        "?format=json"
        "&date=2018"
        "&per_page=20000"
    )


    response = requests.get(
        url,
        timeout=120
    )


    response.raise_for_status()


    payload = response.json()


    if (
        not isinstance(
            payload,
            list
        )
        or
        len(
            payload
        )
        <
        2
        or
        payload[
            1
        ]
        is None
    ):

        return pd.DataFrame()


    rows = []


    for item in payload[
        1
    ]:

        value = item.get(
            "value"
        )


        iso3 = item.get(
            "countryiso3code"
        )


        if (
            value is None
            or
            not iso3
        ):

            continue


        try:

            value = float(
                value
            )

        except Exception:

            continue


        rows.append(
            {
                "ISO3":
                    iso3,

                "GovernmentEffectiveness":
                    value,

                "GovernmentEffectiveness_Year":
                    2018,

                "GovernmentEffectiveness_Code":
                    indicator,
            }
        )


    return pd.DataFrame(
        rows
    )


government = pd.DataFrame()


for code in GOVERNANCE_CODES:

    print(
        "Trying government-effectiveness code:",
        code
    )


    candidate = fetch_single_wb_indicator(
        valid_iso_codes,
        code
    )


    print(
        "  observations:",
        len(
            candidate
        )
    )


    if len(
        candidate
    ) > 0:

        government = (
            candidate.copy()
        )


        break


if len(
    government
) == 0:

    print(
        "\nWARNING: Government effectiveness still unavailable."
    )


    macro[
        "GovernmentEffectiveness"
    ] = np.nan


else:

    if (
        "GovernmentEffectiveness"
        in macro.columns
    ):

        macro = macro.drop(
            columns=[
                "GovernmentEffectiveness"
            ],
            errors="ignore"
        )


    macro = (
        macro
        .merge(
            government,
            on="ISO3",
            how="left",
            validate="many_to_one"
        )
    )


# ============================================================
# 5. COVERAGE QA
# ============================================================

print("\n")
print("=" * 110)
print("REPAIRED VARIABLE COVERAGE")
print("=" * 110)


for variable in [
    "HDI_2018",
    "GII_2018",
    "GovernmentEffectiveness"
]:

    if variable not in macro.columns:

        continue


    print(
        variable,
        ":",
        int(
            macro[
                variable
            ]
            .notna()
            .sum()
        ),
        "/",
        len(
            macro
        )
    )


# ============================================================
# 6. SIMPLE DISTRIBUTION QA
# ============================================================

print("\n")
print("=" * 110)
print("REPAIRED VARIABLE DISTRIBUTIONS")
print("=" * 110)


for variable in [
    "HDI_2018",
    "GII_2018",
    "GovernmentEffectiveness"
]:

    if variable not in macro.columns:

        continue


    values = (
        macro[
            variable
        ]
        .dropna()
    )


    if len(
        values
    ) == 0:

        continue


    print(
        "\n",
        variable
    )


    print(
        values
        .describe()
        .round(
            4
        )
        .to_string()
    )


# ============================================================
# 7. GDP CORRELATIONS
# ============================================================

print("\n")
print("=" * 110)
print("CORRELATION WITH GDP")
print("=" * 110)


for variable in [
    "HDI_2018",
    "GII_2018",
    "GovernmentEffectiveness"
]:

    if variable not in macro.columns:

        continue


    temp = (
        macro[
            [
                "LogGDPpcPPP_2018",
                variable
            ]
        ]
        .dropna()
    )


    if len(
        temp
    ) < 10:

        continue


    rho = (
        temp
        .corr(
            method="spearman"
        )
        .iloc[
            0,
            1
        ]
    )


    print(
        f"{variable}: rho = {rho:.4f}"
    )


# ============================================================
# 8. RERUN ONLY REPAIRED VARIABLES
#
# Uses functions already defined in previous cell:
#
#   freedman_lane_test
#   bh_fdr
#   RESPONSES
# ============================================================

REPAIRED_VALIDATORS = [
    variable
    for variable
    in [
        "HDI_2018",
        "GII_2018",
        "GovernmentEffectiveness"
    ]
    if (
        variable
        in macro.columns
        and
        macro[
            variable
        ]
        .notna()
        .sum()
        >=
        25
    )
]


repair_result_rows = []

test_counter = 0


for response_name, response_data in (
    RESPONSES.items()
):

    systems = (
        response_data[
            "systems"
        ]
    )


    Y_all = (
        response_data[
            "Y"
        ]
    )


    response_index = {
        country:
            i
        for i, country
        in enumerate(
            systems
        )
    }


    for variable in REPAIRED_VALIDATORS:

        # ====================================================
        # UNADJUSTED
        # ====================================================

        temp_meta = (
            macro[
                [
                    "Country",
                    variable
                ]
            ]
            .dropna()
        )


        valid_system_set = set(
            temp_meta[
                "Country"
            ]
        )


        common = [
            country
            for country
            in systems
            if country
            in valid_system_set
        ]


        frame = (
            macro[
                macro[
                    "Country"
                ]
                .isin(
                    common
                )
            ][
                [
                    "Country",
                    variable
                ]
            ]
            .set_index(
                "Country"
            )
            .loc[
                common
            ]
            .reset_index()
        )


        Y = np.vstack(
            [
                Y_all[
                    response_index[
                        country
                    ]
                ]
                for country
                in common
            ]
        )


        test_counter += 1


        result = freedman_lane_test(
            Y=Y,
            frame=frame,
            variable=variable,
            variable_type="Continuous",
            adjusted=False,
            n_permutations=5000,
            seed=(
                RANDOM_SEED
                +
                50000
                +
                test_counter
            )
        )


        if result is not None:

            repair_result_rows.append(
                {
                    "Response":
                        response_name,

                    "Validator":
                        variable,

                    "Test":
                        "Unadjusted",

                    "N_systems":
                        len(
                            common
                        ),

                    **result
                }
            )


        # ====================================================
        # GDP + REGION ADJUSTED
        # ====================================================

        required = [
            "Country",
            variable,
            "LogGDPpcPPP_2018",
            "WB_Region"
        ]


        adjusted_meta = (
            macro[
                required
            ]
            .dropna()
        )


        valid_adjusted_set = set(
            adjusted_meta[
                "Country"
            ]
        )


        common_adjusted = [
            country
            for country
            in systems
            if country
            in valid_adjusted_set
        ]


        if len(
            common_adjusted
        ) < 25:

            continue


        frame_adjusted = (
            macro[
                macro[
                    "Country"
                ]
                .isin(
                    common_adjusted
                )
            ][
                required
            ]
            .set_index(
                "Country"
            )
            .loc[
                common_adjusted
            ]
            .reset_index()
        )


        Y_adjusted = np.vstack(
            [
                Y_all[
                    response_index[
                        country
                    ]
                ]
                for country
                in common_adjusted
            ]
        )


        test_counter += 1


        result_adjusted = freedman_lane_test(
            Y=Y_adjusted,
            frame=frame_adjusted,
            variable=variable,
            variable_type="Continuous",
            adjusted=True,
            n_permutations=5000,
            seed=(
                RANDOM_SEED
                +
                60000
                +
                test_counter
            )
        )


        if result_adjusted is not None:

            repair_result_rows.append(
                {
                    "Response":
                        response_name,

                    "Validator":
                        variable,

                    "Test":
                        "AdjustedForGDPRegion",

                    "N_systems":
                        len(
                            common_adjusted
                        ),

                    **result_adjusted
                }
            )


repair_results = pd.DataFrame(
    repair_result_rows
)


# ============================================================
# 9. FDR
# ============================================================

repair_results[
    "FDR_Q"
] = np.nan


for (
    response_name,
    test_name
), indices in (
    repair_results
    .groupby(
        [
            "Response",
            "Test"
        ]
    )
    .groups
    .items()
):

    repair_results.loc[
        indices,
        "FDR_Q"
    ] = bh_fdr(
        repair_results.loc[
            indices,
            "PermutationP"
        ]
    )


# ============================================================
# 10. SAVE
# ============================================================

macro.to_csv(
    OUT_DIR
    / "extended_context_metadata_repaired.csv",
    index=False
)


repair_results.to_csv(
    OUT_DIR
    / "repaired_high_priority_context_tests.csv",
    index=False
)


# ============================================================
# 11. PRINT RESULTS
# ============================================================

print("\n")
print("=" * 120)
print("REPAIRED HIGH-PRIORITY CONTEXT TESTS")
print("=" * 120)


print(
    repair_results[
        [
            "Response",
            "Validator",
            "Test",
            "N_systems",
            "PartialR2",
            "F",
            "PermutationP",
            "FDR_Q"
        ]
    ]
    .sort_values(
        [
            "Response",
            "Test",
            "FDR_Q"
        ]
    )
    .round(
        5
    )
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 120)
print("ADJUSTED HIGH-PRIORITY FINDINGS")
print("=" * 120)


adjusted_significant = (
    repair_results[
        (
            repair_results[
                "Test"
            ]
            ==
            "AdjustedForGDPRegion"
        )
        &
        (
            repair_results[
                "FDR_Q"
            ]
            <
            0.05
        )
    ]
)


if len(
    adjusted_significant
) == 0:

    print(
        "None of HDI, GII, or government effectiveness "
        "added significant information beyond GDP + region."
    )


else:

    print(
        adjusted_significant[
            [
                "Response",
                "Validator",
                "N_systems",
                "PartialR2",
                "F",
                "PermutationP",
                "FDR_Q"
            ]
        ]
        .round(
            5
        )
        .to_string(
            index=False
        )
    )


print("\n")
print("=" * 120)
print("REPAIR STAGE COMPLETE")
print("=" * 120)

UNDP status: 200
UNDP bytes: 2001263

First UNDP columns:
['iso3', 'country', 'hdicode', 'region', 'hdi_rank_2023', 'hdi_1990', 'hdi_1991', 'hdi_1992', 'hdi_1993', 'hdi_1994', 'hdi_1995', 'hdi_1996', 'hdi_1997', 'hdi_1998', 'hdi_1999', 'hdi_2000', 'hdi_2001', 'hdi_2002', 'hdi_2003', 'hdi_2004', 'hdi_2005', 'hdi_2006', 'hdi_2007', 'hdi_2008', 'hdi_2009']

HDI column: hdi_2018
GII column: gii_2018
Trying government-effectiveness code: GOV_WGI_GE.EST
  observations: 72


REPAIRED VARIABLE COVERAGE
HDI_2018 : 70 / 75
GII_2018 : 68 / 75
GovernmentEffectiveness : 72 / 75


REPAIRED VARIABLE DISTRIBUTIONS

 HDI_2018
count    70.0000
mean      0.8644
std       0.0711
min       0.6800
25%       0.8155
50%       0.8715
75%       0.9248
max       0.9660

 GII_2018
count    68.0000
mean      0.1812
std       0.1271
min       0.0180
25%       0.0795
50%       0.1350
75%       0.2608
max       0.4500

 GovernmentEffectiveness
count    72.0000
mean      0.7923
std       0.7979
min      -0.6666
25%   